# WIMB: train your own model on a free Colab GPU

**How to run it**
1. Check the top right says **T4**. If it doesn't: **Runtime → Change runtime type → T4 GPU → Save**.
2. Click **Runtime → Run all**.
3. In the first minute Google asks to connect **Google Drive**. Click **Connect** and **Allow**
   (the trained model is saved there).
4. Keep this tab open. It takes about 30–40 minutes.
5. When it's done: **File → Save a copy in GitHub** → repository `pratik9yadav/yok`,
   branch `main`, file path `train_colab.ipynb` → **OK**.
   Then tell Claude it's done. Claude reads the results from GitHub.

**What it does:** takes a free model (Qwen3 4B), tests it on shop calls, trains it on
400 examples, tests it again, then tests the saved Ollama file too. Two test sets: 30 calls
and 56 harder ones written in styles the training data does not have. The model goes to Google Drive.

It also runs on Kaggle's free GPU: `run_on_kaggle.py` starts it there with no clicks.

In [ ]:
#@title 1. Check the GPU
import os, shutil, subprocess, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle gives two GPUs; training uses one
# nvidia-smi is not always installed, so also ask torch (in its own process: nothing is imported here yet).
cuda = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"],
                      capture_output=True, text=True).stdout.strip() == "True"
if not (cuda or shutil.which("nvidia-smi")):
    # What the machine has, to see why there is no GPU (no secret values printed).
    print("nvidia devices:", [d for d in os.listdir("/dev") if d.startswith("nvidia")])
    print("Kaggle settings:", {k: v for k, v in os.environ.items()
                               if k.startswith("KAGGLE") and any(w in k for w in ("ACCELERATOR", "GPU", "MACHINE", "RUN_TYPE"))})
    raise RuntimeError("No GPU. Colab: Runtime > Change runtime type > T4 GPU > Save, then Run all. "
                       "Kaggle: verify your phone, then turn on GPU T4 once in the notebook editor (Settings > Accelerator).")
print(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.get_device_name(0))"],
                     capture_output=True, text=True).stdout or "GPU found")

In [ ]:
#@title 2. Install the training tools (about 3–5 minutes)
!pip install -q unsloth

In [ ]:
#@title 3. Settings (and connect Google Drive now, so nothing waits for you later)
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"  #@param {type:"string"}
EPOCHS = 2  #@param {type:"integer"}
# v2: training data built from real products, prices and spellings, plus 151 real Hindi answers.
# v1: the first 400 made-up examples (kept to compare against).
TRAIN_SET = "v2"  #@param ["v2", "v1"]
OUTPUT_NAME = "wimb-shop-model"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
MAKE_OLLAMA_FILE = True  #@param {type:"boolean"}
# This notebook is set up for Qwen3 instruct models (their chat format is used below).

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/tmp" if ON_KAGGLE else "/content"  # big temporary files (Kaggle saves all of /kaggle/working)
if ON_KAGGLE:
    SAVE_DIR = f"/kaggle/working/{OUTPUT_NAME}"  # shown in the notebook's Output tab
elif SAVE_TO_DRIVE:
    SAVE_DIR = f"/content/drive/MyDrive/{OUTPUT_NAME}"
    from google.colab import drive
    drive.mount("/content/drive")
else:
    SAVE_DIR = None

In [ ]:
#@title 4. Training data and scoring (hidden; built from data/ and score.py)
"""Shared by the notebook and the tests: the task's prompt, and how an answer is scored.

The task: read what a shopkeeper said on a phone call and return JSON with four fields.
"""

import json
import re
import time

SYSTEM = (
    "You read what a shopkeeper said on a phone call about a product. "
    "Reply with only a JSON object with these keys: "
    "in_stock (true, false or null), price_inr (a number or null), "
    "delivery (true, false or null), alternative (text or null). "
    "price_inr is the price of the product that was asked for. "
    "Use null when the shopkeeper did not say."
)

FIELDS = ("in_stock", "price_inr", "delivery", "alternative")


def user_message(product: str, said: str) -> str:
    return f"Product: {product}\nShopkeeper: {said}"


def answer_text(answer: dict) -> str:
    """The exact text the model is trained to reply with."""
    return json.dumps({k: answer[k] for k in FIELDS}, ensure_ascii=False)


def parse(text: str) -> dict | None:
    """The last JSON object in the reply that has exactly the four keys, or None.

    Every "{" is tried, so braces inside a value ("M-seal {2 pack}") or other text around the
    answer do not hide it.
    """
    decoder = json.JSONDecoder()
    found = None
    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text, match.start())
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and set(value) == set(FIELDS):
            found = value
    return found


def _tokens(text: str) -> set[str]:
    """Lower-case words, with "40mm" read as "40 mm" and punctuation dropped."""
    text = re.sub(r"(?<=\d)(?=[a-z])|(?<=[a-z])(?=\d)", " ", text.lower())
    return set(re.sub(r"[-_.,;:!?()\[\]{}\"'/]", " ", text).split())


def same(field: str, got, want) -> bool:
    if want is None or got is None:
        return got is None and want is None
    if field in ("in_stock", "delivery"):
        return isinstance(got, bool) and got == want  # 1 and 0 are not true and false
    if field == "price_inr":
        if isinstance(got, bool):
            return False
        try:
            return float(got) == float(want)
        except (TypeError, ValueError):
            return False
    # alternative: the wanted words must all be there, with at most two extra ("40 mm padlock").
    if not isinstance(got, str):
        return False
    have, need = _tokens(got), _tokens(want)
    return bool(have) and need <= have and len(have - need) <= 2


def score(replies: list[str], examples: list[dict]) -> dict:
    """Per-field accuracy, all-four-right accuracy, and the mistakes.

    A reply with no valid answer object (or a missing or extra key) gets every field wrong.
    """
    right = {f: 0 for f in FIELDS}
    all_right = json_ok = 0
    mistakes = []
    for reply, ex in zip(replies, examples, strict=True):
        got = parse(reply)
        if got is not None:
            json_ok += 1
        wrong = list(FIELDS) if got is None else [f for f in FIELDS if not same(f, got[f], ex["answer"][f])]
        for f in FIELDS:
            right[f] += f not in wrong
        if wrong:
            mistakes.append({"said": ex["said"], "want": ex["answer"], "reply": reply, "wrong": wrong})
        else:
            all_right += 1
    n = len(examples)
    return {
        "examples": n,
        "json_ok": json_ok / n,
        "all_correct": all_right / n,
        **{f: right[f] / n for f in FIELDS},
        "mistakes": mistakes,
    }


KEYS = ("all_correct", "json_ok", *FIELDS)


def evaluate(ask, sets: dict[str, list[dict]], deadline: float | None = None) -> dict:
    """Run ask(product, said) over every test set and score it. Every reply is kept.

    After `deadline` (a time.time() value) no more questions are asked; what was answered is
    scored, and "of" says how many there were.
    """
    out = {}
    for name, rows in sets.items():
        replies = []
        for ex in rows:
            if deadline is not None and time.time() > deadline:
                break
            replies.append(ask(ex["product"], ex["said"]))
        done = rows[: len(replies)]
        scored = score(replies, done) if done else {"examples": 0, "mistakes": []}
        out[name] = {**scored, "replies": replies, "of": len(rows)}
    return out


def show(label: str, results: dict, max_mistakes: int = 8) -> None:
    for name, res in results.items():
        n = res["examples"]
        if not n:
            print(f"== {label} [{name}]: no calls answered")
            continue
        partial = f" (only {n} of {res['of']} answered)" if n < res["of"] else ""
        print(f"== {label} [{name}]: all 4 fields right in {res['all_correct']:.0%} of {n} calls{partial}")
        print("   " + "   ".join(f"{k} {res[k]:.0%}" for k in KEYS[1:]))
        for m in res["mistakes"][:max_mistakes]:
            print("\n   said: ", m["said"])
            print("   want: ", answer_text(m["want"]))
            print("   reply:", m["reply"][:200])
            print("   wrong:", ", ".join(m["wrong"]))


def results_payload(model: str, epochs: int, before: dict, after: dict) -> dict:
    """What results.json holds: scores, mistakes and every raw reply, so scores can be recomputed."""
    sets = {}
    for name, a in after.items():
        b = before[name]
        sets[name] = {
            "examples": a["of"],
            "before": {k: round(b[k], 3) for k in KEYS},
            "after": {k: round(a[k], 3) for k in KEYS},
            "mistakes_before": b["mistakes"],
            "mistakes_after": a["mistakes"],
            "replies_before": b["replies"],
            "replies_after": a["replies"],
        }
    return {"model": model, "epochs": epochs, "sets": sets}


def make_modelfile(gguf_name: str) -> str:
    """An Ollama Modelfile whose template is exactly the text the model was trained on.

    Unsloth's own Qwen3 template differs a little (a blank line after the system message, and
    thinking-mode syntax that needs a newer Ollama), so it is not used.
    """
    return (
        f"FROM ./{gguf_name}\n"
        'TEMPLATE """{{ if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{ end }}'
        '{{ if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{ end }}'
        '<|im_start|>assistant\n{{ .Response }}<|im_end|>\n"""\n'
        'PARAMETER stop "<|im_end|>"\n'
        "PARAMETER temperature 0\n"
        f'SYSTEM """{SYSTEM}"""\n'
    )


TRAIN_V2 = [json.loads(line) for line in r'''
{"product": "CEAT 3.00-18 tyre", "said": "No no, it's there. We'll deliver by evening", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bosch drill machine", "said": "kami nahi, bharpur ahe, rate sadenau hazar ahe", "answer": {"in_stock": true, "price_inr": 9500, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "संपलं काका", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Not available right now.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हो, बल्ब शिल्लक आहे। बरं सुटा एक २९० ला, पूर्ण पॅकेट ४३०", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "tyre sampla", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "aj nahin hai, kal aayegaa.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "tube padliye sutta ek 940 la purna packet 1600 porga pathavto", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "ख़त्म हो गया। Daybetter वाला चलेगा? उसका दो सौ सत्तर है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Daybetter"}}
{"product": "Bosch screwdriver bit set", "said": "arre haa ready stock hai tin sau pachattar final price hai bhejne wala koi nahin hai aapko aana padega", "answer": {"in_stock": true, "price_inr": 375, "delivery": false, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "Yes, available, our price is 1,402, MRP is 2,000, no delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": 1402, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हो, ठेवतो आम्ही, पण आत्ता संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "That we don't have. I can give Golden Bullet drill instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Golden Bullet drill"}}
{"product": "Havells 1200 mm ceiling fan", "said": "arre kya bataun, subah se bahut rush hai, haan ji, mil jayega, aapko number kisne diya? acha thik hai, 2510... ek minute, rate list dekhta hun... haan sattais sau pachas, market mein maal ki kami chal rahi hai, ghar pe bhej denge.", "answer": {"in_stock": true, "price_inr": 2750, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "wo model available nahin hai, Michelin wiper available hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Michelin wiper"}}
{"product": "BMB welding machine", "said": "taintalis sau lagega, acha haan, machine hai", "answer": {"in_stock": true, "price_inr": 4300, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "हो, आहे शिल्लक। १८०० रुपये, कमी नाही होणार। पोरगा पाठवतो, ६० रुपये रिक्षाचे।", "answer": {"in_stock": true, "price_inr": 1800, "delivery": true, "alternative": null}}
{"product": "Agrico TATA plier", "said": "पक्कड संपली नाही, आहे, 255 रुपये", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "aahe na, mhanje sutta ek ekshe sattar la, purna pyaket pavaneteenshe.", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "नल है मिल जाएगा 320 MRP लिखा है 210 लगा देंगे", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "haan bolo, available hai, 260 final.", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Yes, we keep that, I would send it, but my boy is on leave today, come and take it", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "haan sir, bataiye, 90 rupaye hai, han, maal pada hai, network issue hai, aawaz break ho rahi hai, delivery nahi hoti, pickup karna padega, dekho season time hai na, demand jyada hai", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "1.5 sq mm nahi hai, 1 sq mm wala hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 sq mm"}}
{"product": "Shakti 250A welding machine", "said": "मशीन नाहीये, संपली, Vormir वाला आहे, 5130 ला, पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Vormir"}}
{"product": "Philips 5W LED bulb", "said": "Haan ahe, kya hai na matlab panchvis ghetle tar ekshe pandhara pramane, ek piece savvashe", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "Abhi nahin, order pe mangva denge", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "haan, wahi wala hai bhaiya", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "एक मिनिट हां... बोला, पंधराशे पन्नास फिक्स, हो, आहे", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हॅलो आवाज येतोय का हां बोला बारा नग शिल्लक आहेत अजून काही", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "haan ji, oil available hai, 410 rupaye, matlab delivery nahin karte, aake le jao", "answer": {"in_stock": true, "price_inr": 410, "delivery": false, "alternative": null}}
{"product": "Cheston angle grinder", "said": "हो, बोला दादा, ते आमच्याकडे नसतं, बरं Khadija grinder आहे, त्याचे साडेबाराशे रुपये.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Khadija grinder"}}
{"product": "teflon tape", "said": "टेप आहे, मिळेल, दहा घेतले तर बावीस प्रमाणे, एक पीस पंचवीस, डिलिव्हरी करत नाही.", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "Yes, it's there. If you take 12 it's 115 each, one piece is 130", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "It's not finished, we have plenty", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "Yes sir, we have. Basically, it was 235, now reduced to 199.", "answer": {"in_stock": true, "price_inr": 199, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "aahe, donch urlet, price charashe tees aahe.", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "Moonstruck 2000W immersion rod", "said": "आमच्याकडे आहे, रेट कमी झालाय, पाचशे पंच्याऐंशी होता, आता सव्वा पाचशे", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Michelin tyre", "said": "हाँ बोलिए. वो हम नहीं रखते. बताइए क्या करना है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "milel na, ahe, sutta pannas la, 10 nagancha box ghetla tar charshe sath, porga pathavto, dekho aur kuch?", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "Roots horn", "said": "नमस्कार, बोला. शिल्लक नाही. Minda Black वाला घ्या, ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Minda Black"}}
{"product": "10W-30 engine oil 1 L", "said": "ho sir, bola, kay aahe na sorry, out of stock aahe, Diwali chi gardi aahe na", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crasts LED torch", "said": "हैलो, कौन? हाँ, रखा है. आठ सौ पैंसठ रुपये. भेजने वाला कोई नहीं है, आ के ले जाओ.", "answer": {"in_stock": true, "price_inr": 865, "delivery": false, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "haan, kaun? acha, bolo, abhi khatam hai bhai, chahiye toh Orient heater de dunga, home delivery nahi hai bhai.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Orient heater"}}
{"product": "9W LED bulb", "said": "han, bees piece pade hain, ek sau tees rupaye, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 130, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "सातशे साठ फिक्स. पुट्टी संपली नाही, आहे. हां तर ठीक आहे?", "answer": {"in_stock": true, "price_inr": 760, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "7Ah nahi hai, 5Ah wala hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "5Ah"}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हाँ रखा है एक पीस 92 का मिलता है पैकेट 415 का", "answer": {"in_stock": true, "price_inr": 415, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "aaj nahi, udya yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "बरं BMB संपलं, शक्ती मशीन आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "शक्ती मशीन"}}
{"product": "Carzex LED headlight", "said": "लाईट पडलीये, अठराशे होतील.", "answer": {"in_stock": true, "price_inr": 1800, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "yes available hai ek sau chalis final", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "haan haan, bolo, market mein maal ki kami chal rahi hai, umm ji, available hai, aaj subah se light nahi hai, kal Sunday tha toh dukaan band thi, bhaiya aap ruko do minute, phone pe hun, 2499 ka hai, isse kam nahi hoga, delivery ho jayegi, bata dena.", "answer": {"in_stock": true, "price_inr": 2499, "delivery": true, "alternative": null}}
{"product": "AmiciTools water pump", "said": "अभी नहीं है, ऑर्डर दिया है, दो दिन में आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "आहे की, मिळेल। 40 ला देतो। दुकान दहा वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "ho bola, sade saha hajar la deto, haan ahe", "answer": {"in_stock": true, "price_inr": 6500, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "Yes... one minute, no, that got finished yesterday", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "मशीन पडलीये. रेट ५२०० आहे.", "answer": {"in_stock": true, "price_inr": 5200, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "helmet sampla nahi aahe savva aathshe hotil pathavla asta pan aaj porga nahi yeun ghya", "answer": {"in_stock": true, "price_inr": 825, "delivery": false, "alternative": null}}
{"product": "Hella horn", "said": "Haan ha, hai, acha nau sau das, aath sau nahin bola maine.", "answer": {"in_stock": true, "price_inr": 910, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "३५० रुपये कमी नाही होणार स्टॉकमध्ये आहे", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Agrico TATA plier", "said": "हॅलो आवाज येतोय का हां बोला ती संपली हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "३३० रुपयांना पडेल। हो, ऑइल शिल्लक आहे।", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "hello? haan, awaaz aa rahi hai, bolo, haan haan, hai, umm chaar sau assi rupaye ka hai, delivery nahi karte, aake le jao.", "answer": {"in_stock": true, "price_inr": 480, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हां बोला साहेब, हो दादा, आहे, दिवाळीची गर्दी आहे ना, रेट पंचवीस आहे, डिलिव्हरी नाही, दुकानात येऊन घ्या, कंपनीने परत रेट वाढवले, पोरगा चहा आणायला गेलाय, बाजारात माल कमी आहे सध्या", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "मतलब ५२० का आता है, लेकिन अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "आज नाही, उद्या येईल। बरं पंधरा होतील।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "आहे ना, 355 प्लस GST", "answer": {"in_stock": true, "price_inr": 355, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "abhi nahi hai aur kuch?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "milel na aahe 12 ghetle tar 50 pramane ek pees 60 gharpoch milel", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "Cover nahi hai, khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "मिल जाएगा, है अपने पास, मतलब हमारा रेट 305, MRP तो 325 है.", "answer": {"in_stock": true, "price_inr": 305, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हॅलो? आवाज येतोय का? हां, बोला, हो, आहे, रेट शेठ सांगतील", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "plate sampli nahi, aahe, aadhi 625 hota, ata 780 zala, gharpoch nahi, tumhala yava lagel.", "answer": {"in_stock": true, "price_inr": 780, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "No, we don't have it. We have white cement, same thing", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "white cement"}}
{"product": "Havells 1500W immersion rod", "said": "हॅलो आवाज येतोय का हां बोला थांबा बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "Available hai, koi problem nahi, teen sao bees mein de dunga, dukaan nau baje tak khuli hai, free delivery hai.", "answer": {"in_stock": true, "price_inr": 320, "delivery": true, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "haan, motor hai, season time hai na, demand jyada hai, chhabees sau ka hai, sir aj bahut busy day hai, network issue hai, aawaz break ho rahi hai, ladka de aayega, 30 rupaye auto ka", "answer": {"in_stock": true, "price_inr": 2600, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Haan, kaun? acha, bolo aaj subah se light nahi hai haan, maal pada hai aapko number kisne diya? acha thik hai acha haan haan, sun raha hun market mein maal ki kami chal rahi hai 90 ka padega shaam tak pahuncha denge", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ek minute ha... haan bola, samplay, udya dupari yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "हो, टायर शिल्लक आहे। सव्वीसशेला देतो। डिलिव्हरी? नाही हो, ते नाही करत।", "answer": {"in_stock": true, "price_inr": 2600, "delivery": false, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "कंपनीने परत रेट वाढवले। मालक नाहीत, नंतर फोन करा। काय सांगू, सकाळपासून गर्दी आहे। बाजारात माल कमी आहे सध्या। दिवाळीची गर्दी आहे ना। अजून काही?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "एक मिनिट हां... बोला. आहे ना. सव्वा चार हजार आणि 18 टक्के GST वेगळा. डिलिव्हरी करत नाही.", "answer": {"in_stock": true, "price_inr": 4250, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हो हो समजलं पण ते नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बरं थांबा बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हैलो, कौन? नहीं मिलेगा भाई। और कुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "haan haan, bolo, matlab hai na, kitne chahiye? rate 170 hai", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Aegon angle grinder", "said": "dekho khatam ho gaya tha... nahi, ek bacha hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "नाही ना, संपलंय। आज नाही, उद्या येईल। बघा डिलिव्हरी नाही, दुकानात येऊन घ्या।", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट संपलीये। Jupiter clutch plate घ्या, ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Jupiter clutch plate"}}
{"product": "1 inch PVC elbow", "said": "अभी नहीं है। CPVC elbow है, 10 का।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "CPVC elbow"}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "बरं आमच्याकडे नसतं, ऑर्डर करून मागवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "yes, bola, kay aahe na ahe na, theek ahe?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "15 नग शिल्लक आहेत, घरी पाठवतो.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Ek minute ha... haan bola, wire sampli nahi, ahe, gharapoch milel", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "नाहीये, मागवून देतो, दोन दिवस लागतील", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "ho aahe... nahi nahi, to dusra aahe, ha nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "हॅलो? आवाज येतोय का? हां, बोला, स्टॉकमध्ये आहे, तुम्हाला चारशे पन्नासला देतो, MRP सातशे वीस आहे, पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "Hello, haan boliye. haan sir, wo model available hai. chhah so pachanve rupaye lagenge", "answer": {"in_stock": true, "price_inr": 695, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "sava teen sau last rate hai, ji haan, rakha hai", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "ट्यूब आहे, मिळेल, तुम्हाला ४७५ ला देतो, MRP ५४० आहे, पोरगा पाठवतो.", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "available hai barah ka full box 2680 single piece do sau pachaas matlab sorry delivery service nahi hai", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Bagha ho, kulup milel, teenshe navvad rupayanna padel, delivery nahi, dukanat yeun ghya.", "answer": {"in_stock": true, "price_inr": 390, "delivery": false, "alternative": null}}
{"product": "Aegon angle grinder", "said": "बरं तुम्हाला 1400 ला देतो। हो, मशीन शिल्लक आहे। अजून काही?", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Roots horn", "said": "नमस्ते, बोलिए, वो तो कल ही ख़त्म हो गया, Favourite Bikerz horn दे सकता हूँ, 865 रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Favourite Bikerz horn"}}
{"product": "Bosch drill machine", "said": "sampla nahi ajun, aahe, 9970 la deto", "answer": {"in_stock": true, "price_inr": 9970, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "जी बोलिए, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, कल सुबह आ जाएगा, अभी नहीं, आज सुबह से लाइट नहीं है, डिलीवरी फ़्री है, आवाज़ कट रही है, ज़ोर से बोलिए, एक ग्राहक खड़ा है, एक मिनट, मतलब ठीक है?", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Easy4buy horn", "said": "Basically, yes, we have it. 575 for you, printed price is 865. No delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": 575, "delivery": false, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "Hello, yes, I can hear you. Available, sir. 530 rupees. OK?", "answer": {"in_stock": true, "price_inr": 530, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "Bola saheb, mahit nahi, godown madhe baghava lagel, Diwali chi gardi aahe na, aawaj tutatoy, motyane bola, ek giraik ahe, ek minute, company ne parat rate vadhavle.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "Haan bola, kay pahije? ho, oil ahe, porga chaha aanayla gelay, kay sangu, sakal pasun gardi aahe, 1499 hotil, ho, sola piece aahet, aawaj tutatoy, motyane bola", "answer": {"in_stock": true, "price_inr": 1499, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "बोला ना, कंपनीने परत रेट वाढवले, हो, ऑइल आहे, हां हां, ऐकतोय, सव्वापाचशेला देतो, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ho battery ahe 1450 rupayanna padel", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Nahi bhai, yahan nahi milega, local clutch plate chalega? 770 ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local clutch plate"}}
{"product": "4 inch door hinge", "said": "नमस्कार, बोला, बिजागरी नाहीये, संपली, बरं आज नाही, उद्या येईल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat", "said": "haan bola kay pahije ho dada ahe mhanje aata sadesolashe aahe aadhi 1280 hota", "answer": {"in_stock": true, "price_inr": 1650, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "Stock madhe ahe, bara purna packet 1750, sutte pan ahet 1040 la ek.", "answer": {"in_stock": true, "price_inr": 1750, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "nahi hai, mangva dunga, do din lagenge", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "नमस्ते, बोलिए. वो तो कल ही ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes, we keep that. The mechanical seal costs 120, the one you asked for is 170", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "Not available right now. See, we have JK tyre, that's 1150", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "JK tyre"}}
{"product": "12V 7Ah battery", "said": "battery aahe, milel saheb", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bola, hai ji, mil jayega, 60 lagega", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Normally 70, but it's finished.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina bike cover", "said": "हैलो, कौन? देखिए हाँ, कवर है. आपके लिए दो सौ पैंसठ. डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 265, "delivery": false, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "namaskar, bola, abhi dukaan band kar raha hun, kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "sampla nahi ajun, aahe, pandhrashe hotil.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "नहीं है रुको रुको है एक पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "Yes sir, tell me. Basically, yes, it's there. 3,940 rupees.", "answer": {"in_stock": true, "price_inr": 3940, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "है भाई, अच्छा एक हज़ार पचास रुपये", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Aadhi sattar la hota, ata ahe ki nahi baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "khatam ho gaya tha... nahi, ek bacha hai, 170 rupaye lagenge", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "Mil jayega, hai, dekho MRP toh chaar sau hai, apna rate dhai sau", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "Yes sir, tell me. Yes sir, we have. Market is short of stock these days. Delivery is 30 extra. The item is 775, delivery separate. My boy has gone for tea. OK?", "answer": {"in_stock": true, "price_inr": 775, "delivery": true, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "सध्या नाही, मागवावं लागेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बघा आहे... एक मिनिट, नाही, ते काल संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "namaskar, bola, bara aawaj tutatoy, motyane bola, owner nahit, nantar call kara, kay sangu, sakal pasun gardi aahe, company ne parat rate vadhavle, porga chaha aanayla gelay", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हो, ट्यूब आहे। चारशे सत्तर होतील", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "आहे की, मिळेल, सोळाशे पन्नास रुपये", "answer": {"in_stock": true, "price_inr": 1650, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "No V-GRIP, only Retina cover.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Retina cover"}}
{"product": "Cheston 200A welding machine", "said": "Ek minute... ha bolo, available hai, paune chaar hajar last rate hai", "answer": {"in_stock": true, "price_inr": 3750, "delivery": null, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "helo, haan bolo nahin bhai, yaha nahi milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "Diwali chi gardi ahe na, bulb sampla nahi, ahe, company ne parat rate vadhavle, savvacharshe padel, 440 cha rate juna aahe, to nahi, ek giraik ahe, ek minute.", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "motor sampli nahi ahe arre rate badalla ata 1140 aadhi 1330 hota thik hai", "answer": {"in_stock": true, "price_inr": 1140, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "dekho ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "aaj nahi hai kal aayega 530 ka milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "Hello? It comes to 160. Basically, yes, we keep that", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "Ji, available hai, 395 lagega", "answer": {"in_stock": true, "price_inr": 395, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "है भाई आपके लिए सवा दो सौ दुकान दस बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "Matlab ho, aahe aamchyakade, aapla rate tinashe pach, MRP tinashe pannas ahe", "answer": {"in_stock": true, "price_inr": 305, "delivery": null, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "cover hai, mil jaega, 150 lagega, Retina cover chahiye to wo bhi hai, 145 wala, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "han ji, mil jayega, pone chaar sau men de dunga.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "haan toh rate 525 ke aas paas hoga, stock malik se puchna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "हाँ जी, मिल जाएगा, 470 लगा देंगे, वैसे MRP 620 लिखा है.", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "आत्ता सांगता येणार नाही, संध्याकाळी फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "Hello? Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Han, kaun? acha, bolo. rate teen sao sath hai, par abhi stock nahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "Hello? kuch sunai nahi de raha, baad mein karo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "tape aahe, milel, padhra rupayanna padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 15, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "Bijagri aahe, milel, kay aahe na sutta 30 la, 12 nagancha box ghetla tar 295, gharpoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "हो दादा, आहे, चौदाशे रुपये होतील, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "Ho sir, available aahe, daha ghetle tar 250 pramane, ek piece 270", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "ek customer khada hai, ek mint, awaaz kat rahi hai, zor se bolo, ha haan, sun raha hun, pankha khatam nahi hua, hai, gyarah sau pachas lagega", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "एक मिनिट हां... बोला। ती संपली हो।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "tube sampli nahi ahe purna packet 1550 sutte pan ahet 1060 la ek", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "han sir, boliye, awaaz kat rahi hai, zor se bolo, haan ji, tape available hai, acha aapko number kisne diya? acha theek hai, rate 25 hai, arre kya bataun, subah se bahut rush hai, bas 22 piece bache hain", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Plug samplay, Pulsar 180 wala aahe, 210 la.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pulsar 180"}}
{"product": "JY Super LED torch", "said": "available hai, teen sao pachattar rupaye ka hai", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "1250 la padel, wire padliye.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "haan bola, kay pahije? baghto aani kalavto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हां बोला साहेब हां तर ऑइल आहे की नाही माहीत नाही गोडाऊनमध्ये बघतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "han ji, boliye, abhi busy hun, aadhe ghante bad phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Actually, yes, available. 1250, final price. I'll send it with our boy", "answer": {"in_stock": true, "price_inr": 1250, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "aahe ki, milel teenshe panchvis padel, donashe navvad cha rate juna aahe, to nahi address pathva, pathavto thik hai?", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "मिल जाएगा है अपने पास ७७०० लगेगा और कुछ", "answer": {"in_stock": true, "price_inr": 7700, "delivery": null, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "नाही नाही, आहे, 750 रुपयांना पडेल.", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "Crompton Hill Briz ceiling fan", "said": "ji, available hai, 2250 hai, 2240 nahi, delivery nahi karte, aake le jao.", "answer": {"in_stock": true, "price_inr": 2250, "delivery": false, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "aamhi te thevat nahi. Speedwav headlight ahe, 1450 la", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Speedwav headlight"}}
{"product": "Windek tyre", "said": "tyre pada hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "company ne parat rate vadhavle, Diwali chi gardi aahe na, ho, thevto aamhi, aahe, haan haan, aiktoy, aaj sakali light navhti, tumhala 723 la deto, theek ahe?", "answer": {"in_stock": true, "price_inr": 723, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "aamhi te thevat nahi, tyachya aivaji 32A MCB deu shakto.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "32A MCB"}}
{"product": "V-GRIP steering cover for Ritz", "said": "म्हणजे स्टॉकमध्ये आहे, आमचा रेट पाचशे पंचेचाळीस, MRP 820", "answer": {"in_stock": true, "price_inr": 545, "delivery": null, "alternative": null}}
{"product": "Benz helmet", "said": "बरं हो, हेल्मेट शिल्लक आहे, पावणेनऊशे आहे, आठशे पंचावन्न नाही", "answer": {"in_stock": true, "price_inr": 875, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Bola dada, aahe, 10ch urlet, rate sheth sangtil, porga pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "mhanje ho, savvees piece aahet, tumhala 365 la deu", "answer": {"in_stock": true, "price_inr": 365, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Network issue hai, aawaz break ho rahi hai, please thodi der baad call karo, sir aaj bahut busy day hai, season time hai na, demand jyada hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ESN 2000W immersion rod", "said": "Han, maal pada hai, sadhe teen sau hoga, ladka de aayega, chalis rupaye auto ka.", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Company ne is mahine phir rate badhaya, kal Sunday tha toh dukaan band thi, available hai, abhi rate do sau ho gaya, pichle mahine ek so sattar tha.", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, we have it. See, it's 180 rupees.", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan tar hai, 2 piece rakhe hai, delivery nahin hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "अच्छा बल्ब नहीं है, खत्म हो गया, LED headlight bulb रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "LED headlight bulb"}}
{"product": "Tata Agrico plier", "said": "पक्कड संपली नाही, आहे, दोनशे पंचावन्न रुपये होतील.", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "nahiye, aaj nahi, udya yeil, 1250 la deto, bagha delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Sorry sir, abhi out of stock hai, uski jagah 2.25 mfd capacitor le sakte ho.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.25 mfd capacitor"}}
{"product": "9W LED bulb", "said": "haan sir, available hai 90 + GST extra", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Haan bola. ji haa, rakha hai. 320 hoga", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर आहे, मिळेल। १३५० होतील। 2.5 sq mm wire पण आहे, ते १७३० ला", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "हो, मशीन शिल्लक आहे, बघा साडेनऊ हजार आणि 18 टक्के GST वेगळा", "answer": {"in_stock": true, "price_inr": 9500, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "aahe... ek minute, nahi, te kal sampla.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan rakhte hain, par abhi khatam hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb sampla.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Milel na, aahe, aadhi 76 hota, aata 100 zala", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Malak nahit, nantar phone kara bhau", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ, छह पीस पड़े हैं, एक सौ चालीस में दे देंगे", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes, bola, ho, plug milel, rate donashe daha ahe sir", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ख़त्म नहीं हुआ, अभी भी है। घर पर भेज देंगे।", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "V-Guard Clado exhaust fan", "said": "pankha sampala, sadhya nahiye, order dilay, don divsat yeil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "Shakti संपलं, iBELL machine आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "iBELL machine"}}
{"product": "Modx steering cover for i10", "said": "कव्हर पडलंय, चारशे पन्नास रुपये होतील, पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "acha haan, mil jayega bhai, 320 ka ek, 6 loge toh 285 ke hisaab se laga dunga", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "Aaj sakali light navhti, pankha sampla nahi, aahe, aata akarashe aahe, aadhi 850 hota, ek giraik aahe, ek minute, theek aahe?", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "Ji boliye, haan, ready stock hai, price 2500 hai, free delivery hai", "answer": {"in_stock": true, "price_inr": 2500, "delivery": true, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "mhanje wo toh kal hi khatam ho gaya ek customer khada hai ek minute", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "ho bola, machine sampli nahi, aahe, 10250 rupaye hotil, gharpoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 10250, "delivery": false, "alternative": null}}
{"product": "Goodyear tyre", "said": "३४५० रुपये होतील। टायर संपला नाही, आहे। घरी पाठवतो", "answer": {"in_stock": true, "price_inr": 3450, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "तुम्हाला साडे पाच हजारला देतो। हां तर मोटर पडलीये", "answer": {"in_stock": true, "price_inr": 5500, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "हो मशीन आहे तुम्हाला पावणेपाच हजारला देतो डिलिव्हरी नाही दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": 4750, "delivery": false, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "Abhi nahi hai, order diya hai, do din me aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "bagha bulb sampala nahi, aahe, donashe chalis la padel, delivery deto", "answer": {"in_stock": true, "price_inr": 240, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb samplay kay aahe na 7w led bulb aahe tyache 100 rupees", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "7w led bulb"}}
{"product": "50 mm padlock", "said": "haan sir, bataiye, matlab owner abhi available nahi hain, baad mein call kijiye", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "Available, sir. It was 205, now reduced to 175.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "dekho hai bhai, aapke liye 3590, bhejne wala koi nahi hai, aapko aana padega", "answer": {"in_stock": true, "price_inr": 3590, "delivery": false, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हो बल्ब शिल्लक आहे एक पीस 315, 6 घेतले तर 285 प्रमाणे लावतो", "answer": {"in_stock": true, "price_inr": 315, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ek minute ha... haan bola, oil aahe, milel, pavanecharshe fix, aahe, ajun 17 dabe padlet, address pathva, pathavto.", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "Sold out, sir, it's very busy since morning", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "Haan bhaiya, bataiye, ha, mil jayega bhai, awaaz kat rahi hai, zor se bolo, market me maal ki kami chal rahi hai, company ne rate badla, ab atharah sau pachas hai, pehle 2110 tha, Diwali ka season hai na, sab yahi maang rahe hain, ek customer khada hai, ek minute", "answer": {"in_stock": true, "price_inr": 1850, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "haan aahe.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ti sampli ho", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "pakkad padliye, kay aahe na tumhala 400 la deto, MRP 480 aahe, porga pathavto.", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "We have it, yes. Actually, that will be Rs 250. You'll have to collect it from the shop", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Haan tar ho ho, samajla, pan te nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "aahe, bavisch urlet delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बरं आत्ता सांगता येणार नाही, संध्याकाळी फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "ho dada, aahe, sadeaathshe la padel", "answer": {"in_stock": true, "price_inr": 850, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "म्हणजे नाही कसं, आहे ना, २७०० ला पडेल", "answer": {"in_stock": true, "price_inr": 2700, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "बोला ना, पंधरा नग शिल्लक आहेत, पाच हजार दोनशे रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 5200, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "haan, mil jayega bhai, pehle 2350 tha, ab 3050 ho gaya.", "answer": {"in_stock": true, "price_inr": 3050, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "सेठ जी बाहर गए हैं, वही बताएँगे। त्योहार का सीज़न है ना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "ho, plate aahe, rate sheth sangtil", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "हैलो, कौन? पहले ८६०० का था, अभी है कि नहीं देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "hai bhai, khula ek 170 ka, poora packet do sau pachasi.", "answer": {"in_stock": true, "price_inr": 285, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "ho, milel paket 1750 la aahe, sutta ek 1080 la", "answer": {"in_stock": true, "price_inr": 1750, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "han ji, boliye check karke batata hun network issue hai, aawaz break ho rahi hai season time hai na, demand jyada hai sir aaj bahut busy day hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "हाँ भाई, बोलो, हाँ, है, ढाई हज़ार का पड़ेगा, शाम तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 2500, "delivery": true, "alternative": null}}
{"product": "Vega Crux helmet", "said": "hello? aawaj yet nahiye, nantar kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Arre oil khatam nahi hua, hai, ab chaar sau chalis hai, pehle 330 tha", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "haan, kaun? acha, bolo, dekho ha hai, ek hajar chhah sau plus 18 percent GST, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "haan ahe bara pathavto 20 rupaye delivery charge lagel 430 la aahe delivery charge vegla", "answer": {"in_stock": true, "price_inr": 430, "delivery": true, "alternative": null}}
{"product": "Windek tyre", "said": "hallo? haan, awaaz aa rahi hai, bolo, tyre nahin hai, khatam ho gaya, acha Michelin wala de sakta hun, wo 1100 ka padega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Michelin"}}
{"product": "water pump seal kit", "said": "pehle 320 ka tha, abhi hai ki nahi dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "ji boliye, ha, stock mein 22 piece hai, khud aake le jaiye.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "एक मिनिट हां... बोला, हो, आहे, म्हणजे तुम्हाला दीडशेला देतो", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "स्टॉकमध्ये आहे, सातशे साठ... थांबा, लिस्ट बघतो... सहाशे ऐंशी, पाठवायला माणूस नाही, इथे येऊन घ्या.", "answer": {"in_stock": true, "price_inr": 680, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "नमस्कार, बोला, हो, आहे शिल्लक, पावणेतीनशे होतील.", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "hai ji, mil jaega, aapke liye 650, shop 8 baje tak open hai", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "गीज़र पड़ा है, २८०० रुपये का है", "answer": {"in_stock": true, "price_inr": 2800, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Yes yes, I understood, but we don't have it.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ji haan, rakha hai, rate seth ji batayenge", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "No sir, not there.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Only 23 pieces left, anything else?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "माल खत्म है. अरे क्या बताऊँ, सुबह से बहुत भीड़ है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "Yes, go ahead. Yes, it's there. Price is 705. We can deliver.", "answer": {"in_stock": true, "price_inr": 705, "delivery": true, "alternative": null}}
{"product": "Mastech multimeter", "said": "हाँ, माल पड़ा है, 790 प्लस GST", "answer": {"in_stock": true, "price_inr": 790, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "Ha, stock men nau piece hai. rate seth ji batayenge. delivery free hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "MK 16A switch", "said": "हां तर हो, ठेवतो आम्ही, आहे। एक पीस 165, 20 घेतले तर 135 प्रमाणे लावतो।", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "ho aahe, pavaneteenshe plus GST", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "haan, stock me hai, 400 ka hai, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "hello? aawaj yetoy ka? haan bola, aamhi te thevat nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "एक ग्राहक खड़ा है, एक मिनट, मिल जाएगा, है अपने पास, बाज़ार में माल की कमी चल रही है, 400... नहीं नहीं, 440 है, त्योहार का सीज़न है ना", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "bas bara piece bache hain, kal Sunday tha toh dukaan band thi, awaaz kat rahi hai, zor se bolo, 840 last rate hai, Diwali ka season hai na, sab yahi maang rahe hain", "answer": {"in_stock": true, "price_inr": 840, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haa, ready stock hai, nabbe ka padega, bhej deta, par aj ladka nahi aya, aake le jao", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, है, सत्तर में दे देंगे, ठीक है?", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हां बोला, बॅटरी पडलीये, तुम्हाला 1200 ला देतो, बघा पोरगा पाठवतो, अजून काही?", "answer": {"in_stock": true, "price_inr": 1200, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "आहे की, मिळेल। ५६० रुपये होतील। कळवा मग।", "answer": {"in_stock": true, "price_inr": 560, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "Haan toh haan rakhte hain, par abhi khatam hai Usha fan hai, same quality", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Usha fan"}}
{"product": "Crompton 1500W immersion rod", "said": "Haan, hai apne paas, 489 rupaye", "answer": {"in_stock": true, "price_inr": 489, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "haan, stock mein hai, khula ek sao pachasi ka, 6 ka poora box ek hajaar mein, mhanje theek aahe?", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "Haan, wahi wala hai awaaz kat rahi hai, zor se bolo chhabbis sau ka hai, isse kam nahin hoga ek customer khada hai, ek minute are kya bataun, subah se bahut rush hai bata dena", "answer": {"in_stock": true, "price_inr": 2600, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "Han, ready stock hai, do sau paintalis hai, do sau sath nahi, sorry, delivery service nahin hai.", "answer": {"in_stock": true, "price_inr": 245, "delivery": false, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "hai abhi chhah piece bache hain 690 hoga haan toh sir aaj bahut busy day hai network issue hai aawaz break ho rahi hai aur kuch chahiye", "answer": {"in_stock": true, "price_inr": 690, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "oil rakha hai, mil jayega. sadhe panch sau hai, GST alag", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "CSI 1 L geyser", "said": "देखिए हाँ जी, गीज़र रखा है। बताइए क्या करना है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Oil rakha hai, mil jayega, rate 480 hai sir, delivery ho jayegi sir", "answer": {"in_stock": true, "price_inr": 480, "delivery": true, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "Stock khatm ho gaya hai, acha Bajaj heater hai, uska rate 8800", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj heater"}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "आहे आहे, 1380 होतील, घरी पाठवतो, 60 वेगळे", "answer": {"in_stock": true, "price_inr": 1380, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "हॅलो? आवाज येतोय का? हां, बोला, स्टॉक संपलाय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "मशीन संपली.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes, we have 12 pieces. 120 rupees", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "आहे आहे। रेट बदललाय, आता 1100, आधी 910 होता।", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "geyser aahe, milel, tehatisshe pannas padel aata, magchya mahinyat 2840 la hota", "answer": {"in_stock": true, "price_inr": 3350, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "maal sampala Bosch wala ahe, tyache 2890 rupaye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "Retina bike cover", "said": "Ho, thevto aamhi, pan atta samplay. mhanje tyachya aivaji Tradon wala deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Tradon"}}
{"product": "12V 7Ah battery", "said": "बैटरी ख़त्म नहीं हुई, है, एक हज़ार एक सौ रुपये का है.", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "stock madhe ahe packet 435 la ahe, sutta ek 255 la pickup karava lagel", "answer": {"in_stock": true, "price_inr": 435, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "sampala nahi ajun, ahe, 40 hotil.", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "हाँ जी, मिल जाएगा। ५०० रुपये का है", "answer": {"in_stock": true, "price_inr": 500, "delivery": null, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "aahe aahe, kay aahe na 1150 la deto", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "tumhala navvad la deto, bijagri padliye", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "हो, मिळेल। घरपोच देतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "ho bola, ho, milel, 1380 rupaye hotil, bagha gharapoch milel", "answer": {"in_stock": true, "price_inr": 1380, "delivery": true, "alternative": null}}
{"product": "Ritco tap", "said": "hai ji, mil jayega teen sao sath MRP hai, do sau pachis mein ho jayega", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "han ji, boliye, ek customer khada hai, ek min, machine khatam ho gayi, Stanley drill chalega to bolo, pichle hafte bhi kisi ne yahi poocha tha, ha haan, sun raha hun, company ne is mahine phir rate badhaya, dukan pe aake lena padega, bata dena.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Stanley drill"}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "mil jayega, hai panch hazar chhah sau last rate hai.", "answer": {"in_stock": true, "price_inr": 5600, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "ho dada, aahe, teen hajar tinashe pannas... thamba, list baghto... sadatisshe pannas", "answer": {"in_stock": true, "price_inr": 3750, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ho, oil milel.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ हाँ, है। भेजने वाला कोई नहीं है, आ के ले जाओ। ठीक है?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "20 kg नाहीये, वॉल पुट्टी 40 किलो आहे. कंपनीने परत रेट वाढवले. हां हां, ऐकतोय. आज सकाळपासून लाईट नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "वॉल पुट्टी 40 किलो"}}
{"product": "Metro Moulding helmet lock", "said": "Haan haan, hai, 165 ka milega, MYRA lock bhi hai, wo 230 ka, dukan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 165, "delivery": false, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "helmet nahiye, sampla kaka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "१९०० रुपये लगेंगे वायर ख़त्म नहीं हुई है घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 1900, "delivery": true, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "Kya hai na jara velane phone kara, thik hai?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Bola saheb. ahe na. rate 160 aahe.", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Yes, bola bagha ti sampli ho", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "ho ahe, bara donshe panchyaainshi MRP ahe, pan ekshe sattar la deu, gharee pathavto", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "SARK tap", "said": "Yes... one minute, no, that got finished yesterday.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ, माल पड़ा है, देखिए तीन सौ चालीस में दे देंगे.", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप संपलं 40 mm वाला आहे 500 ला", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 mm"}}
{"product": "teflon tape", "said": "Arre hai ji, mil jaega, terah bola kya? galti se bola, pandrah hai", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "तीनशे सत्तर रुपये, कमी नाही होणार, हो, आहे शिल्लक.", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "bagha ho, battery milel. 950 rupayanna padel", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "Waxpol engine oil", "said": "kya hai na hai, bilkul hai, MRP toh 360 hai, apna rate 295.", "answer": {"in_stock": true, "price_inr": 295, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "हां बोला साहेब, मोटर पडलीये, अकराशे पन्नासला देतो", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हो दादा, आहे, म्हणजे एक पीस 70, दहा घेतले तर 61 प्रमाणे लावतो, घरी पाठवतो.", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "haan, ready stock hai, loose piece 460 ka, full packet 725", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "abhi stock mein nahi hai ibell wala chalega toh bolo dekho delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "ibell"}}
{"product": "Raaisin car cover", "said": "ha, ready stock hai, printed MRP 4990, our price 4200", "answer": {"in_stock": true, "price_inr": 4200, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ हाँ, है. अस्सी में दे देंगे", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "hello, bola na. aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes sir, we have, earlier it was 35, now it's 40", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "nahi hai 40 kg wala rakha hai delivery nahi karte aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "40 kg"}}
{"product": "8 mm drill bit", "said": "हो, आहे शिल्लक, तुम्हाला एकशे पन्नासला देतो, डिलिव्हरी फ्री आहे", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "abhi dukaan band kar raha hun kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "लाईट संपली Ridroid headlight आहे त्याचे सतराशे रुपये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Ridroid headlight"}}
{"product": "Rainfun car cover", "said": "Mil jayega, hai, 645 lagega, hai, abhi 25 piece bache hain", "answer": {"in_stock": true, "price_inr": 645, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "machine rakhi hai, mil jayegi, sadhe barah sau plus GST, delivery nahin karte, aake le jao.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "Nahiye, matlab 3 socket board aahe, tyache 390 rupees", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 socket board"}}
{"product": "Shakti 220A welding machine", "said": "ho machine milel bara sade saha hajar ani 18 takke gst vegla porga pathavto", "answer": {"in_stock": true, "price_inr": 6500, "delivery": true, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "hai bhai ji.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हो, ठेवतो आम्ही, आहे, 110 होतील, पोरगा पाठवतो.", "answer": {"in_stock": true, "price_inr": 110, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb aahe milel tumhala 140 la deto", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "ho bola, light sampli nahi, aahe, bagha aaj sakaadi light navhti, kay sangu, sakal pasun gardi ahe, 1960 la deto", "answer": {"in_stock": true, "price_inr": 1960, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "१७५० रुपये होतील। मशीन संपली नाही, आहे। पत्ता पाठवा, पाठवून देतो", "answer": {"in_stock": true, "price_inr": 1750, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "ahe... ek minute, nahi, te kal sampla Fevicol 500 g ahe, same quality.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Fevicol 500 g"}}
{"product": "Philips 5W LED bulb", "said": "स्टॉक में है, एक सौ बीस में दे देंगे, RSCT bulb भी है, वो एक सौ पचपन का.", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "matlab pichle hafte bhi kisi ne yahi poocha tha, bhaiya aap ruko do minute, phone pe hun, abhi nahi hai, Diwali ka season hai na, sab yahi maang rahe hain, ladka chai lene gaya hai, bolo kya karna hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "पता नहीं, गोदाम में देखना पड़ेगा. बाज़ार में माल की कमी चल रही है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "abhi available nahi hai, order place kiya hai, Monday tak aayegaa, 5000 ka milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "हाँ बोलिए हाँ माल पड़ा है 2000 है GST अलग से आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 2000, "delivery": false, "alternative": null}}
{"product": "Bosch horn", "said": "hai, abhi 3 piece bache hain, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "Not out of stock, we have it. Price is 2310.", "answer": {"in_stock": true, "price_inr": 2310, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Haan toh khatam ho gaya tha... nahin, ek bacha hai, 25 ka hai, isse kam nahin hoga.", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "Abhi stock me nahi hai, naya mal Monday ko aayega, ek hazar hoga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "haan bola, aadhi pastis la hota, aata aahe ki nahi baghava lagel.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ, रखा है. 110 का पड़ेगा. भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 110, "delivery": false, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "पहले 1590 का था, अभी है कि नहीं देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "स्टॉक संपलाय.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "बोला, काय पाहिजे? 2999 रुपयांना पडेल। बरं आहे ना। डिलिव्हरी नाही, दुकानात येऊन घ्या।", "answer": {"in_stock": true, "price_inr": 2999, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "Why not, it's available. Price is 290. Pickup only.", "answer": {"in_stock": true, "price_inr": 290, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "Nahi asa nahi, aahe rate 30 aahe ghari pathavto.", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "It's in stock. Price is 100.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "Hello, who is this? I'm busy right now, call in half an hour, there was no power since morning", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "haan sir, boliye, kal subah tak aa jayega, abhi nahi hai, savaa sau ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "Yes sir, tell me. Yes, we have 17 pieces. One customer is here, one minute. Market is short of stock these days. Rs 4670 only. It's festival season, everyone is asking. See, tell me what to do", "answer": {"in_stock": true, "price_inr": 4670, "delivery": null, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "Maal khatam hai. Usha fan chalega toh bolo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Usha fan"}}
{"product": "16A MCB", "said": "stock madhe aahe dada", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "मतलब हाँ हाँ है सात सौ चालीस है GST अलग से", "answer": {"in_stock": true, "price_inr": 740, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Kay aahe na nahi milega bhai, Fevicol 500 g hai, uska rate 390", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Fevicol 500 g"}}
{"product": "9W LED bulb", "said": "bola saheb, bulb samplay, bagha aaj sakali light navhti.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी नाहीये, संपली. अजून काही?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ji haan rakha hai pehle 1680 tha ab 1400 ho gaya sasta hua hai", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "हो, कव्हर आहे, 4620... नाही नाही, 4200.", "answer": {"in_stock": true, "price_inr": 4200, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "sorry sir, abhi out of stock hai, bhaiya aap ruko do minute, phone pe hun.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "नमस्कार, बोला, AmiciTools संपलं, Techtest iron आहे, चालेल? कंपनीने परत रेट वाढवले, बघा बाजारात माल कमी आहे सध्या", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Techtest iron"}}
{"product": "12V 7Ah battery", "said": "स्टॉक में है, पहले 1170 था, अब चौदह सौ पचास हो गया", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "ho ahe chhapil kimmat 145, aamhi 115 la deto.", "answer": {"in_stock": true, "price_inr": 115, "delivery": null, "alternative": null}}
{"product": "Havells Cista room heater", "said": "It's 2260 rupees, but right now no stock.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ek minute... haan bolo rate pachaas ke aas paas hoga, stock malik se puchna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "Hello, yes, I can hear you. We have 8 pieces in stock right now. Rate I'll confirm and tell you", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हो दादा, आहे, आधी सहाशे पंचाण्णव होता, आता पाचशे ऐंशीला देतोय", "answer": {"in_stock": true, "price_inr": 580, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "bagha 4 inch sampla, steel hinge aahe, chaalel?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "steel hinge"}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ हाँ, है। एक सौ पन्द्रह... नहीं नहीं, एक सौ है। होम डिलीवरी हो जाएगी।", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हैलो, कौन? रुकिए, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "एक मिनिट हां बोला स्टॉकमध्ये नाहीये त्याच्याऐवजी Dio air filter देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Dio air filter"}}
{"product": "wall putty 20 kg", "said": "पुट्टी पडलीये, पावणे सहाशे, GST वेगळा", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "माल संपला.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हो मिळेल म्हणजे ४५५ रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "थोड़ी देर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "That we don't have. I can give Dow sealant instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Dow sealant"}}
{"product": "CarMobCentral car cover for Ritz", "said": "दिवाळीची गर्दी आहे ना, हो, कव्हर आहे, सहाशे पंचेचाळीस रुपये, कमी नाही होणार, पोरगा चहा आणायला गेलाय, हां तर बाजारात माल कमी आहे सध्या, आवाज तुटतोय, मोठ्याने बोला", "answer": {"in_stock": true, "price_inr": 645, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "Yes, we have 21 pieces. 119/-. We can deliver.", "answer": {"in_stock": true, "price_inr": 119, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "That we don't have. I can give local putty instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local putty"}}
{"product": "Bajaj 3 L geyser", "said": "geyser rakha hai, mil jayega, 2820 rupees, home delivery available hai, 100 charge", "answer": {"in_stock": true, "price_inr": 2820, "delivery": true, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "ek minute ha... haan bola, aahe, ajun 9 piece padlet, gharpoch milel", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "नमस्कार, बोला, पावणेतीनशेला देतो, स्टॉकमध्ये आहे, संध्याकाळपर्यंत पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "मशीन पडलीये, म्हणजे 5300 रुपयांना पडेल.", "answer": {"in_stock": true, "price_inr": 5300, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "Hai ji, mil jayega, 165 rupaye lagenge, stock me sirf 6 piece hai.", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "Haa, ready stock hai khula ek ek so pachpan ka, poora packet do sau pachpan shaam tak pahuncha denge.", "answer": {"in_stock": true, "price_inr": 255, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "kya hai na kya hai na bijagri aahe ki nahi mahit nahi, godown madhe baghto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ माल पड़ा है दस लगेगा अच्छा दो पीस हैं स्टॉक में", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "nahi hai, Vega helmet hai, uska rate 675, ajun kahi?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vega helmet"}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "bulb pada hai aapko sava chaar sau mein de denge, MRP chhah sau chalis hai bhijwa dunga theek aahe?", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "cover aahe, milel, chhapil kimmat 605, aamhi 385 la deto", "answer": {"in_stock": true, "price_inr": 385, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "Hai bhai, 2020 mein de dunga, arre 9 baje tak main yahin hun, kalva mag", "answer": {"in_stock": true, "price_inr": 2020, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "hai, abhi athara piece bache hain, 1450 ka milega", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "haan haan, hai, 480 hai, 380 nahi", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "आहे ना. १५५० रुपये, कमी नाही होणार. पोरगा पाठवतो, ५० रुपये रिक्षाचे.", "answer": {"in_stock": true, "price_inr": 1550, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "han, tala hai, 230 lagega, 215 wala rate purana hai, wo nahi, arre free delivery hai.", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Sorry, finished. Ball valve is available, 170 rupees. Actually, home delivery is available", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "ball valve"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर पड़ी है. १९०० है, GST अलग से", "answer": {"in_stock": true, "price_inr": 1900, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "नमस्कार बोला 385 ला देतो लाईट पडलीये हां तर पत्ता पाठवा पाठवून देतो", "answer": {"in_stock": true, "price_inr": 385, "delivery": true, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "तुम्हाला पंधराशे पन्नासला देतो, हो, मशीन शिल्लक आहे", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "putty sampli Diwali chi gardi ahe na ek giraik aahe, ek minute 40 kg wala ahe, tyache 940 rupaye thik hai?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 kg"}}
{"product": "water pump seal kit", "said": "जी, उपलब्ध है, मतलब local seal kit का रेट २६५ है, ये वाला ३१०", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "haan bhai, bolo, mil jayega, hai, 1400 mein ho jayega, upar MRP 2000 likha hai, delivery free hai", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "स्टॉक में है. तीन सौ दस और 18 परसेंट GST अलग. मतलब आपको आकर लेना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हो, पुट्टी आहे. 720 रुपयांना पडेल. घरी पाठवतो, 100 वेगळे.", "answer": {"in_stock": true, "price_inr": 720, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "Kami nahi, bharpur aahe, address pathva, pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "rate 810 ahe. kay aahe na ho, thevto aamhi, aahe. address pathava, pathavto. theek aahe?", "answer": {"in_stock": true, "price_inr": 810, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "260 ka padega haan, ready stock hai sorry, delivery service nahi hai.", "answer": {"in_stock": true, "price_inr": 260, "delivery": false, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "Ho aahe, eka nagache 210, 12 nagancha box 2260 la.", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Nahiye, magvun deto, don diwas lagtil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Ek minute ha... haan bola, ti sampli ho, porga chaha aanayla gelay, company ne parat rate vadhavle, kal Ravivar hota mhanun dukan band hota, thread seal tape aahe, 15 la, theek aahe?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "thread seal tape"}}
{"product": "Legrand Mylinc 6A switch", "said": "है भाई, एक पीस एक सौ पचास, 12 लोगे तो 135 के हिसाब से.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "milel na ahe eka nagache pannas, 50 nagancha box 2190 la", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "haan toh hai, bilkul hai, dus loge toh 210 ke hisaab se, ek ka do sau pachas", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Crompton 800W room heater", "said": "ha, maal pada hai rate do hajar pachaas hai bara ghar pe bhej dunga, pachaas lagega bhejne ka", "answer": {"in_stock": true, "price_inr": 2050, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Haan, stock mein hai, home delivery nahi hai bhai", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "हो, मशीन आहे, किंमत नंतर सांगतो, आत्ता लिस्ट नाही, ठीक आहे?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "Ho aahe, 280 aahe, 235 nahi, aur kuch?", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "umm motor nahi hai, khatam ho gayi, AmiciTools wala hai, uska rate 1200", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools"}}
{"product": "Bosch horn", "said": "ek minute ha haan bola bagha nahiye magvun deto don divas lagtil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Windek tyre", "said": "कल सुबह आ जाएगा, अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "Woh hum nahin rakhte, Rico wala chalega? 350 ka hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Rico"}}
{"product": "teflon tape", "said": "Bagha ho dada, ahe, 12 ghetle tar 13 pramane, ek peice 15", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "We have it, yes, Themisto multimeter is also there at 820, and this one is 795, no delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": 795, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan haan, samajh gaya, lekin wo abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "It's in stock. Rs 10 only. Let me know.", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "शिल्लक नाही, Bajaj वाला आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj"}}
{"product": "Modx steering cover for Xcent", "said": "आहे आहे, साडे चारशेला देऊ, छापील किंमत 520 आहे, डिलिव्हरी फ्री आहे.", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Khadija angle grinder", "said": "wo item nahi rakhte hum sir.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "ji sir, tell me, network issue hai, aawaz break ho rahi hai, haan toh sir aaj bahut busy day hai, haa, ready stock hai, MRP to 545 hai, apna rate 440, season time hai na, demand jyada hai.", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "Rs 3,170 only. Basically, yes, we have it. Delivery is free.", "answer": {"in_stock": true, "price_inr": 3170, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ahe... ek minute, nahi, te kal sampala, Pulsar 180 wala ahe, chaalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pulsar 180"}}
{"product": "JY Super LED torch", "said": "No, we don't have it. See, we have Aryshaa torch, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Aryshaa torch"}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "namaskar, bola, cover ahe, milel, rate malakanna vicharun sangto", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "bijagri sampli, sanga kay karaycha", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "जी बोलिए। हाँ तो हैलो? आवाज़ नहीं आ रही, बाद में करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "haan bolo, dekho helmet nahin hai, khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "Yes, we keep that. Delivery is not a problem, we'll send it", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "Ho ho, samajla, pan te nahiye Jupiter clutch plate chalnar asel tar sanga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Jupiter clutch plate"}}
{"product": "Crispy LED torch", "said": "stock madhe aahe 435 nahi nahi 395", "answer": {"in_stock": true, "price_inr": 395, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "Rate ektisshe pannas ahe, ahe na, delivery deto.", "answer": {"in_stock": true, "price_inr": 3150, "delivery": true, "alternative": null}}
{"product": "Cheston angle grinder", "said": "मशीन संपलीये, आत्ता नाहीये, संध्याकाळी येईल, घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "Haan haan, aiktoy, Diwali chi gardi ahe na, mahit nahi, godown madhe baghava lagel, aaj sakali light navhti, company ne parat rate vadhavle", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "haan, stock mein hai, terah sau pachas mein de dunga, delivery ka assi extra", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes yes, available, see, it comes to 110.", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "नमस्ते, बोलिए, पहले 1590 का था, अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, maal pada hai, haan toh 12 ka dabba 1190 ka, ek piece ek so bees, delivery nahi karte, aake le jao.", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "Induro plier", "said": "plas pada hai Agrico plier wala 460 mein aata hai, aapka wala 500 delivery free hai", "answer": {"in_stock": true, "price_inr": 500, "delivery": true, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "Hello, who is this? Not in stock now, it will come tomorrow evening. Market is short of stock these days. It's festival season, everyone is asking. Okay so, there was no power since morning. OK?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "आमच्याकडे आहे, एकशे नव्वद रुपये, कमी नाही होणार, डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "देखिए मिल जाएगा, है अपने पास, सवा नौ सौ प्लस GST.", "answer": {"in_stock": true, "price_inr": 925, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "tube sampli nahi, ahe, MRP 220 ahe, tumhala 185 la deto", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan ha, hai 10 ka hai", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "ji sir, tell me, sorry sir, abhi out of stock hai, ladka chai lene gaya hai, kal Sunday tha toh dukaan band thi, aaj subah se light nahi hai, bolo kya karna hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Han ji, boliye, abhi nahi hai, order diya hai, do din me aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "ek minute ha... haan bola, ho, pakkad milel, kay aahe na aadhi 515 hota, aata 420 la detoy", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Aahe... ek minute, nahi, te kal sampla.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Roots horn", "said": "Hello, yes, I can hear you. Sold out, sir", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "ji boliye, arre bulb nahi hai, khatam ho gaya, Daybetter bulb chalega? do sau tees ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Daybetter bulb"}}
{"product": "50 mm padlock", "said": "हाँ तो ताला ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "2.5 mfd wala khatam hai, 2.25 mfd wala hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.25 mfd"}}
{"product": "10W-30 engine oil 1 L", "said": "hai, bilkul hai, bagha 410 ka padega, mhanje shaam tak pahuncha denge, bata dena", "answer": {"in_stock": true, "price_inr": 410, "delivery": true, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "milel na, aahe, rate sadesahashe aahe, mhanje ghari pathavto, pannas vegle", "answer": {"in_stock": true, "price_inr": 650, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "70 rupaye hotil ho, ahe aamchyakade delivery nahi karat bata dena.", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "aahe ki, milel, savvashe la padel.", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "mahit nahi godown madhe baghava lagel haan haan aiktoy", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "hello, haan boliye, umm available hai, koi problem nahi.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "stock madhe aahe kaka", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "Haan bola, kal Ravivar hota mhanun dukan band hota, aaj sakali light navhti, geyser aahe, milel, 2580 hotil, ek giraik aahe, ek minute, sandhyakali pathavun deto, Diwali chi gardi aahe na", "answer": {"in_stock": true, "price_inr": 2580, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "म्हणजे बॅटरीचा स्टॉक बघून सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ, रखा है, एक पीस 90, 10 लोगे तो 83 के हिसाब से", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Haan ji, boliye khatam ho gaya CPVC wala hai, uska rate 15", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "CPVC"}}
{"product": "Autofurnish car cover for Santro Xing", "said": "हो, आहे शिल्लक, चारशे पस्तीस रुपये, पाठवायला माणूस नाही, इथे येऊन घ्या.", "answer": {"in_stock": true, "price_inr": 435, "delivery": false, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "Haan, ready stock hai, MRP 400 hai, discount ke baad 339 padega, same day delivery kar denge", "answer": {"in_stock": true, "price_inr": 339, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "abhi nahi hai, order diya hai, do din mein aa jaega, aapke liye 1500", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "We don't have it, but I can order it, takes two days", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "हाँ हाँ, है, आपके लिए 2949, ठीक है?", "answer": {"in_stock": true, "price_inr": 2949, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "Ji sir, tell me, khatam ho gaya, umm network issue hai, aawaz break ho rahi hai, sir aaj bahut busy day hai, season time hai na, demand jyada hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "Mil jayega, hai, single piece paune do sau, dus ka full box ek hajaar panch sau pachas, same day delivery kar denge.", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "pankha rakha hai, mil jayega, arre ek hajaar chhah sau hoga, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "मोटर पडलीये रेट ३९०० आहे पाठवतो ५० रुपये डिलिव्हरी चार्ज लागेल", "answer": {"in_stock": true, "price_inr": 3900, "delivery": true, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "Pankha padlay, don hazar rupayanna padel", "answer": {"in_stock": true, "price_inr": 2000, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "hello haan bola tumhala pachshe pandhara la deu ho cover aahe", "answer": {"in_stock": true, "price_inr": 515, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Uh, we have it, yes. 90 rupees. Tell me what to do.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Owner abhi available nahi hain, baad mein call kijiye", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Yes, we have 8 pieces", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "Aahe sir, ready stock aahe, 12 nagancha box don hajar tinashe la, sutta ek savvadonshe", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Ji haan, rakha hai delivery nahin hai", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Haan, wahi wala hai, dekho thik hai?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "संपलं नाही अजून, आहे, तुम्हाला २९५० ला देतो", "answer": {"in_stock": true, "price_inr": 2950, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ho, plug milel, 120 rupayanna padel, local spark plug pahije asel tar te pan aahe, 145, pickup karava lagel", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "ji, available hai rate pandrah hai", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब आहे की नाही माहीत नाही, गोडाऊनमध्ये बघतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan tar ahe ki, milel, rate 50 aahe", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "जी, उपलब्ध है। हमारा रेट ५८०, MRP तो ७०० है", "answer": {"in_stock": true, "price_inr": 580, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब संपला नाही आहे 230 रुपये घरपोच नाही तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 230, "delivery": false, "alternative": null}}
{"product": "HP engine oil", "said": "aahe na, chhapil kimmat 395, aamhi 325 la deto", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "mil jayega, hai, rate saat sau pandrah hai.", "answer": {"in_stock": true, "price_inr": 715, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "bagha haan bola... nahi, te sadhya nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "Out of stock", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "बाजारात माल कमी आहे सध्या। हो, आहे। दिवाळीची गर्दी आहे ना। तुम्हाला 190 ला देतो, MRP 290 आहे। कंपनीने परत रेट वाढवले।", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "देखिए बल्ब ख़त्म हो गया LED वाला दे सकता हूँ 120 रुपये का आपको आकर लेना पड़ेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "LED"}}
{"product": "Sulfar Lutian water pump", "said": "Nahi hai, mangwa dunga, do din lagenge.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "nahi kasa, ahe na, 350 rupayanna padel", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "haan bhai, bolo, abhi stock mein nahi hai, Orient heater hai, barah sau pachas ka, delivery ho jayegi.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Orient heater"}}
{"product": "Ceat Milaze tyre", "said": "kal subah tak aa jayega, abhi nahin hai, khud aake le jaiye", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "जी, उपलब्ध है। रेट मालिक बताएँगे।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "Yes, tell me, okay so, it's in stock, 3800, final price, anything else?", "answer": {"in_stock": true, "price_inr": 3800, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "499 lagega. han haan, hai.", "answer": {"in_stock": true, "price_inr": 499, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "ahe sir, ready stock ahe. terashe fix. sanga kay karaycha.", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "Yes, we keep that. 3,750, final price. We don't deliver", "answer": {"in_stock": true, "price_inr": 3750, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "Yes, we have it, boss.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Hello, good morning. Yes, available. For you, 90. Home delivery is available", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "kya hai na nahi milega bhai bosch wiper hai 630 ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "bosch wiper"}}
{"product": "Activa air filter", "said": "Yes, available. Price changed, it's 290 now, was 260 last month. No delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": 290, "delivery": false, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "आहे ना पावणेपाचशेला देऊ छापील किंमत आठशे पंचेचाळीस आहे", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Bosch wiper blade", "said": "आहे की, मिळेल. छापील किंमत 390 आहे, आम्ही 360 ला देतो. पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 360, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "paach sau pachas rupaye hai, haan toh available hai, koi problem nahi, aur kuch?", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Hello? I can't hear you, call again later.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Ladka dekh ke batayega, ruko zara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "पंखा संपला नाही, आहे. तीन हजार आठशे रुपये", "answer": {"in_stock": true, "price_inr": 3800, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "Haan rakhte hain, par abhi khatm hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Yes sir, tell me, uh, no 10 mm, only the 8 mm one", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "8 mm"}}
{"product": "Cheston 500W drill machine", "said": "नहीं है Bosch drill चलेगा उसका दो हज़ार पचास है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch drill"}}
{"product": "AutoPowerz LED headlight", "said": "nahiye bahutek ho ho sapadla ahe sahashe tees rupayanna padel", "answer": {"in_stock": true, "price_inr": 630, "delivery": null, "alternative": null}}
{"product": "Retina bike cover", "said": "cover aahe ka baghava lagel, nantar sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mufasa 4W LED bulb", "said": "Han, wahi wala hai haan toh 340 ka hai bhijava dunga.", "answer": {"in_stock": true, "price_inr": 340, "delivery": true, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "Yes sir, we have, ₹705, Vega helmet is also there, that one is 770, I'll send it with our boy.", "answer": {"in_stock": true, "price_inr": 705, "delivery": true, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "nahi aahe, Vega wala chalnar asel tar sanga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vega"}}
{"product": "4 inch door hinge", "said": "haan, stock mein hai. assi ka hai, isse kam nahi hoga. haan toh khud aake le jaiye.", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ते आमच्याकडे नसतं 40 किलो वाला आहे आमच्याकडे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 किलो"}}
{"product": "9W LED bulb", "said": "Ha sir, wo model available hai, last month 79 tha, ab 90 hai.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Carbanao car mat for Swift", "said": "हाँ, माल पड़ा है, आपके लिए तेरह सौ पचास, भेज देंगे, 50 रुपये डिलीवरी चार्ज लगेगा.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "nahi vatta... thamba, ahe, ek urlay. 860 hotil. delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 860, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "25 nag shillak ahet", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "price 610 ahe, ho, aahe aamchyakade, porga pathavto", "answer": {"in_stock": true, "price_inr": 610, "delivery": true, "alternative": null}}
{"product": "Avita switch", "said": "haan, mil jayega bhai, teen sau paintalis ka padega", "answer": {"in_stock": true, "price_inr": 345, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes sir, tell me, we have 30 pieces in stock right now, company increased rates again, it's festival season, everyone is asking, for you, 70, it's very busy since morning, we don't deliver", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "बोला काय पाहिजे रेट पावणेनऊशे आहे पण आत्ता स्टॉक नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "Yes yes, available. Printed price is 475, I'll give it at 395", "answer": {"in_stock": true, "price_inr": 395, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "Bulb khatam nahi hua, hai, 249 ka padega, delivery nahi karte, aake le jao, dukaan 10 baje tak khuli hai, ajun kahi?", "answer": {"in_stock": true, "price_inr": 249, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "ho, 26 piece aahet, pachshe sath la deto, delivery deto", "answer": {"in_stock": true, "price_inr": 560, "delivery": true, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "वो तो कल ही ख़त्म हो गया, क्रॉम्पटन फैन है, वो ले लीजिए, लड़का भेज देता हूँ", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "क्रॉम्पटन फैन"}}
{"product": "Siron 60W soldering iron", "said": "haan, hai apne paas bhaiya", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "stock samplay, udya sakali maal yeil, aatta nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "abhi dukaan band kar raha hun, kal subah phone karo sir.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Agrico TATA plier", "said": "aadhi donashe sattar la hota, aata aahe ki nahi baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "Available, sir. I don't know the rate, the owner will tell you.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "हॅलो? आवाज येतोय का? हां, बोला, आवाज तुटतोय, मोठ्याने बोला, काय सांगू, सकाळपासून गर्दी आहे, आहे आहे, हां तर आज सकाळपासून लाईट नाही, 6200 फिक्स, पोरगा पाठवतो, 40 रुपये रिक्षाचे, कळवा मग", "answer": {"in_stock": true, "price_inr": 6200, "delivery": true, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "mhanje ho, bulb ahe. ekshe panchavis plus GST. delivery nahi karat", "answer": {"in_stock": true, "price_inr": 125, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "han, wahi vala hai, best price das", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan ji, boliye, matlab ladka dekh ke batayega, ruko zara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "ho aahe... nahi nahi, to dusra aahe, ha nahiye, AUTOSiTY grip chalnar asel tar sanga.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AUTOSiTY grip"}}
{"product": "Speedwav handle grip", "said": "Hai bhai, 455 lagega", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "Roots horn", "said": "hai, abhi pachees piece bache hain, dekho aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "हां बोला साहेब हां तर आत्ता नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "haan, stock mein hai, ek hajar panch sau hai, sorry sorry, sawa hajar, bhejne wala koi nahi hai, aapko aana padega.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haa ji plug available hai rate puch ke batata hun ghar pe bhej denge bolo kya karna hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "ji haan, rakha hai, gyarah hazar nau sau final price hai, dekho delivery? nahi bhai, wo nahi hota", "answer": {"in_stock": true, "price_inr": 11900, "delivery": false, "alternative": null}}
{"product": "Crasts LED torch", "said": "Bara stock madhe ahe, 825 rupaye, kami nahi honar", "answer": {"in_stock": true, "price_inr": 825, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "नाही कसं, आहे ना. म्हणजे 750 रुपये होतील", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ahe 17ch urlet yeun nyava lagel", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "Bazarat maal kami aahe sadhya, haan haan, aiktoy, ek giraik aahe, ek minute, ti sampli ho, kal Ravivar hota mhanun dukan band hota, kay aahe na company ne parat rate vadhavle, theek aahe?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "han bulb hai full packet 380 loose peice 86 ka", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "machine khatm ho gayi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "VVM bike cover for Honda", "said": "Yes, available. 380 rupees. Shop closes at 7", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Hai na, kitne chahiye? 40 hai, GST alag, bhijwa dunga.", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
{"product": "JCB plier", "said": "Pakkad sampli nahi, ahe, rate 420 aahe sir", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "hello? aawaj yetoy ka? haan bola, ho, 12 piece aahet, rate malakanna vicharun sangto", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "वो हम नहीं रखते आज सुबह से लाइट नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "It's in stock. 80, GST extra", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "han, kaun? acha, bolo, dekhna padega bhai, yaad nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "Cover cha stock baghun sangto. haan tar ek giraik aahe, ek minute", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "ahe na, don hajar pachshe... thamba, list baghto... don hajar shambhar.", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "Khatam nahi hua, abhi bhi hai, dekho paune chaar sau rupaye.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "3000 ka aata hai, lekin abhi khatam hai, abhi stock nahi hai, kal tak aa jayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch drill machine", "said": "ho, thevto aamhi, pan aatta samplay", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "हाँ, माल पड़ा है. खुला एक 330 का, पूरा पैकेट 600.", "answer": {"in_stock": true, "price_inr": 600, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "Hello, good morning. We have it, yes. 370, final price. Let me know", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "Yes sir, we have. Price is 190. See, we can deliver, 80 rupees delivery charge", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "अभी सत्ताईस पीस ही बचे हैं। साढ़े पाँच सौ का पड़ेगा। आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 550, "delivery": false, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "hello, bola na, ho aahe, 410 hotil", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "acha mahit nahi godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "kay aahe na ahe na, rate badalla, aata sahashe pannas, aadhi 580 hota, address pathva, pathavto.", "answer": {"in_stock": true, "price_inr": 650, "delivery": true, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "Abhi nahi hai boss.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "Yes, we have it. 1200, the 1370 rate is old, not that.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "1950 का है, इससे कम नहीं होगा, हाँ हाँ, है", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "rate 460 aahe sir, ho sir, available aahe, delivery nahi, shop varun pickup kara", "answer": {"in_stock": true, "price_inr": 460, "delivery": false, "alternative": null}}
{"product": "AO Smith 15 L geyser", "said": "haa ji, boliye. arre nahi hai ji, khatm", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "स्टॉक में है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "jara velane phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "hello, bola na, ahe, 21ch urlet, ek hazar charshe fix", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "kya hai na aj nahi hai, kal aayega, unnees sau last rate hai, bhejne vala koi nahin hai, aapko aana padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "Bola saheb, ahe, 22ch urlet, company ne parat rate vadhavle, 45 hotil, porga chaha aanayla gelay, bazarat maal kami ahe sadhya, aawaj tutatoy, motyane bola, sanga kay karaycha", "answer": {"in_stock": true, "price_inr": 45, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "ख़त्म हो गया, Havells geyser चलेगा? उसका 2820 है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells geyser"}}
{"product": "Autoplus helmet lock", "said": "हाँ भाई बोलो है ना कितने चाहिए साढ़े तीन सौ में दे देंगे", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan ji boliye rate 790 ke aas paas hoga stock malik se puchna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बरं हॅलो? आवाज येत नाहीये, नंतर करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "haan ji, boliye, abhi stock mein nahi hai, chahiye toh ball valve de dunga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ball valve"}}
{"product": "50 mm padlock", "said": "420 rupaye, tala pada hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 420, "delivery": true, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "जी बोलिए. अभी नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ तो अभी नहीं है, ऑर्डर पर मंगवा देंगे, होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Aptech multimeter", "said": "hello? han, awaaz aa rahi hai, bolo. abhi stock men nahi hai, naya maal Monday ko aayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "mhanje ti sampli ho, company ne parat rate vadhavle, aaj sakaadi light navhti, kay sangu, sakal pasun gardi ahe, 3 inch wala ahe, 70 la", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 inch"}}
{"product": "50 mm padlock", "said": "ताला पड़ा है, 320 का पड़ेगा, मतलब हाँ, 27 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "plate khatam nahin hui, hai, 420 ka padega.", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "Yes sir, tell me. Yes, we keep that. I'll give it for 230, the MRP is 365. Actually, we don't deliver.", "answer": {"in_stock": true, "price_inr": 230, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हो, ठेवतो आम्ही, आहे भाऊ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "कल सुबह आ जाएगा अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Rate 60 hai, par abhi stock nahi hai, sanga kay karaycha.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "abhi stock men nahi hai. awaaz kat rahi hai, zor se bolo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Haan bhaiya, bataiye, ha, wahi wala hai, company ne is mahine phir rate badhaya, aapko number kisne diya? acha theek hai, pehle 32 tha, ab 40 ho gaya", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "देखिए नहीं है, बॉश बिट सेट रखा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉश बिट सेट"}}
{"product": "2 inch paint brush", "said": "haan sir bataiye han ha hai rate abhi pata nahi malik se puchna padega delivery ho jayegi sir theek hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "Maal khatam hai, Havells pump hai, wo lelo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells pump"}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "ji boliye. pankha nahi hai, khatam ho gaya. Bajaj wala chalega to bolo. dekho ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Bajaj"}}
{"product": "4 inch door hinge", "said": "bijagri sampli, sadhya nahiye, order dilay, don divsat yeil, sandhyakali pathavun deto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "We have it, yes. For you, 400.", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "ho, nal milel, tumhala 225 la deto, MRP 360 aahe, gharpoch milel", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "अच्छा मिल जाएगा है अपने पास खुला एक 950 का पूरा पैकेट 1600", "answer": {"in_stock": true, "price_inr": 1600, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "light aahe, milel, ek piece tinashe pannas la milto, packet savva paachashe la, aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ha, kabza hai 30 lagega ab, pehle 34 ka tha.", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "oil padlay delivery nahi ho te nahi karat", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बोला ना, पुट्टी आहे की नाही माहीत नाही, गोडाऊनमध्ये बघतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "हो बोला दादा आमच्याकडे आहे आमचा रेट 555 MRP 635", "answer": {"in_stock": true, "price_inr": 555, "delivery": null, "alternative": null}}
{"product": "Themisto multimeter", "said": "ho dada, aahe. sahashe vees rupaye. address pathva, pathavto. bata dena.", "answer": {"in_stock": true, "price_inr": 620, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ho, ahe aamchyakade, ajun kahi?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "aahe ki, milel, 775 la deto, ho, aekonees piece aahet, sandhyakali pathavun deto", "answer": {"in_stock": true, "price_inr": 775, "delivery": true, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "Basically, we have 23 pieces in stock right now", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया, plastic float valve रखा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "plastic float valve"}}
{"product": "4 inch door hinge", "said": "हो, आहे शिल्लक, एकशे वीस रुपयांना पडेल.", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "है भाई, चार सौ पचास लगेगा, भेजने वाला कोई नहीं है, आ के ले जाओ.", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "haan tar baghto ani kalavto.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "Mil jayega, hai, MRP paach sau nabbe hai, but aapko saadhe chaar sau me de denge.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "Abhi confirm nahi kar sakta, godown chek karna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "tube ahe, milel, bagha sadeaathshe rupaye hotil", "answer": {"in_stock": true, "price_inr": 850, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "ho ahe porga chaha aanayla gelay gharee pathavto 80 vegle haan haan aiktoy maalache 1329 delivery vegli", "answer": {"in_stock": true, "price_inr": 1329, "delivery": true, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "हैलो, कौन? त्योहार का सीज़न है ना, लड़का चाय लेने गया है, आज सुबह से लाइट नहीं है, बाज़ार में माल की कमी चल रही है, पंखा ख़त्म नहीं हुआ, है, अभी रेट उन्नीस सौ पचास हो गया, पिछले महीने 1810 था, ठीक है?", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "It used to be 230, I'll have to check if it's there", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हॅलो? आवाज येतोय का? हां, बोला. 1.5 sq mm नाहीये, 2.5 sq mm wire आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.5 sq mm wire"}}
{"product": "4 inch door hinge", "said": "Available, sir. 90/-.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "hai na kitne chahiye 594 laga denge waise mrp 900 likha hai ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 594, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Ek minute ha... haan bola, aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "Hello, good morning. Basically, yes, it's there. 330 for you, printed price is 375.", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "Aahe, ajun teraa piece padlet, 580 la padel, ghari pathavto", "answer": {"in_stock": true, "price_inr": 580, "delivery": true, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "ek minute... haan bolo, do hajar panch sau mein de dunga, available hai", "answer": {"in_stock": true, "price_inr": 2500, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan, kaun? acha, bolo, malik abhi nahi hain, baad mein phone karo, ek customer khada hai, ek minute, kal Sunday tha toh dukaan band thi, aapko number kisne diya? acha thik hai, market mein maal ki kami chal rahi hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "yes sir, boliye, bulb pada hai, last month tin sau assi tha, ab char sau pachis hai", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Abhi dukaan band kar raha hun, kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "Ji, available hai. ek ka 199, barah ka dabba 2040 ka. kay aahe na khud aake le jaiye.", "answer": {"in_stock": true, "price_inr": 199, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हॅलो? आवाज येत नाहीये, नंतर करा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हो मिळेल घरपोच देतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Abhi nahi, order pe mangwa denge.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "namaskar, bola, ho, milel, 60 fix", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "haan hai... ek minute, nahi, wo kal khatam ho gaya, Osram headlight hai mere paas", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Osram headlight"}}
{"product": "iBELL IBL drill machine", "said": "bola saheb ho machine aahe 4790 rupaye kami nahi honar", "answer": {"in_stock": true, "price_inr": 4790, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "Crompton Greaves संपलं, AmiciTools pump आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools pump"}}
{"product": "1 inch PVC elbow", "said": "Actually, yes, we have it. It comes to 40. Pickup only.", "answer": {"in_stock": true, "price_inr": 40, "delivery": false, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "हां तर नाय हो, नाहीये. त्याच्याऐवजी Crokrok mat देऊ शकतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crokrok mat"}}
{"product": "fan capacitor 2.5 mfd", "said": "Thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "nahi asa nahi, aahe, 20 fix", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "haan bola, kay pahije? te model available nahi, M-seal available ahe, chaalel ka? bara home delivery aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "M-seal"}}
{"product": "Bajaj 3 L geyser", "said": "Available, sir, tell me what to do", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "बघा हो मिळेल सुटा सव्वा तीनशेला सहा नगांचा बॉक्स घेतला तर एक हजार सातशे संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "hello, haa bolo, haan, maal pada hai, rate seth ji batayenge", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JY Super LED torch", "said": "हाँ बोलिए बाज़ार में माल की कमी चल रही है सेठ जी बाहर गए हैं वही बताएँगे लड़का चाय लेने गया है हाँ तो त्योहार का सीज़न है ना कंपनी ने फिर रेट बढ़ा दिया आवाज़ कट रही है ज़ोर से बोलिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "Ha, kaun? acha, bolo, abhi busy hun, aadhe ghante baad phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "bagha haan aahe, tumhala 280 la deto, MRP 340 aahe", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "Yes, tell me. We have 23 pieces in stock right now", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हाँ हाँ, है। 12 लोगे तो 110 के हिसाब से, एक का 119", "answer": {"in_stock": true, "price_inr": 119, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "ek minute... ha bolo, malik abhi nahi hain, bad mein phone karo, ek customer khada hai, ek minut, haa ha, sun raha hun, umm Diwali ka season hai na, sab yahi maang rahe hain.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "haan toh wo item hum keep nahi karte. MK switch hai, same quality.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "MK switch"}}
{"product": "R-tek multimeter", "said": "We have 16 pieces in stock right now, it comes to 455.", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "Yes, we keep that. One piece 220, if you take 10 I'll do 185 each", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "yes available hai price untalis sau pachas hai ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 3950, "delivery": true, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "नहीं है... रुको रुको, है, एक पड़ा है, तीन सौ पचानवे फ़ाइनल, लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 395, "delivery": true, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "Not available right now. We have the Modx one, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Modx"}}
{"product": "AO Smith 15 L geyser", "said": "गिझर पडलाय. 6000 रुपयांना पडेल. दुकान 10 वाजेपर्यंत उघडं आहे.", "answer": {"in_stock": true, "price_inr": 6000, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "aahe ki, milel ata pavaneteenshe ahe, aadhi 340 hota aur kuch?", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Savsol 20W-40 engine oil", "said": "म्हणजे ऑइल संपलं नाही, आहे। 325... थांबा, लिस्ट बघतो... 270", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "संपलंय साहेब।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "haan ha, hai 1600 rupees ka hai kya hai na ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "MK 16A switch", "said": "stock madhe aahe, sutta 165 la, vees nagancha box ghetla tar 2720, porga pathavto, aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 165, "delivery": true, "alternative": null}}
{"product": "Kingsway car cover", "said": "Your voice is breaking, speak loudly. Sold out, sir. There was no power since morning. We have the Autofurnish one, that's 460. My boy has gone for tea. Tell me what to do", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Autofurnish"}}
{"product": "9W LED bulb", "said": "Yes yes, available. ₹90", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "Lagta hai nahi hai... ha mil gaya, hai, paanch sau chalis hoga, bhijwa dunga", "answer": {"in_stock": true, "price_inr": 540, "delivery": true, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "मशीन आहे, मिळेल, रेट कमी झालाय, 2110 होता, आता 1950", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "hello, haan bola, ho, oil milel, bill pahije asel tar GST vegla, rate 430, pickup karava lagel", "answer": {"in_stock": true, "price_inr": 430, "delivery": false, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "yes, bola, light sampliye, bazarat mal kami ahe sadhya, theek ahe?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "I thought it was finished, but there's one left. 330/-.", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "ho, milel, MRP 660 aahe, tumhala 430 la deto", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Nahi vatta... thamba, ahe, ek urlay", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "hello haan awaaz aa rahi hai bolo kay aahe na haan maal pada hai pichasi plus 18 percent gst", "answer": {"in_stock": true, "price_inr": 85, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho, bijagri ahe, 10 nagancha box 745 la, sutta ek 90, bara delivery nahi karat", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "हो, बोला दादा, ट्यूब संपली नाही, आहे, 730 MRP आहे, पण 410 ला देऊ", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "आमच्याकडे आहे डिलिव्हरीची काळजी करू नका पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan toh hai ji, mil jayega, 90 rupaye.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "Machine sampli. Cheston wala aahe, 4700 la", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cheston"}}
{"product": "1 inch PVC elbow", "said": "See, it's not there right now, it's ordered, should come by Monday", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हैलो आवाज़ नहीं आ रही बाद में करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "hai, bilkul hai 5400 plus GST", "answer": {"in_stock": true, "price_inr": 5400, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "220 lagega, han, hai apne paas", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "haan bola ho milel sadesatrashe rupayanna padel", "answer": {"in_stock": true, "price_inr": 1750, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "hello, bola na, ho daada, ahe, aek hazar rupaye.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "हैलो? हाँ, आवाज़ आ रही है, हाँ हाँ, है, 820 MRP लिखा है, 545 लगा देंगे.", "answer": {"in_stock": true, "price_inr": 545, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "जरा वेळाने फोन करा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "Out of stock. My boy has gone for tea. We were closed yesterday, Sunday. OK?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "नमस्कार, बोला, नाहीये, Accessoreez lock चालणार असेल तर सांगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Accessoreez lock"}}
{"product": "Tradon bike cover for Access", "said": "Normally 150, but it's finished. Not in stock now, it will come tomorrow evening", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप आहे मिळेल डिलिव्हरी नाही दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "आहे की, मिळेल, तीनशे पंचाहत्तर होतील, पोरगा पाठवतो, ऐंशी रुपये रिक्षाचे.", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Xtreme MB Windtone horn", "said": "namaste ji, boliye, abhi dukaan band kar raha hun, kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "Out of stock, madam", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "हॅलो? आवाज येतोय का? हां, बोला. हो, मशीन आहे. पाच हजार दोनशे आणि 18 टक्के GST वेगळा", "answer": {"in_stock": true, "price_inr": 5200, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "Sadhya out of stock aahe, new stock parva yenar aahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "hello, bola na, aahe aahe, aapla rate 1420, MRP 2510 aahe", "answer": {"in_stock": true, "price_inr": 1420, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "Yes, go ahead, yes sir, we have, 260 rupees", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bhaiya bataiye nahi hai 9ah wala chalega pandrah sau ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "9ah"}}
{"product": "Ritco tap", "said": "nal hai, mil jaega, address bhej do, bhej dete hain.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "See, it's finished, sorry", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "han sir wo model available hai 1400 lagega bhejne mein koi problem nahi", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "namaskar, bola, ho, thevto aamhi, ahe, aata daha ahe, aadhi 8 hota", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "बोला, काय पाहिजे? आहे ना. रेट 260 आहे", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "hello aawaj yetoy ka haan bola bagha ahe ahe saat hazar naushe rupayanna padel aur kuch", "answer": {"in_stock": true, "price_inr": 7900, "delivery": null, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "abhi available nahi hai, order place kiya hai, Monday tak aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "haan wahi vala hai 330 men ho jayega upar mrp 365 likha hai khud aake le jaiye", "answer": {"in_stock": true, "price_inr": 330, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "Price 438 ahe, ho, milel.", "answer": {"in_stock": true, "price_inr": 438, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टीचा स्टॉक बघून सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "Yes, it's there. Rate I'll confirm and tell you", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "नाही नाही, आहे, बरं चौतीसशेला पडेल.", "answer": {"in_stock": true, "price_inr": 3400, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Battery khatam nahi hui, hai. pandreh sau ka padega. 12V 9Ah battery chahiye toh wo bhi hai, ek hajar saat sau pachas wala. dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Duravolt LED batten", "said": "Ek minute ha... haan bola, tube sampliye, atta nahiye, parva paryant yeyil, ekshe sattar fix", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "हो, हेल्मेट शिल्लक आहे, सव्वा सातशे होतील, पोरगा पाठवतो, तीस रुपये रिक्षाचे", "answer": {"in_stock": true, "price_inr": 725, "delivery": true, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "मिल जाएगा, है अपने पास, एक सौ चालीस लगेगा", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "हां बोला, आज सकाळपासून लाईट नाही, हां हां, ऐकतोय, काय सांगू, सकाळपासून गर्दी आहे, एक गिऱ्हाईक आहे, एक मिनिट, शेठ बाहेर गेलेत, ते आले की सांगतो, ठीक आहे?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "ji boliye, ji han, rakha hai, MRP 660 likha hai, 420 laga denge.", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "दोनशे सत्तर फिक्स। आहे आहे", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "७० ला येतो, पण सध्या संपलाय।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "cover ahe, milel, 615 la deto", "answer": {"in_stock": true, "price_inr": 615, "delivery": null, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "Aamhi te thevat nahi, kal Ravivar hota mhanun dukan band hota.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "ha sir, bataiye. not available sir. uski jagah AmiciTools wala le sakte ho", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools"}}
{"product": "Makita angle grinder", "said": "मशीन आत्ता नाहीये संपली", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "मशीन है, मिल जाएगी. हाँ तो 5700 लगेगा. डिलीवरी का 50 एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 5700, "delivery": true, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "बोला ना। अठ्ठावीसशे पन्नासला देतो। म्हणजे स्टॉकमध्ये आहे।", "answer": {"in_stock": true, "price_inr": 2850, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "हाँ भाई, बोलो, अच्छा चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "ji sir, tell me, yes, available hai, arre price 1230 hai, bata dena", "answer": {"in_stock": true, "price_inr": 1230, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "है ना कितने चाहिए तीस रुपये लगेंगे भेजने वाला कोई नहीं है आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ji, available hai, sadhe teen sau ka hai, isse kam nahi hoga, ghar pe bhej dunga, 100 lagega bhejne ka.", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "बोला, काय पाहिजे? शेठ बाहेर गेलेत, ते आले की सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो? आवाज येत नाहीये, नंतर करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "Hello, yes, I can hear you. Yes, available. That will be Rs 615. Pickup only. OK?", "answer": {"in_stock": true, "price_inr": 615, "delivery": false, "alternative": null}}
{"product": "Goldcave tap", "said": "नमस्ते, बोलिए, पता नहीं, गोदाम में देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "Metro संपलं, Michelin वाला आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Michelin"}}
{"product": "Pulsar 150 spark plug", "said": "हाँ बोलिए। है ना, कितने चाहिए? आपके लिए एक सौ पचास। शाम तक पहुँचा देंगे।", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "बोला, काय पाहिजे? आवाज तुटतोय, मोठ्याने बोला, ती संपली हो, उद्या सकाळी माल येईल, आत्ता नाहीये, हां तर साडेअठराशे रुपये, पोरगा चहा आणायला गेलाय, घरपोच देतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ भाई बोलो आज सुबह से लाइट नहीं है चौदह पीस हैं स्टॉक में एक ग्राहक खड़ा है एक मिनट अरे क्या बताऊँ सुबह से बहुत भीड़ है पचास का पड़ेगा", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "है भाई, अब १२५० है, पहले १०८० था, डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "pata nahi, godown mein dekhna padega bhaiya", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बाजारात माल कमी आहे सध्या बॅटरी आत्ता नाहीये संपली कंपनीने परत रेट वाढवले 9Ah वाला आहे चालेल का कळवा मग", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "9Ah"}}
{"product": "16A MCB", "said": "ते कालच संपलं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "आहे की, मिळेल। म्हणजे पंचावन्नला पडेल।", "answer": {"in_stock": true, "price_inr": 55, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "Nahi hai... ruko ruko, hai, ek pada hai, 235 ka hai.", "answer": {"in_stock": true, "price_inr": 235, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "एक मिनिट हां... बोला। प्लग संपलाय। म्हणजे लोकल वाला घ्या, ते चालेल। घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "लोकल"}}
{"product": "Avita switch", "said": "Han ji, boliye, aapko number kisne diya? acha theek hai, helo? kuch sunai nahi de raha, baad me karo, ek customer khada hai, ek minut, market me maal ki kami chal rahi hai, awaaz kat rahi hai, zor se bolo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "pankha padlay aata naushe panchavann ahe aadhi aathshe tees hota delivery nahi shop varun pickup kara", "answer": {"in_stock": true, "price_inr": 955, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "haan sir, boliye acha haan hai 25 rupaye lagenge.", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "ji, available hai, 200 laga denge, waise MRP 305 likha hai.", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "Ek minute ha... haan bola, bagha ho, machine ahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan bhaiya, bataiye, dekho dekhna padega bhai, yaad nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "kya hai na tera nag shillak aahet sandhyakali pathavun deto sanga kay karaycha.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "आज सकाळपासून लाईट नाही. आवाज तुटतोय, मोठ्याने बोला. कंपनीने परत रेट वाढवले. बघतो आणि कळवतो. काय सांगू, सकाळपासून गर्दी आहे.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी पडलीये, 1150 आहे, 1450 नाही, म्हणजे येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 1150, "delivery": false, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "pankha ahe, milel. rate savva hazar ahe.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "ha sir, boliye, thodi der baad call karo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "tala khatam nahi hua, hai, 60 mm padlock ka rate do sau hai, ye wala do sao assi", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "haan bhai, bolo, pichle hafte bhi kisi ne yahi poocha tha, mal khatm hai, company ne is mahine phir rate badhaya, aur kuch chahiye?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "एक मिनट... हाँ बोलिए, सोलह सौ पचास का पड़ेगा, पंखा पड़ा है", "answer": {"in_stock": true, "price_inr": 1650, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आहे ना, आठशे सत्तर सांगितलं का? चुकलो, सातशे आहे, घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 700, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "ट्यूब संपली नाही, आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "hai hai... nahin yaar, last piece subah bik gaya, Anchor batten hai, wo le lo, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Anchor batten"}}
{"product": "Sonik G-7 helmet", "said": "Hello, bola na, kay aahe na stock madhe aahe, tumhala sadesatshe la deto, MRP aathshe pannas aahe", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "एक ग्राहक खड़ा है, एक मिनट, स्टॉक में है, देखिए चार सौ पंद्रह लगा देंगे, वैसे MRP 480 लिखा है, कंपनी ने फिर रेट बढ़ा दिया, आवाज़ कट रही है, ज़ोर से बोलिए, भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 415, "delivery": false, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "maal sampla, mhanje tyachya aivaji Ravirandal sealant deu shakto, bata dena", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Ravirandal sealant"}}
{"product": "Bosch drill machine", "said": "Ha ji, boliye kya hai na haan, stock mein hai 8300 rupaye lagenge dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 8300, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "We have it, yes, Rs 200 only", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Haan sir, bataiye. haan sir, available hai. 65 rupaye. delivery free hai", "answer": {"in_stock": true, "price_inr": 65, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "kay aahe na nahiye, magvun deto, don divas lagtil, thik hai?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "haan tar nahi kasa, aahe na, pathavayla manus nahi, ithe yeun ghya", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "mahit nahi, godown madhe baghava lagel.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "hello bola na sadhya nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "pankha padlay, mhanje delivery deto, 20 charge lagel, maalache 3200, delivery vegli.", "answer": {"in_stock": true, "price_inr": 3200, "delivery": true, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "हाँ जी मिल जाएगा सात सौ पचपन रुपये का है घर पर भेज देंगे चालीस अलग से", "answer": {"in_stock": true, "price_inr": 755, "delivery": true, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "Yes, available. I'll give it for 1250, the MRP is 2210.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "Bagha milel na, ahe, delivery chi kalji karu naka, pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "hello, ha bolo, haan toh haa ji, mil jaega, chalis hoga, bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "ho pankha milel chovisshe rupees kami nahi honar delivery deto daha vajeparyant me ithech ahe", "answer": {"in_stock": true, "price_inr": 2400, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हां बोला साहेब। वायर संपली नाही, आहे। रेट शेठ सांगतील। डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "I thought it was finished, but there's one left. Pickup only", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "स्टॉकमध्ये आहे.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "matlab nau sau pachpan rupaye cover pada hai.", "answer": {"in_stock": true, "price_inr": 955, "delivery": null, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "Available, sir. 1,390 rupees. We can deliver, 60 rupees delivery charge", "answer": {"in_stock": true, "price_inr": 1390, "delivery": true, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "ek customer khada hai ek minute woh hum nahi rakhte pichle hafte bhi kisi ne yahi poocha tha market mein maal ki kami chal rahi hai aapko number kisne diya acha thik hai agrico plier chalega ek sau nabbe ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "agrico plier"}}
{"product": "Ravirandal silicone sealant", "said": "hello, ha bolo, available hai, 210 ka hai, isse kam nahin hoga, bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "संपलंय, त्याच्याऐवजी 12V 5Ah battery देऊ शकतो, डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "12V 5Ah battery"}}
{"product": "Philips Astra Line 20W LED batten", "said": "आहे ना, ७६० पडेल, ७८०चा रेट जुना आहे, तो नाही", "answer": {"in_stock": true, "price_inr": 760, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "एक मिनिट हां बोला आज सकाळपासून लाईट नाही बाजारात माल कमी आहे सध्या पोरगा चहा आणायला गेलाय टायर आहे का बघावं लागेल नंतर सांगतो एक गिऱ्हाईक आहे एक मिनिट", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हैलो? हाँ, आवाज़ आ रही है, देखिए अभी बिज़ी हूँ, आधे घंटे बाद फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "बोला, काय पाहिजे? ती संपली हो. त्याच्याऐवजी Bosch वाला देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "Anchor LED batten", "said": "ट्यूब आहे, मिळेल, पाठवतो, 60 रुपये डिलिव्हरी चार्ज लागेल, वस्तू 195 ला आहे, डिलिव्हरी वेगळी.", "answer": {"in_stock": true, "price_inr": 195, "delivery": true, "alternative": null}}
{"product": "MYRA helmet lock", "said": "ji boliye ho aahe aamchyakade aapla rate 170 mrp 200 ahe", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "Hello, who is this? My boy has gone for tea. It's 2700 rupees. Yes sir, we have. See, one customer is here, one minute. Market is short of stock these days", "answer": {"in_stock": true, "price_inr": 2700, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हो दादा, आहे, बरं 40 होतील, संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "Haan sir, wo model available hai. matlab price baarah sau hai.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "Shillak nahi saheb", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "ek minute, check karun sangto, ek giraik aahe, ek minute", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "maal kal aayega abhi nahi hai bhejne wala koi nahi hai aapko aana padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "Yes, tell me. Yes, we have it. Price changed, it's 3510 now, was 2810 last month. We'll deliver by evening", "answer": {"in_stock": true, "price_inr": 3510, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "बरं बल्ब संपलाय। LED वाला आहे, चालेल का? डिलिव्हरी करत नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "LED"}}
{"product": "Sulfar Lutian water pump", "said": "acha abhi busy hun aadhe ghante baad phone karna", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "udya sakali maal yeil, aatta nahiye, acha bata dena", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "nahi nahi, aahe, tumhala 1500 la deto.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट पडलीये साहेब.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हो, टेप शिल्लक आहे, बिल हवं असेल तर GST वेगळा, रेट वीस", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "हेल्मेट संपलं नाही, आहे। ४३० ला पडेल।", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "Yes, go ahead. It's in stock. Now 2,260, before it was 1870.", "answer": {"in_stock": true, "price_inr": 2260, "delivery": null, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "नाही ना, संपलंय साहेब", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "ha han, bolo, ha, hai apne paas, rate pachaas hai, dekho theek hai?", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "एक मिनिट हां... बोला. कव्हर संपलं नाही, आहे. 745 ला देतो", "answer": {"in_stock": true, "price_inr": 745, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ahe na kimat nantar sangto aatta list nahi delivery nahi shop varun pickup kara sanga kay karaycha", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Company increased rates again. It's in stock. My boy has gone for tea. We were closed yesterday, Sunday. 1150, final price.", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "aatta nahiye, kalva mag.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "abhi nahi hai, order diya hai, do din me aa jaega, delivery free hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "ho, ahe aamchyakade, sutta ek 8 la, purna paket 60", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "maal kal aayega, abhi nahi hai. umm home delivery nahi hai bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "Haan bola, kay pahije? bara aahe, ajun 7 piece padlet, 140 fix", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "ek customer khada hai, ek minut, Diwali ka season hai na, sab yahi maang rahe hain, haan ji, mil jayega, are kya bataun, subah se bahut rush hai, 745 bola kya? galti se bola, 590 hai, pichle hafte bhi kisi ne yahi poocha tha, thik hai?", "answer": {"in_stock": true, "price_inr": 590, "delivery": null, "alternative": null}}
{"product": "JK 90/100-10 tyre", "said": "It's finished, sorry.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "Yes, we keep that, the box of 10 is 2270, single piece 259, home delivery is available.", "answer": {"in_stock": true, "price_inr": 259, "delivery": true, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हाँ, है। मतलब 625 MRP लिखा है, 415 लगा देंगे। डिलीवरी फ़्री है।", "answer": {"in_stock": true, "price_inr": 415, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "रेट 30च्या आसपास असेल, स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "150 ला पडेल, म्हणजे आहे की, मिळेल, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "bara tube padliye. tumhala charshe tees la deto, MRP 765 aahe. matlab address pathava, pathavto", "answer": {"in_stock": true, "price_inr": 430, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Kam nahin hai, bahut pada hai, aapke liye derh sau", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "Bara hai na, kitne chahiye? 3340 hai, 3270 nahi bagha dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 3340, "delivery": false, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "Yes, tell me. Uh, we have 15 pieces in stock right now. That will be Rs 1160", "answer": {"in_stock": true, "price_inr": 1160, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "wo model available nahi hai, kal Sunday tha toh dukaan band thi, chahiye toh big teflon roll de dunga, arre kya bataun, subah se bahut rush hai, awaaz kat rahi hai, zor se bolo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "Kingsway car mat for Ecosport", "said": "Not in stock now, it will come tomorrow evening. It comes to 2010", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "kay aahe na ahe ahe, aapla rate battisshe pannas, MRP pach hazar tinashe ahe.", "answer": {"in_stock": true, "price_inr": 3250, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "haan sir boliye nahi milega bhai acha chahiye toh local seal kit de dunga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "silicone sealant", "said": "हाँ जी, मिल जाएगा. अभी रेट पौने तीन सौ हो गया, पिछले महीने दो सौ पंद्रह था", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब संपलाय, 12W LED bulb आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12W LED bulb"}}
{"product": "MK 16A switch", "said": "140 rupaye ka hai haan toh haa, mil jaega bhai", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Induro plier", "said": "Yes, go ahead, yes, it's there, 515 plus GST, we don't deliver.", "answer": {"in_stock": true, "price_inr": 515, "delivery": false, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "Cover nahi hai, khatam ho gaya, matlab Cross Fingers cover hai, wo le lo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cross Fingers cover"}}
{"product": "teflon tape", "said": "Mhanje ho, tape milel, panchavis rupayanna padel, pathavayla manus nahi, ithe yeun ghya.", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "ATC bike cover", "said": "pehle ek sao paintalis ka tha, abhi hai ki nahi dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "Hai, bilkul hai, kya hai na 635 ka packet hai, khule bhi hain 410 wale.", "answer": {"in_stock": true, "price_inr": 635, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "मतलब आज सुबह से लाइट नहीं है, हाँ, माल पड़ा है, 310... नहीं नहीं, 275 है, त्योहार का सीज़न है ना, और कुछ?", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ख़त्म हो गया था... नहीं, एक बचा है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "70 la padel, haan tar ho, milel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Speedwav handle grip", "said": "bola saheb, company ne parat rate vadhavle, aaj sakali light navhti, hai, bilkul hai, rate badh gaya, 395 se 495 ho gaya, bhijava dunga, aur kuch?", "answer": {"in_stock": true, "price_inr": 495, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "हॅलो कोण बोलतंय ती संपली हो दिवाळीची गर्दी आहे ना", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "hello, bola na, bagha aatta gadbadit aahe, ardhya tasane phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "हो, कव्हर आहे. MRP १५१० आहे, तुम्हाला ११०० ला देतो. घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 1100, "delivery": false, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "Retina संपलं, V-GRIP वाला आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "V-GRIP"}}
{"product": "Activa clutch plate", "said": "म्हणजे संपलंय. लोकल वाला देऊ शकतो, 610 पडेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल"}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "हाँ बोलिए, आवाज़ कट रही है, ज़ोर से बोलिए, माल खत्म है, कंपनी ने फिर रेट बढ़ा दिया, त्योहार का सीज़न है ना", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हॅलो? आवाज येतोय का? हां, बोला। काय सांगू, सकाळपासून गर्दी आहे। बावीसच पीस उरलेत। हां हां, ऐकतोय। आवाज तुटतोय, मोठ्याने बोला। आज सकाळपासून लाईट नाही। तुम्हाला पंचवीसला देतो। संध्याकाळपर्यंत पाठवून देतो।", "answer": {"in_stock": true, "price_inr": 25, "delivery": true, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "We have it, yes. Actually, I'll give it for 470, the MRP is 625.", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "नहीं है... रुको रुको, है, एक पड़ा है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "udya sakaali mal yeil, aatta nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Yes, it's there, it's 320, not 335", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "हो, मिळेल. 170 सांगितलं का? चुकलो, 190 आहे", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "cover aahe, milel, ek customer khada hai, ek minute, rate badalla, aata 220, aadhi 250 hota, market mein maal ki kami chal rahi hai, arre kya bataun, subah se bahut rush hai, gharpoch milel.", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हाँ भाई, बोलो। देखिए नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "See, owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "मतलब नहीं है... रुको रुको, है, एक पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "हो, आहे शिल्लक", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "hello, bola na, ek hazar pachshe la yeto, pan sadhya samplay", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan bhaiya bataiye tape khatam ho gayi sir aaj bahut busy day hai network issue hai aawaz break ho rahi hai season time hai na demand zyada hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "ji ha, rakha hai. dekho Packy Poda mat bhi hai, uska 1300, aur aap wala 1299 ka", "answer": {"in_stock": true, "price_inr": 1299, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, go ahead. It's festival season, everyone is asking. Out of stock. We were closed yesterday, Sunday.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "Hello, who is this? It's in stock. That will be Rs 165", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "Ji boliye. gyaarah sau pachas ka padega. ladka chai lene gaya hai. company ne is mahine phir rate badhaya. han, wahi vala hai. ek customer khada hai, ek mint. pichle hafte bhi kisi ne yahi poocha tha. bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "नमस्ते, बोलिए. हाँ, रखा है. आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "हाँ, माल पड़ा है, रेट बदल गया है, अब १०५० है, पहले ९१५ था", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "Hai, bilkul hai. khula ek do sau saath ka, poora packet chaar sau painsath. delivery nahin karte, aake le jao", "answer": {"in_stock": true, "price_inr": 465, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "arre malak nahit, nantar phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Haa, maal pada hai, rate puch ke batata hun.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "हाँ भाई, बोलो, स्टॉक में है, १५५० का पड़ेगा, बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Induro plier", "said": "पक्कड पडलीये पाचशे पंचेचाळीस आहे चारशे नव्वद नाही", "answer": {"in_stock": true, "price_inr": 545, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "Haan bola, kay pahije? sahashe tees fix, light aahe, milel, ghari pathavto", "answer": {"in_stock": true, "price_inr": 630, "delivery": true, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "haan ji, mil jayega, 850 ka hai, isse kam nahin hoga, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 850, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "abhi khatm hai, sham tak aa jayega, rate 160 hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ahe ajun 19 piece padlet rate sheth sangtil", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "kya hai na ha hai... ek minute, nahi, wo kal khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "बोला ना, स्टॉकमध्ये आहे, आता सत्तर आहे, आधी एकोणसाठ होता.", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan bola haan tar ho, don piece ahet 720 la padel", "answer": {"in_stock": true, "price_inr": 720, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "अच्छा Crompton Greaves नहीं है Kirloskar वाला है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Kirloskar"}}
{"product": "Wipro 22W LED batten", "said": "umm tube khatam nahi hui hai 430 ka padega delivery ka 100 extra", "answer": {"in_stock": true, "price_inr": 430, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "Sorry, finished, I can give the AmiciTools one instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools"}}
{"product": "BMB welding machine", "said": "machine khatam ho gayi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "मालक नाहीत, नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "aatta sangta yenar nahi, sandhykali phone kara. bazarat maal kami ahe sadhya.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haa ji boliye aath so ka milega ha stock men hai bhejne wala koi nahi hai aapko aana padega", "answer": {"in_stock": true, "price_inr": 800, "delivery": false, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "Not sure, I'll have to check the godown. It's very busy since morning.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "ek minute ha... haan bola, pankha samplay, Orient wala aahe, tyacha price pavanechar hajar, kay aahe na pickup karava lagel.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Orient"}}
{"product": "10W-30 engine oil 1 L", "said": "Sorry, finished", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "Yes, speak, no, we don't have it, actually, market is short of stock these days", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Matlab hai ji, mil jaega, pichle mahine chhiyasath tha, ab pachattar.", "answer": {"in_stock": true, "price_inr": 75, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "wo item nahi rakhte hum, Stanley wala hai, 7200 ka.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Stanley"}}
{"product": "1 inch PVC elbow", "said": "Yes it's... no sorry, that's the other one, this one is not there", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Kami nahi, bharpur ahe.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Plug ahe, milel, rate badalla, ata 120, aadhi 140 hota, yeun nyava lagel.", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "dekho ji haan, rakha hai. pehle 365 tha, ab teen sao ho gaya, sasta hua hai. sorry, delivery service nahi hai.", "answer": {"in_stock": true, "price_inr": 300, "delivery": false, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "हॅलो? आवाज येतोय का? हां, बोला, आधी 945 ला होता, आता आहे की नाही बघावं लागेल.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "han ji, mil jayega, 300 rupaye lagenge, aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "Ji, available hai, rate 510 hai, kya hai na 9 baje tak main yahin hun", "answer": {"in_stock": true, "price_inr": 510, "delivery": null, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "haan, hai apne paas matlab pehle 170 tha, ab 185 ho gaya", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "जी बोलिए, मतलब ७८० का पड़ेगा, पुट्टी पड़ी है.", "answer": {"in_stock": true, "price_inr": 780, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "हेल्मेट पडलंय", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "आहे की, मिळेल भाऊ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "बघा कव्हर संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "बघा टेप पडलीये, रेट वीस आहे, big teflon roll हवं असेल तर ते पण आहे, 16", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "बघा हो आहे... नाही नाही, तो दुसरा आहे, हा नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "है भाई। डिलीवरी का 100 एक्स्ट्रा। सामान तेरह सौ पचास का है, डिलीवरी अलग से", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "Goodyear tyre", "said": "Yes sir, boliye, bhaiya aap ruko do min, phone pe hun, pichle hafte bhi kisi ne yahi poocha tha, stock men sirf tis piece hai, best price chhattis sau pachas, ek customer khada hai, ek minute, free delivery hai, aur kuch?", "answer": {"in_stock": true, "price_inr": 3650, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan, stock mein hai, 660 bola kya? galti se bola, aath so bees hai", "answer": {"in_stock": true, "price_inr": 820, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "Haan haan, bolo, savaa teen hajar ka hai, isse kam nahi hoga, hai ji, mil jayega, kya hai na address bhej do, bhej dete hain.", "answer": {"in_stock": true, "price_inr": 3250, "delivery": true, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "kal Sunday tha to dukan band thi, chaar sau pachas last rate hai, awaaz kat rahi hai, zor se bolo, ha hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "हां बोला साहेब, बरं बिजागरी संपलीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Benz helmet", "said": "Yes, available. 855, final price. We don't deliver.", "answer": {"in_stock": true, "price_inr": 855, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "हो, आहे शिल्लक. ६० ला देतो. अजून काही?", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Hella horn", "said": "bara nahi hai ji, khatam. Favourite Bikerz wala rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Favourite Bikerz"}}
{"product": "Rainfun car cover", "said": "आहे की, मिळेल, पाचशे ऐंशी रुपये, बरं डिलिव्हरी फ्री आहे", "answer": {"in_stock": true, "price_inr": 580, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "आहे की, मिळेल, तीस पडेल, 24चा रेट जुना आहे, तो नाही", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Rs 40 only. Yes sir, we have", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "Okay so, no... wait, yes, we have one piece left, it comes to 1220, no delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": 1220, "delivery": false, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "Yes sir, we have. Okay so, we'll send it, 50 for delivery. Item price is 1390, delivery is extra.", "answer": {"in_stock": true, "price_inr": 1390, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "Hello? Yes, available. 110, final price. OK?", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "हाँ बोलिए. त्योहार का सीज़न है ना. रेट दो हज़ार तीन सौ है. लड़का चाय लेने गया है. कंपनी ने फिर रेट बढ़ा दिया. हाँ जी, मिल जाएगा. लड़का भेज देता हूँ.", "answer": {"in_stock": true, "price_inr": 2300, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "टेप संपली नाही, आहे, १५ होतील, घरपोच देतो, दुकान ८ वाजेपर्यंत उघडं आहे.", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "R-tek multimeter", "said": "हाँ जी मिल जाएगा सात सौ MRP लिखा है चार सौ तीस लगा देंगे डिलीवरी नहीं करते दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 430, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haan ji, plug available hai, ab 180 hai, last month 140 tha.", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "नल ख़त्म हो गया Goldcave वाला चलेगा उसका २२० है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Goldcave"}}
{"product": "Activa air filter", "said": "है ना, कितने चाहिए? रेट 190 है, भेजने में कोई दिक्कत नहीं", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "गिझर पडलाय, 2600 आहे, 2380 नाही, संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 2600, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "assi last rate hai, ha, hai apne paas, aapko number kisne diya? acha thik hai, Diwali ka season hai na, sab yahi maang rahe hain, delivery free hai.", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "आत्ता नाहीये, संध्याकाळी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "ho bola, maal sampala, Aryshaa torch aahe, ekshe navvad la, gharpoch milel", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Aryshaa torch"}}
{"product": "Favourite Bikerz FBZ horn", "said": "haan bolo aaj nahi hai, kal aayega aapko number kisne diya? acha thik hai kal Sunday tha toh dukan band thi 340 ka hai aj subah se light nahin hai company ne is mahine phir rate badhaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "hai bhai, rate abhi pata nahi, malik se puchna padega, home delivery nahin hai bhai.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "helmet padlay, tumhala charashe panchyannav la deto, MRP pachshe pasasht aahe", "answer": {"in_stock": true, "price_inr": 495, "delivery": null, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "ते मिळणार नाही इथे. बरं क्रॉम्प्टन वाला चालणार असेल तर सांगा. डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "क्रॉम्प्टन"}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "hello, haa bolo Diwali ka season hai na, sab yahi maang rahe hai ha ha, hai last time 570 tha, abhi 715 hai dukaan pe aake lena padega pichle hafte bhi kisi ne yahi poocha tha.", "answer": {"in_stock": true, "price_inr": 715, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "है भाई। 100 रुपये लगेंगे। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए।", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "पहले २८५० का था, अभी है कि नहीं देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "नहीं नहीं, है ना", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "Sold out, sir. Actually, the Speedwav one is there, you can take that", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Speedwav"}}
{"product": "2 inch paint brush", "said": "No, we don't have it. We have paint roller, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "paint roller"}}
{"product": "Studds Urban helmet", "said": "hello? aawaj yetoy ka? haan bola, ho, helmet ahe, aathshe pandhra rupaye.", "answer": {"in_stock": true, "price_inr": 815, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "Yes, it's there, printed price is 1580, I'll give it at 1150", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "Basically, not today, it comes tomorrow", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "pankha hai ki nahi, dekh ke batata hun.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "मशीन पडलीये। बरं रेट शेठ सांगतील। पाठवायला माणूस नाही, इथे येऊन घ्या।", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Mihsp water pump", "said": "Not available sir", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "bijagri aahe, milel, rate malakanna vicharun sangto, aur kuch?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Haan sir, bataiye, abhi busy hun, aadhe ghante baad phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "nahi aahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "हाँ जी, मिल जाएगा। GST मिलाकर 1520। भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 1520, "delivery": false, "alternative": null}}
{"product": "Orient ABS room heater", "said": "Yes, bola, owner nahit, nantar call kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "कमी नाही, भरपूर आहे, पावणे दोनशेला देतो", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "matlab haa, mil jayega bhai. 753 hai, 580 nahi. bhejne wala koi nahi hai, aapko aana padega", "answer": {"in_stock": true, "price_inr": 753, "delivery": false, "alternative": null}}
{"product": "Aegon angle grinder", "said": "Kay aahe na dedh hajar rupaye, mhanje machine padi hai.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "हॅलो? आवाज येतोय का? हां, बोला। मशीन आहे का बघावं लागेल, नंतर सांगतो। हां तर आज सकाळपासून लाईट नाही", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Nahi milega bhai, aur kuch?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "बघा आमच्याकडे आहे, 50 नगांचा बॉक्स 1290 ला, सुटा एक 30", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "umm aaj subah se light nahi hai. helmet nahi hai, khatam ho gaya. bhaiya aap ruko do min, phone pe hun. ladka chai lene gaya hai. ek customer khada hai, ek minat", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ हाँ, है, 330 का पड़ेगा, local air filter भी है, वो 295 का, डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "stock madhe ahe, tumhala 440 la deu, delivery nahi, shop varun pickup kara, kalva mag.", "answer": {"in_stock": true, "price_inr": 440, "delivery": false, "alternative": null}}
{"product": "Bosch wiper blade", "said": "नहीं है जी, ख़त्म", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "hello? aawaj yetoy ka? haan bola, ahe na, aapla rate char hazar pannas, MRP 4810 ahe, sandhyakali pathavun deto", "answer": {"in_stock": true, "price_inr": 4050, "delivery": true, "alternative": null}}
{"product": "Crompton 800W room heater", "said": "Ha sir, bataiye, available hai, koi problem nahi, last month 1330 tha, ab 1750 hai, delivery ho jayegi sir", "answer": {"in_stock": true, "price_inr": 1750, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ बोलिए, मालिक नहीं हैं, बाद में फ़ोन कीजिए.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "Ho, pankha aahe, atharashe rupaye hotil.", "answer": {"in_stock": true, "price_inr": 1800, "delivery": null, "alternative": null}}
{"product": "Benz helmet", "said": "haan sir bataiye wo model available nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "Aaj subah se light nahi hai, thodi der baad call karo, haan haan, sun raha hun, bolo kya karna hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "ek minute... haan bolo. haan toh abhi khatam hai bhai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "hello? aawaj yetoy ka? haan bola wire sampli", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ रखते हैं पर अभी ख़त्म है उसकी जगह 5Ah वाला दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "5Ah"}}
{"product": "Accessoreez helmet lock", "said": "ho sir, available ahe. eka nagache 280, daha nagancha box 2400 la. home delivery aahe", "answer": {"in_stock": true, "price_inr": 280, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "nahi nahi, hai na.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "helmet pada hai, apna rate char sau pachas, MRP toh 600 hai.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "Ji, available hai, haan toh aapko paune panch sau men milega, MRP 545 hai, aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "बरं आमच्याकडे नसतं, ऑर्डर करून मागवतो, येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fluke multimeter", "said": "Owner abhi available nahi hain, baad mein call kijiye.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हो दादा, आहे, पासष्टला पडेल, आठ वाजेपर्यंत मी इथेच आहे", "answer": {"in_stock": true, "price_inr": 65, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "Bhaiya aap ruko do minute, phone pe hun, haa haan, sun raha hun, kal Sunday tha to dukaan band thi, hai bhai, 365 hoga, delivery ho jayegi, aapko number kisne diya? acha thik hai", "answer": {"in_stock": true, "price_inr": 365, "delivery": true, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "ek minute ha haan bola te kalach sampla havells wala ghya te chalel porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "havells"}}
{"product": "Activa clutch plate", "said": "Haan bolo, plate aahe, milel, umm 900 hotil.", "answer": {"in_stock": true, "price_inr": 900, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी संपली नाही, आहे। 1150 ला देतो", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "Aahe, 20ch urlet, rate malakanna vicharun sangto", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Can you call after some time?, boss.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "haan bola kay pahije ahe ki milel sadeteenshe la padel ajun kahi", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "No no, it's there, see, Rs 200 only.", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Available, sir. Price is 80. I'll send it with our boy", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Cheston angle grinder", "said": "एक मिनिट हां... बोला, आहे की, मिळेल, तुम्हाला एक हजार चारशेला देतो, पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": 1400, "delivery": false, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "Actually, it used to be 935, I'll have to check if it's there.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Kyun nahi milega, hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Yes, we have it. We can deliver, 20 rupees delivery charge. 340 for the item, plus delivery", "answer": {"in_stock": true, "price_inr": 340, "delivery": true, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "available hai, bhejne mein koi problem nahi", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ जी, बल्ब रखा है. भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो कोण बोलतंय आहे की मिळेल 120 रुपयांना पडेल पत्ता पाठवा पाठवून देतो", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "haan, wahi wala hai, matlab 190 plus GST.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "haan, stock mein hai, 60 rupaye lagenge, delivery ka 20 extra.", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हो, बल्ब आहे, MRP 335 आहे, तुम्हाला 285 ला देतो, घरपोच देतो", "answer": {"in_stock": true, "price_inr": 285, "delivery": true, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "Namaskar, bola kay aahe na samplay, udya dupari yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Haan tar ho sir, available aahe, rate 250 aahe sir, delivery deto sir, aath vajeparyant mi ithech aahe.", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "सवा पाँच सौ का है, इससे कम नहीं होगा। जी, उपलब्ध है।", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "Ho, 28 piece aahet, pachshe pannas rupayanna padel", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "माल संपला, सांगा काय करायचं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "I thought it was finished, but there's one left. See, that will be Rs 170", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "hello? ha, awaaz aa rahi hai, bolo, haa, mil jayega bhai, printed MRP che sau tees, our price paach sau bees, khud aake le jaiye", "answer": {"in_stock": true, "price_inr": 520, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "Hello? aawaj yetoy ka? haan bola stock samplay M-seal ghya, te chalel gharapoch milel", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "M-seal"}}
{"product": "water pump seal kit", "said": "Hello? ha, awaaz aa rahi hai, bolo, hai bhai, 340 final, delivery nahi karte, aake le jao.", "answer": {"in_stock": true, "price_inr": 340, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "Haan aahe, delivery chi kalji karu naka, pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "haa bhai, bolo, haan, mil jaega bhai, aapko sava chaar sau me de denge, MRP 755 hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "पता नहीं, गोदाम में देखना पड़ेगा, अच्छा आवाज़ कट रही है, ज़ोर से बोलिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "haan bhai, bolo, bhaiya aap ruko do mint, phone pe hun, haan ha, sun raha hun, awaaz kat rahi hai, zor se bolo, kal subah tak aa jayega, abhi nahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "han sir, wo model available hai, home delivery available hai, 50 charge, item price chah sau tees, delivery charge alag", "answer": {"in_stock": true, "price_inr": 630, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "बोला ना, स्टॉकमध्ये आहे, हां तर वीस रुपये.", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Yes, we keep that. I don't know the rate, the owner will tell you. We don't deliver. Uh, anything else?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "लड़का चाय लेने गया है, आज सुबह से लाइट नहीं है, कंपनी ने फिर रेट बढ़ा दिया, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, BMB नहीं है, Vormir machine है, ठीक है?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vormir machine"}}
{"product": "wall putty 20 kg", "said": "पुट्टी संपली नाही, आहे, तुम्हाला साडेनऊशेला देतो, हां तर 8 वाजता दुकान बंद होतं", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हो, ठेवतो आम्ही, आहे. एका नगाचे १२०, २० नगांचा बॉक्स २१३० ला", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "बावीस नग शिल्लक आहेत. पावणेचारशे होतील", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "आहे की, मिळेल. तीनशे पंधराला पडेल.", "answer": {"in_stock": true, "price_inr": 315, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "मतलब बल्ब है, मिल जाएगा, एक पीस ३१०, ६ लोगे तो २६० के हिसाब से", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "Haan bola, kay pahije? bazarat maal kami aahe sadhya, porga chaha aanayla gelay, ho, cover aahe, 255 fix, theek aahe?", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "acha han sir, available hai, aapko panch sau pachpan mein milega, MRP 860 hai.", "answer": {"in_stock": true, "price_inr": 555, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Activa nahi hai, Dio wala hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Dio"}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "Ha sir, boliye, haan, maal pada hai, paune chaar sau final", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "हॅलो, कोण बोलतंय? आहे ना। आमचा रेट 4500, MRP 5350।", "answer": {"in_stock": true, "price_inr": 4500, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हॅलो, कोण बोलतंय? उद्या सकाळी माल येईल, आत्ता नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "Ho bola nahi milanar bazarat maal kami aahe sadhya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "pankha sampla nahi, aahe, sadenaushe hotil, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 950, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "दिवाळीची गर्दी आहे ना, तेराशे पन्नास रुपये, बॅटरी पडलीये, बाजारात माल कमी आहे सध्या, आवाज तुटतोय, मोठ्याने बोला, पत्ता पाठवा, पाठवून देतो, कळवा मग", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "Ho, aahe aamchyakade, eka nagache ekshe daha, 10 nagancha box naushe pannas la.", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "ji han rakha hai 150 ka ek, 25 loge toh 125 ke hisaab se laga dunga", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "bagha hai na, kitne chahiye? sava do sau rupaye, thik hai?", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "हो, १६ पीस आहेत, हां तर २२०० होतील", "answer": {"in_stock": true, "price_inr": 2200, "delivery": null, "alternative": null}}
{"product": "Hella horn", "said": "Bara hai, bilkul hai, sawa nau sau mein de dunga, matlab bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 925, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "हां बोला हेल्मेट आहे का बघावं लागेल नंतर सांगतो दिवाळीची गर्दी आहे ना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Aapko number kisne diya? acha theek hai, haan, ready stock hai, kya hai na aapke liye sava sau, kal Sunday tha toh dukan band thi, home delivery available hai, bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "Havells Cista room heater", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया। Orient heater है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Orient heater"}}
{"product": "10W-30 engine oil 1 L", "said": "haan bolo... nahi, wo nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "Okay so, yes yes, available. MRP 605, but our price is 385", "answer": {"in_stock": true, "price_inr": 385, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "kya hai na ladka dekh ke batayega, ruko zara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "Maal khatam hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ बोलिए २६० का आता है लेकिन अभी ख़त्म है शाम तक माल आ जाएगा अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "haan bolo... nahi, wo nahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "Ha bhaiya, bataiye, thodi der bad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "आमच्याकडे आहे, चौदाशे पन्नास रुपये, पाठवतो, वीस रुपये डिलिव्हरी चार्ज लागेल", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "Yes, we keep that, MRP is 2,210, we'll give it for 1250.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ती संपली हो, म्हणजे त्याच्याऐवजी 2.5 sq mm वाला देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.5 sq mm"}}
{"product": "Philips 5W LED bulb", "said": "बल्ब संपला नाही, आहे। तुम्हाला 125 ला देतो, MRP 155 आहे। पाठवायला माणूस नाही, इथे येऊन घ्या।", "answer": {"in_stock": true, "price_inr": 125, "delivery": false, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "ha sir, bataiye, abhi khatam hai bhai, network issue hai, aawaz break ho rahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "nahin milega bhai, Lakshmi pump hai, 2950 ka.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Lakshmi pump"}}
{"product": "4 socket extension board", "said": "520 lagega, haan sir, wo model available hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 520, "delivery": true, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "Abhi stock me nahi hai, naya mal Monday ko ayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ho sir bola company ne parat rate vadhavle bazarat maal kami aahe sadhya mhanje ho sir available ahe ek giraik ahe ek minute 235 nahi nahi savvadonshe theek ahe", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "Yes sir, we have, actually, price is 1,500, delivery is 50 extra.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "ऑइल संपलं. Mobil oil आहे, १२०० ला", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Mobil oil"}}
{"product": "AutoClean wiper blade", "said": "Aahe aahe, aapla rate tinashe panchechalis, MRP pachshe sattar aahe, delivery deto", "answer": {"in_stock": true, "price_inr": 345, "delivery": true, "alternative": null}}
{"product": "Bosch wiper blade", "said": "हाँ हाँ, है, मतलब आपको ३४५ में दे देंगे, MRP ३७५ है, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 345, "delivery": false, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "हाँ तो कवर है मिल जाएगा लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "ahe... ek minute, nahi, te kal sampala, AutoPowerz headlight aahe aamchyakade, delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "AutoPowerz headlight"}}
{"product": "Pulsar 150 spark plug", "said": "Ho aahe. daha ghetle tar 150 pramane, ek piece 170.", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "कमी नाही, भरपूर आहे, दोनशे साठ रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Hello, good morning, yes yes, available, ₹190, we can deliver, my boy has gone for tea, your voice is breaking, speak loudly, it's festival season, everyone is asking, anything else?", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "pankha sampla nahi, aahe savva teen hazar fix", "answer": {"in_stock": true, "price_inr": 3250, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "है भाई, 1450... नहीं नहीं, 1270 है.", "answer": {"in_stock": true, "price_inr": 1270, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "hai, bilkul hai. MRP chah sau panch hai, par teen sau painsath de dena.", "answer": {"in_stock": true, "price_inr": 365, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "मालिक नहीं हैं बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "Han sir, boliye, pankha hai, mil jayega, chah hajar final, address bhej do, bhej dete hain", "answer": {"in_stock": true, "price_inr": 6000, "delivery": true, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "एकोणीसशे पन्नास रुपये, कमी नाही होणार, पंखा पडलाय, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "It's in stock. Uh, our price is 175, MRP is 315", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "मिल जाएगा है अपने पास अब दो सौ दस है पहले 190 था आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 210, "delivery": false, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "Ho, tube milel. pavanesatshe rupaye. gharpoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 675, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "माल संपला बरं उद्या सकाळी माल येईल आत्ता नाहीये एक हजार पन्नासला देतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "बल्ब संपला नाही, आहे, म्हणजे २६० रुपये, कमी नाही होणार", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "ho, aahe aamchyakade. packet 80 la aahe, sutta ek 9 la", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "म्हणजे थांबा, बघून सांगतो।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "battery khatam ho gayi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "जी बोलिए, पता नहीं, गोदाम में देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "हो, बोला दादा। आहे की, मिळेल। आता पस्तीसशे आहे, आधी तीन हजार होता। बरं डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": 3500, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Hai hai... nahi yaar, last peice subah bik gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "Haa bhai, bolo, haan, stock me hai, delivery free hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Anchor LED batten", "said": "हो दादा, आहे। MRP दोनशे दहा आहे, तुम्हाला पावणेदोनशेला देतो। घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Aaj subah se light nahi hai, hai bhai, Diwali ka season hai na, sab yahi maang rahe hain, matlab han haa, sun raha hun, abhi rate panch sau chalis ho gaya, pichle mahine chaar sau pachasi tha, bhaiya aap ruko do min, phone pe hun, aur kuch?", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "haan toh hai, bilkul hai, do hajar nau sau pachas... nahi nahi, teen hajar hai", "answer": {"in_stock": true, "price_inr": 3000, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "1100 rupayanna padel battery padliye gharapoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 1100, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "Ek mint... haan bolo, arre nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Hella horn", "said": "hello? kuch sunai nahi de raha, baad me karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हॅलो? आवाज येतोय का? हां, बोला, बल्ब संपलाय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हॅलो कोण बोलतंय ती संपली हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "pehle 2950 ka tha abhi hai ki nahi dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "No... wait, yes, we have one piece left, Rs 1400 only", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "arre haa rakhte hain par abhi khatm hai retina cover hai mere paas", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "retina cover"}}
{"product": "Accessory Mart car mat for Duster", "said": "ji sir, tell me, haan, ready stock hai, aapke liye 3510, aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": 3510, "delivery": false, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "गिझर संपलाय, Bajaj geyser आहे आमच्याकडे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj geyser"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "No sir, not there. We have 8 mm bolt, same thing.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "8 mm bolt"}}
{"product": "12V 7Ah battery", "said": "Ji sir, tell me, haan, ready stock hai, pandra sau ka hai, sorry, delivery service nahi hai.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "बोला, काय पाहिजे? रेट 190 आहे. म्हणजे आहे की, मिळेल", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "अभी नहीं है, local glue है, तीन सौ साठ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local glue"}}
{"product": "2 inch paint brush", "said": "haan bola, kay pahije? ho, milel, sattar fix", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "कंपनीने परत रेट वाढवले, आवाज तुटतोय, मोठ्याने बोला, आज सकाळपासून लाईट नाही, काय सांगू, सकाळपासून गर्दी आहे, आज नाही, उद्या येईल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "संपलंय साहेब.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Yes, it's there. Now 2100, before it was 2,600.", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "aahe ek minute nahi te kal sampala", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ जी, बताइए. हाँ, है. रेट पूछ कर बताता हूँ. मतलब होम डिलीवरी हो जाएगी. ठीक है?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "संपले सगळे, म्हणजे हॅवेल्स वाला आहे, अकराशे पन्नासला, डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "हॅवेल्स"}}
{"product": "Activa clutch plate", "said": "ti sampli ho, udya sakali maal yeil, aatta nahiye.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "nahiye bahutek... ho ho, sapadla, ahe. sandhyakali pathavun deto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "RNG Eko tyre", "said": "बोला ना, टायर संपलाय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ तो है भाई.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "Yes, it's there. Printed price is 200, I'll give it at 160. We don't deliver", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "रेट सहाशे साठ आहे, पण आत्ता स्टॉक नाही.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "1550 ka hai, haa, wahi wala hai", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हो, ठेवतो आम्ही, आहे, तुम्हाला 230 ला देतो, घरपोच देतो, 9 वाजेपर्यंत मी इथेच आहे", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "नाही वाटतं... थांबा, आहे, एक उरलाय, एकशे नव्वद रुपये.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "stock me sirf solah piece hai rate puch ke batata hun same day delivery kar denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bosch wiper blade", "said": "hello, ha boliye, matlab dekhna padega bhai, yaad nahi hai, pichle hafte bhi kisi ne yahi poocha tha", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "Hai na, kitne chahiye? 6 ka full box 1580, single piece 300 home delivery available hai", "answer": {"in_stock": true, "price_inr": 300, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "Ji boliye 90 final hai bhai bhejne wala koi nahi hai, aapko aana padega company ne is mahine phir rate badhaya Diwali ka season hai na, sab yahi maang rahe hain arre kya bataun, subah se bahut rush hai thik hai?", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "Kay aahe na ho, ahe aamchyakade, rate sheth sangtil, delivery deto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "MK 16A switch", "said": "ji available hai 220 bola kya galti se bola pone do sau hai dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "वो हम नहीं रखते, आवाज़ कट रही है, ज़ोर से बोलिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "Ek mint... haa bolo, mil jayega, hai, aapke liye unnis sau pachas, haan toh address bhej do, bhej dete hai", "answer": {"in_stock": true, "price_inr": 1950, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "plate sampli nahi, aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "ho sir, bola, 16A nahiye, 20A wala aahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "20A"}}
{"product": "12V 7Ah battery", "said": "yes, available hai, ladka de aayega, 40 rupaye auto ka, maal ka 1200, delivery alag", "answer": {"in_stock": true, "price_inr": 1200, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "It's very busy since morning. Actually, company increased rates again. We were closed yesterday, Sunday. 4 socket is not there, we have 3 socket board. It's festival season, everyone is asking.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 socket board"}}
{"product": "Retina steering cover for Alto", "said": "Actually, 290/-. We have it, yes", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "1520 final ji, available hai.", "answer": {"in_stock": true, "price_inr": 1520, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "Machine padi hai. 2200 plus GST. bara khud aake le jaiye", "answer": {"in_stock": true, "price_inr": 2200, "delivery": false, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "haan ready stock hai pehle 1150 tha ab terah sau pachas ho gaya address bhej do bhej dete hain", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Hai, bilkul hai, 950 rupees, theek aahe?", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "अच्छा हाँ हाँ, समझ गया, लेकिन वो अभी नहीं है। local brake shoe है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local brake shoe"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "jara velane phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Hello, yes, I can hear you, we have it, yes, that will be Rs 370.", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "आहे ना, बाराशे रुपये, येऊन न्यावं लागेल.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": false, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "Ha, kaun? acha, bolo, machine hai, mil jayegi, 3600 hoga", "answer": {"in_stock": true, "price_inr": 3600, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "Aamchyakade nasta te", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "Haa bhaiya, bataiye, umm bas 10 piece bache hain, dukan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "आज नहीं है, कल आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "Ho bola. haan tar ahe ki, milel. aadhi ekshe vees hota, aata deedashe zala", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "४५०० होतील, हो, मोटर आहे", "answer": {"in_stock": true, "price_inr": 4500, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर नहीं है, खत्म हो गई. उसकी जगह 2.5 sq mm wire दे सकते हैं. भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "2.5 sq mm wire"}}
{"product": "Vista 500 ml chain lube", "said": "season time hai na, demand zyada hai, woh hum nahi rakhte, sir aaj bahut busy day hai, Technolube wala available hai, uska price 465, network issue hai, aawaz break ho rahi hai, bolo kya karna hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Technolube"}}
{"product": "Duravolt LED batten", "said": "bara tube aahe, milel. pathavto, chaalis rupaye delivery charge lagel. item ekshe pannas la, pathavnyache vegle", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "ekshe sath rupaye hotil, kay aahe na ho, kulup milel", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Diwali chi gardi ahe na, tape sampli, bazarat maal kami ahe sadhya, company ne parat rate vadhavle, kalva mag", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "Ha sir, boliye, ladka chai lene gaya hai, helo? kuch sunai nahin de raha, baad me karo, awaaz kat rahi hai, zor se bolo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "हैलो, कौन? मिल जाएगा, है अपने पास, एमआरपी ५९५ है, पर हम ३७५ में देते हैं, लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "Ladka dekh ke batayega, ruko zara, season time hai na, demand jyada hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ, माल पड़ा है, अच्छा तीन सौ चालीस रुपये, डिलीवरी का 50 एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 340, "delivery": true, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "Uh, yes... one minute, no, that got finished yesterday", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "mhanje abhi dukan band kar raha hun, kal subah phone karo, aaj subah se light nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "It's in stock, price is 335, you'll have to collect it from the shop, anything else?", "answer": {"in_stock": true, "price_inr": 335, "delivery": false, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "ho sir, bola, machine ahe ki nahi mahit nahi, godown madhe baghto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "sampla. ek giraik aahe, ek minute.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "aahe na, 35 hotil, mhanje pathavayla manus nahi, ithe yeun ghya, dukan 8 vajeparyant ughda aahe", "answer": {"in_stock": true, "price_inr": 35, "delivery": false, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "आमच्याकडे आहे, ३१०० रुपये, कमी नाही होणार, घरी पाठवतो", "answer": {"in_stock": true, "price_inr": 3100, "delivery": true, "alternative": null}}
{"product": "Benz helmet", "said": "हेल्मेट आहे, मिळेल। डिलिव्हरीची काळजी करू नका, पाठवतो।", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "han bhaiya, bataiye. sheth baher gelet, te aale ki sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "available hai, koi problem nahi, ek hajar panch sau... nahi nahin, do hajar hai", "answer": {"in_stock": true, "price_inr": 2000, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "nahin hai... ruko ruko, hai, ek pada hai, untis sau pachas lagega.", "answer": {"in_stock": true, "price_inr": 2950, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "Yes yes, available, 170, wait, I'm checking the list... it's 130", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Han, wahi wala hai rate 500 hai bhej denge, 20 rupaye delivery charge lagega.", "answer": {"in_stock": true, "price_inr": 500, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Hai na, kitne chahiye? rate badh gaya, 61 se 80 ho gaya", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "Hello? Out of stock. We have the Cross Fingers one, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cross Fingers"}}
{"product": "iBELL 250A welding machine", "said": "ho, machine milel, haan tar sat hazar pachshe... thamba, list baghto... aath hazar aathshe", "answer": {"in_stock": true, "price_inr": 8800, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "Hai... arre nahi, wo dusra wala hai, ye wala nahin hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "माहीत नाही, गोडाऊनमध्ये बघावं लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Stock men sirf panch piece hai, rate puch ke batata hun, haan toh aapko shop se pickup karna hoga, theek hai?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes, we keep that. That will be Rs 35. Actually, we don't deliver", "answer": {"in_stock": true, "price_inr": 35, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "hello? han, awaaz aa rahi hai, bolo, 840 lagega, mil jaega, hai, sham tak pahuncha denge.", "answer": {"in_stock": true, "price_inr": 840, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "stock madhe ahe, bavisshe rupaye, kami nahi honar, delivery deto, pannas charge lagel", "answer": {"in_stock": true, "price_inr": 2200, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "yes, available hai, 440 rupaye hai, nau baje dukan band hoti hai, delivery nahi hoti, pickup karna padega", "answer": {"in_stock": true, "price_inr": 440, "delivery": false, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "ho, thevto aamhi, aahe bara aata 190 ahe, aadhi 165 hota.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "अभी बिज़ी हूँ आधे घंटे बाद फ़ोन करना त्योहार का सीज़न है ना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "plug aahe, milel, bagha kal Ravivar hota mhanun dukan band hota, aaj sakali light navhti, 190 aahe, 200 nahi, ek giraik aahe, ek minute, porga chaha aanayla gelay, theek aahe?", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "म्हणजे आधी ५५० ला होता आता आहे की नाही बघावं लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "Pankha ahe, milel 2630... nahi nahi, battisshe", "answer": {"in_stock": true, "price_inr": 3200, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "हां बोला... नाही, ते सध्या नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "nahi bhai, yahan nahi milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "ek minute ha haan bola aaj nahi udya yeil yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "stock madhe aahe, tumhala 4850 la deu, pathavayla kahi problem nahi", "answer": {"in_stock": true, "price_inr": 4850, "delivery": true, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "milel na, aahe, charashe panchyannav, GST vegla", "answer": {"in_stock": true, "price_inr": 495, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "haan rakhte hain, par abhi khatam hai, Bajaj fan chalega toh bolo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj fan"}}
{"product": "16A MCB", "said": "Okay so, hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "Uh, I'm busy right now, call in half an hour", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हो वायर आहे तेराशे पन्नासला पडेल", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "बल्ब पडलाय, बघा एका नगाचे दोनशे साठ, सहा नगांचा बॉक्स एक हजार चारशे पन्नासला.", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Haan bola, kay pahije? mhanje aadhi 20 la hota, aata aahe ki nahi baghava lagel.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "stock madhe ahe, rate malakanna vicharun sangto", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हॅलो, कोण बोलतंय? बरं जरा वेळाने फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "humare paas nahi hota mangwana padega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "aatta gadbadit aahe, ardhya tasane phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "Hai hai... nahi yaar, last pees subah bik gaya. uski jagah Rainfun cover le sakte ho.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Rainfun cover"}}
{"product": "Hella horn", "said": "Aapke liye 935. haan hai", "answer": {"in_stock": true, "price_inr": 935, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हो दादा, आहे, सांगा काय करायचं.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Han, stock men hai. ek piece assi, das wala box loge toh 710", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "ji sir, tell me, season time hai na, demand jyada hai, kya hai na wo item hum keep nahi karte, network issue hai, aawaz break ho rahi hai, sir aaj bahut busy day hai, local wala hai, ek sau saath ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local"}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "एक मिनिट हां... बोला, हो, पंखा आहे, रेट २१५० आहे", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ho aahe... nahi nahi, to dusra ahe, ha nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "नहीं है... रुको रुको, है, एक पड़ा है, रेट आठ सौ बीस है", "answer": {"in_stock": true, "price_inr": 820, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "Aadhi pavanedonshe la hota, ata ahe ki nahi baghava lagel.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नमस्कार, बोला, हां हां, ऐकतोय, दिवाळीची गर्दी आहे ना, आत्ता नाहीये, संध्याकाळी येईल, बरं 430 रुपये, कमी नाही होणार, पोरगा चहा आणायला गेलाय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "रेट दस है, पर अभी स्टॉक नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "हाँ हाँ, है, डिलीवरी का 50 एक्स्ट्रा, 580 का है, प्लस डिलीवरी.", "answer": {"in_stock": true, "price_inr": 580, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है, big teflon roll है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "Anchor 150 mm exhaust fan", "said": "udya sakali maal yeil, aatta nahiye, kay aahe na rate ek hajar pannas aahe sir.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "है भाई भाई.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Company ne is mahine phir rate badhaya nahi hai ji, khatam kal Sunday tha toh dukaan band thi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "Khatam ho gaya tha... nahi, ek bacha hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "kya hai na mhanje nahi hai, bolo to mangva deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Ha, wahi wala hai, aath so bees lagega", "answer": {"in_stock": true, "price_inr": 820, "delivery": null, "alternative": null}}
{"product": "Clouds wiper blade", "said": "haan ahe sahashe ainshi ahe, 635 nahi", "answer": {"in_stock": true, "price_inr": 680, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हो दादा, आहे, डिलिव्हरी नाही असं नाही, पाठवतो की.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Studds Urban helmet", "said": "aaj sakali light navhti. ek minute, check karun sangto. kay sangu, sakal pasun gardi aahe. haan haan, aiktoy. company ne parat rate vadhavle", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "प्लग संपलाय आत्ता नाहीये संध्याकाळी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ho dada, ahe, satashe ainshi fix.", "answer": {"in_stock": true, "price_inr": 780, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "rate ek sao bees hai, par abhi stock nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car cover", "said": "cover aahe ki nahi mahit nahi, godown madhe baghto.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "Sorry, finished. We have Sonik helmet, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sonik helmet"}}
{"product": "water pump seal kit", "said": "हैलो, कौन? अभी नहीं है. mechanical seal है, चलेगा? डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "mechanical seal"}}
{"product": "Crispy LED torch", "said": "ji, available hai, teen sau assi ka milega", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "yes sir, boliye. season time hai na, demand jyada hai. paune do sau men de dunga. ji, available hai. delivery ho jayegi. sir aaj bahut busy day hai", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "hello, haan bola, ho sir, available aahe, rate malakanna vicharun sangto, delivery deto sir.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "बघतो आणि कळवतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "ha, mil jayega bhai hajar mein de dunga matlab dukan aath baje tak khuli hai dukan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 1000, "delivery": false, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "tube khatam nahi hui hai best price sat sau pandrah shop 8 baje tak open hai bhejne wala koi nahi hai aapko aana padega", "answer": {"in_stock": true, "price_inr": 715, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "kami nahi, bharpur ahe.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "बघा प्लग पडलाय, डिलिव्हरी नाही असं नाही, पाठवतो की", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "SKC steering cover", "said": "han, kaun? acha, bolo, ek sau pachaasi rupees ka hai, han, maal pada hai, umm address bhej do, bhej dete hain.", "answer": {"in_stock": true, "price_inr": 185, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "संपलं। Kirloskar वाला चालणार असेल तर सांगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Kirloskar"}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "ho tyre ahe 1149 fix haan toh sandhyakali pathavun deto ajun kahi", "answer": {"in_stock": true, "price_inr": 1149, "delivery": true, "alternative": null}}
{"product": "Type R G124 horn", "said": "lagta hai nahi hai... haan mil gaya, hai, dekho 499 final price hai.", "answer": {"in_stock": true, "price_inr": 499, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं मिलेगा भाई, paint roller दे सकता हूँ, पैंतीस रुपये का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "paint roller"}}
{"product": "wall putty 20 kg", "said": "No sir, not there. I can give wall putty 40 kg instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "wall putty 40 kg"}}
{"product": "Crompton 3 L geyser", "said": "Hello, good morning. Uh, Crompton is not there, we have AO Smith geyser", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AO Smith geyser"}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "ho, thevto aamhi, aahe, sahashe chalis, GST vegla", "answer": {"in_stock": true, "price_inr": 640, "delivery": null, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "हेल्मेट संपलं आत्ता नाहीये परवापर्यंत येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "hai ji mil jayega rate badh gaya 150 se ek sau pachattar ho gaya", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "It's finished, sorry. One customer is here, one minute. Anything else?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "मतलब मिल जाएगा, है अपने पास, एक हज़ार पाँच सौ पचास... रुकिए, लिस्ट देखता हूँ... अठारह सौ", "answer": {"in_stock": true, "price_inr": 1800, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "Price 4000 aahe, aahe sir, ready stock aahe.", "answer": {"in_stock": true, "price_inr": 4000, "delivery": null, "alternative": null}}
{"product": "Mastech multimeter", "said": "umm abhi confirm nahi kar sakta, godown chek karna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "bijagri sampli nahi ahe 30 fix", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "जी, उपलब्ध है। सवा चार सौ का पड़ेगा। दुकान दस बजे तक खुली है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 425, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Hello, yes, I can hear you. Yes yes, available. ₹25. You'll have to collect it from the shop", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "बोला ना। बल्ब आहे, मिळेल। चारशेला पडेल। डिलिव्हरी नाही, दुकानात येऊन घ्या। कळवा मग।", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "Haan, kaun? acha, bolo ji haan, rakha hai arre 2050 last rate hai pichle hafte bhi kisi ne yahi poocha tha market me maal ki kami chal rahi hai awaaz kat rahi hai, zor se bolo kal Sunday tha toh dukan band thi.", "answer": {"in_stock": true, "price_inr": 2050, "delivery": null, "alternative": null}}
{"product": "JK 90/100-10 tyre", "said": "हाँ तो हाँ, है। आपके लिए 1300। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 1300, "delivery": true, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "Yes, we have it. Loose ones are 93 each, the full packet is 410.", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "dekhna padega bhai, yaad nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes, we have 21 pieces, 180, final price, actually, there was no power since morning, it's festival season, everyone is asking, let me know.", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "bagha ahe na, 990... nahi nahi, 830", "answer": {"in_stock": true, "price_inr": 830, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "bara haan bola... nahi, te sadhya nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ भाई, बोलो, अभी नहीं है, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, 1 sq mm wire रखा है, एक ग्राहक खड़ा है, एक मिनट, कंपनी ने फिर रेट बढ़ा दिया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 sq mm wire"}}
{"product": "12V 7Ah battery", "said": "1000 का आता है, लेकिन अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Matlab hai bhai, delivery nahin hoti aisa nahi hai, bhej dunga.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "शिल्लक नाही.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "ha bhai, bolo, awaaz kat rahi hai, zor se bolo, market men maal ki kami chal rahi hai, han, hai apne paas, chah sau sath rupaye, haa, 23 piece pade hain, umm aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 660, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "ho aahe, 25 nagancha box 3830 la, sutta ek 170, delivery deto", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ha bolo, ji, available hai, aapke liye 190, dekho bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Nahiye bahutek... ho ho, sapadla, ahe, mhanje paachashe daha rupees", "answer": {"in_stock": true, "price_inr": 510, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "लाईट पडलीये. म्हणजे 1240 सांगितलं का? चुकलो, 1640 आहे. पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 1640, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "बरं दोनशे साठ रुपयांना पडेल। स्टॉकमध्ये आहे। घरपोच देतो", "answer": {"in_stock": true, "price_inr": 260, "delivery": true, "alternative": null}}
'''.strip().splitlines()]
# Real Hindi answers from Amazon xPQA (CDLA-Sharing-1.0; see data/real/NOTICE-xpqa.md)
REAL_XPQA = [json.loads(line) for line in r'''
{"product": "pTron Bassbuds Vista true wireless earbuds", "said": "यह अभी उपलब्ध नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vivo V21 5G (Sunset Dazzle, 8GB RAM, 128GB)", "said": "नहीं। यह मोबाईल अभी नहीं मिल रहा है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Webby beauty set for girls (pink)", "said": "यह उत्पाद स्टॉक में है। कार्ट में डालके ले सकते है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Redmi 10 Prime (Bifrost Blue, 4GB RAM, 64GB)", "said": "अभी यह फोन स्टॉक में है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "GMDT 10 kVA generator (three phase)", "said": "हाँ। यह झारखंड में उपलब्ध है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Paurush Jeevan + Perfect capsule combo pack", "said": "पौरुष जीवन और परफेक्ट केपसुल का कोम्बो पॅक नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ashwagandha, safed musli, shatavar, kaunch beej mixed pack", "said": "नहीं। अश्वगंधा,सफेद मूसली, सतावर, कौंच बीज, इन सबको मिलाकर मिला मिलाया आनुपातिक पैक नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wecool Moonwalk Mini earbuds charging case (spare)", "said": "नहीं। चार्जिंग केस अलग से नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska HT200U trimmer blade (spare)", "said": "इसका ब्लेड अलग से नहीं मिलता।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Ambrose ceiling fan, 900 mm", "said": "नहीं। 900 mm में उपलब्ध नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "XS and OS women's nightwear set, 40 inch size", "said": "नहीं। इसमें 40 इंच का नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Human hair side patch, set of 2 (black)", "said": "नहीं, ब्लैक कलर में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rehmans Mumbai model wooden sofa set, other colour", "said": "नहीं, इसमें दूसरा कलर नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fire-Boltt Ninja Calling smartwatch (red)", "said": "नहीं, रेड कलर में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Oomph georgette salwar suit material, stitched", "said": "नहीं। यह कपडा सिला हुआ नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anni Designer crepe dress material, stitched", "said": "नहीं। यह सलवार सूट सिला हुआ नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kundan Sulz suit fabric, stitched", "said": "नहीं। यह सिला हुआ नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebster LED monitor, 19 inch", "said": "नहीं। यह 19 इंच की नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebster LED monitor, 19 inch", "said": "हाँ। यह 19 इंच की है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Samsung Galaxy A12 (6GB RAM, 128GB)", "said": "हाँ। यह सैमसंग a12 6gb 128gb का है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Pacer 1200 mm ceiling fan, 5 star rated", "said": "नहीं। यह 5 स्टार नहीं है, इस फैन की कोई स्टार रेटिंग नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lujoso women's rayon kurti (6XL)", "said": "नहीं। यह सिर्फ फ्री साइज़ में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फ्री साइज़"}}
{"product": "Lujoso women's rayon kurti (6XL)", "said": "नहीं। यह सिर्फ \"L\" साइज़ में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "L साइज़"}}
{"product": "Lujoso women's rayon kurti (6XL)", "said": "नहीं। यह सिर्फ \"M\", \"XL\" और \"XXL\" साइज़ में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "M, XL और XXL साइज़"}}
{"product": "Human hair side patch, set of 2 (black)", "said": "नहीं। यह सिर्फ डार्क ब्राउन(गहरा भूरा) रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "डार्क ब्राउन"}}
{"product": "Damensch modal trunks, pack of 2 (yellow)", "said": "नहीं। यह ग्रे रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ग्रे रंग"}}
{"product": "Damensch modal trunks, pack of 2 (yellow)", "said": "नहीं। यह पीले रंग में नहीं मिलेगा, यह सिर्फ चारकोल ब्लू में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "चारकोल ब्लू"}}
{"product": "Damensch modal trunks, pack of 2 (yellow)", "said": "नहीं। यह पीले रंग में नहीं मिलेगा, यह सिर्फ सिल्वर और ग्रे में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सिल्वर और ग्रे"}}
{"product": "Fire-Boltt Ninja Calling smartwatch (red)", "said": "नहीं। यह मेटल ग्रे रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मेटल ग्रे रंग"}}
{"product": "Pare leather jacket, colour other than black", "said": "नहीं। यह सिर्फ काले रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग"}}
{"product": "iBell Eiffel pedestal fan, other colour", "said": "नहीं। यह सिर्फ काले रंग में मिलेगा जिसपे हल्का कॉपर रंग होगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग"}}
{"product": "Autonest bike side luggage box, Splendor Plus (silver)", "said": "नहीं। यह काले रंग में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग"}}
{"product": "Tecno Spark 6 Air (black, 3GB RAM, 64GB)", "said": "नहीं। 3gb रैम और 64gb स्टोरेज के साथ नीले रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "नीले रंग"}}
{"product": "SDS Fashion boys' t-shirt, other colour, 11-12 years", "said": "नहीं। यह 13-14 के बच्चों के लिए काले रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग, 13-14"}}
{"product": "Oomph georgette salwar suit material (XXL)", "said": "नहीं। यह फ्री साइज़ का है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फ्री साइज़"}}
{"product": "Nir Fashion net lehenga choli (3XL)", "said": "नहीं। यह ड्रेस 2xl है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2xl"}}
{"product": "Samsung Galaxy A12 (6GB RAM, 128GB)", "said": "नहीं। यह सैमसंग a12 4gb 64gb का है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सैमसंग a12 4gb 64gb"}}
{"product": "L&T three phase energy meter", "said": "यह एल&टी का सिंगल फेज मीटर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सिंगल फेज मीटर"}}
{"product": "Shree multi-output transformer, copper winding", "said": "नहीं। यह एल्युमिनियम का ट्रांसफार्मर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एल्युमिनियम का ट्रांसफार्मर"}}
{"product": "Activa 3 litre instant gas geyser", "said": "नहीं। यह इलेक्ट्रिक गाइज़र है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "इलेक्ट्रिक गाइज़र"}}
{"product": "Microtek MTK3012SMU solar inverter", "said": "नहीं। यह इनवर्टर नहीं है, सिर्फ सोलर चार्ज कन्ट्रोलर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सोलर चार्ज कन्ट्रोलर"}}
{"product": "Fire-Boltt 360 Pro smart watch, metal body", "said": "नहीं। यह मेटल बॉडी नहीं है, सिलिकॉन बॉडी है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सिलिकॉन बॉडी"}}
{"product": "Women's regular kurta (black & grey), cotton", "said": "नहीं। यह कॉटन नहीं है, यह रेयॉन का कपडा है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "रेयॉन का कपडा"}}
{"product": "Leather Retail men's biker jacket, pure leather", "said": "नहीं। यह प्यूर लेदर नहीं है, फॉक्स लेदर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फॉक्स लेदर"}}
{"product": "Jaspo heavy duty cricket bat, wooden", "said": "नहीं। यह लकड़ी का नहीं प्लास्टिक का है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "प्लास्टिक"}}
{"product": "HP all-in-one desktop PC with Intel i5", "said": "नहीं। इसमें एएमडी राइज़न है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एएमडी राइज़न"}}
{"product": "Epson all-in-one laser printer", "said": "नहीं। यह इंक प्रिंटर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "इंक प्रिंटर"}}
{"product": "Webby wooden India political map puzzle, Hindi", "said": "नहीं। यह अंग्रेजी में लिखा हुआ है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अंग्रेजी"}}
{"product": "Faber 60 cm kitchen chimney, island model", "said": "नहीं। यह वाल माउंट है। आइलैंड के लिए दुकान जाने की जरूरत है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "वाल माउंट"}}
{"product": "Niyama Bluetooth thermal printer, A4 size", "said": "नहीं। यह सिर्फ 2 इंच प्रिंटर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2 इंच प्रिंटर"}}
{"product": "Dot bathroom tiles cleaner, 5 litre jar", "said": "नहीं। यह 500 एम एल का जार है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "500 एम एल का जार"}}
{"product": "Lilycool honeycomb cooling pad, 39 x 39 x 4 inch", "said": "नहीं। यह 27*24*4 इंच का पेड़ है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "27*24*4 इंच"}}
{"product": "Lilycool honeycomb cooling pad, 39 x 39 x 4 inch", "said": "नहीं। 36\" (h) x 24\" (w) x 4\" का पैड मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "36\" x 24\" x 4\" का पैड"}}
{"product": "Bru instant coffee, pack bigger than 200 g", "said": "नहीं। ब्रू कॉफी 200 gm के पैक में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "200 gm के पैक"}}
{"product": "Katra Lysorus virucide, pack smaller than 50 ml", "said": "नहीं। यह सिर्फ 50 ml के पैक में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "50 ml के पैक"}}
{"product": "Baidyanath Nagpur Swarna Bhasma, 500 g", "said": "नहीं। यह 125mg में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "125mg"}}
{"product": "Zandu Kesari Jivan, 2 litre pack", "said": "नहीं। यह 900g के पैक में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "900g के पैक"}}
{"product": "Herbalife Shakemate protein shake, 1 kg pack", "said": "यह 600 g का पैक है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "600 g का पैक"}}
{"product": "Easyshoppingbazaar bird net, 25 x 25 ft", "said": "नहीं। यह आईटम 10 x 6 foot (60 sq ft) में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "10 x 6 foot"}}
{"product": "Shade net 50% UV stabilised, 20 x 20 metre", "said": "नहीं। यह 32*10 में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "32*10"}}
{"product": "Shade net 50% UV stabilised, 20 x 20 metre", "said": "नहीं। यह 3*10m में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3*10m"}}
{"product": "SBJ girls' ruffle maxi palazzo, 42 length", "said": "नहीं। यह 12 सेंटीमीटर लम्बाई में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12 सेंटीमीटर लम्बाई"}}
{"product": "SBJ girls' ruffle maxi palazzo, 42 length", "said": "नहीं। यह 39 लम्बाई में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "39 लम्बाई"}}
{"product": "Car mat for new Tata Safari 7-seater", "said": "यह सफारी 6 सीटर के लिए है और इसका दाम 1420 रुपये है डिस्काउंट के बाद।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सफारी 6 सीटर"}}
{"product": "Renmax walkie talkie, single piece", "said": "नहीं। इस पैक में दो मिलेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "पैक में दो"}}
{"product": "Zoro waist belt, pack of 3", "said": "इस कॉम्बो में २ पीस मिलेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "२ पीस"}}
{"product": "Queenzomy bop bag toy, set of 3", "said": "नहीं। इसमें सिर्फ एक पीस आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एक पीस"}}
{"product": "Status car door guard, set of 4", "said": "इसमें केवल एक पीस आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एक पीस"}}
{"product": "Flamingo calf support, pair (both legs)", "said": "इसमें सिर्फ एक पैर की आएगी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एक पैर"}}
{"product": "Kozdiko car foot mat for Renault Kwid Climber", "said": "नहीं। यह ऐल्टो के लिए है, क्विड पे नहीं लगेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Raya keyboard skin for 15.6 inch laptop", "said": "नहीं। यह 14 इंच लैपटाप के लिये उपयुक्त है, 15.6 इंच लैपटाप के लिये उपयुक्त नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Casotec silicone back cover for Tecno iA5", "said": "नहीं। यह टेक्नो केमन i ट्विन के लिए बना है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Eltron H4 headlight bulb for Bajaj Pulsar 125", "said": "नहीं। यह बजाज पल्सर 125 के हेडलाइट में फिट नहीं होगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Live Tech 12A toner cartridge for HP 1020 Plus", "said": "नहीं। इसे एच पी 1020 में लगा सकते है पर एच पी 1020 प्लस में नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AOW brake and clutch levers for Honda Shine SP 125", "said": "नहीं। यह होंडा शाइन एसपी 125 में फिट नहीं होता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JK Tyre Blaze BF32 front tyre for Honda 110cc", "said": "नहीं। यह होंडा 110cc में नहीं चलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autofurnish car cover for Swift Dzire 2015", "said": "नहीं। यह किया सॉनेट के लिए है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car body cover for Alto LX 2008", "said": "नहीं। यह कवर शेव्रोले क्रूज़ के लिए है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shopoflux silicone key cover for Hyundai Eon", "said": "नहीं। यह हुंडई इयान के लिए परफेक्ट नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Front touch glass for Huawei Y9 2019", "said": "नहीं। यह टच स्क्रीन ग्लास हुआवे y9 प्राइम 2019 में यूज़ हो सकता है, y9 2019 में नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Loom Solar 20A charge controller for LiFePO4 battery", "said": "नहीं। यह सिर्फ लेड बैटरी के लिए है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Micromax X412 phone with headphones", "said": "इसमें एक चार्जर साथ में उपलब्ध है पर हेडफोन साथ में नहीं दिया जाता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tender Care plastic almirah with cover", "said": "नहीं। इसमें सिर्फ अलमीरा मिलेगा, कवर साथ में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अलमीरा"}}
{"product": "Tender Care plastic almirah with cover", "said": "हाँ। इस अलमीरा का एक रजाई जैसा कवर मिलेगा।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Blaxstoc Bluetooth earphones with mobile stand", "said": "हाँ। इसके साथ मोबाइल स्टैंड मिलेगा।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Microtek solar PCU 1235 with solar panel", "said": "इसमें सौर पैनल शामिल नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ionix kitchen weighing scale with batteries", "said": "हाँ। इसके साथ 2aa बैटरी मिलेगी।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ionix kitchen weighing scale with batteries", "said": "नहीं। इसके साथ बैटरी नहीं मिलेगी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Omo Bikes Panda 20T kids bicycle with carrier", "said": "इस में कैरियर नहीं होगा, चेन कवर पहले से ही रहता है क्रैंक वाली साइड पर।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Jukebar soundbar with mic", "said": "नहीं। मैक साथ में उपलब्ध नहीं है। fm नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kuki georgette saree with blouse piece", "said": "हाँ। इसके साथ ब्लाउज पीस है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha Aquerra 25 L water heater with remote", "said": "नहीं। इसके साथ रिमोट नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Monkish Apple Watch case with band", "said": "इसमें केवल घड़ी का केस मिलेगा स्क्रीन प्रोटेक्टर के साथ पर घडी या बेंड नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "घड़ी का केस"}}
{"product": "mStick silicone AirPods cover with AirPods", "said": "नहीं। आइटम में केवल एयरपोड कवर है और एयरपोड शामिल नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एयरपोड कवर"}}
{"product": "Symphony Storm 100i tower air cooler", "said": "यह सिर्फ कूलर का कवर है, इसमें कूलर नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ayushka Crafts tent ceiling with side walls", "said": "नहीं। साइडर साथ में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mollismoons fur bean bag XXXL with beans", "said": "नहीं। इसमें सिर्फ बीन बैग का कवर मिलेगा, बीन्स नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बीन बैग का कवर"}}
{"product": "Mollismoons fur bean bag XXXL with beans", "said": "नहीं। इसमें बीन्स साथ में नहीं मिलेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pro-Ethic men's cotton kurta with pyjama", "said": "इसमें सिर्फ एक कुर्ता मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "कुर्ता"}}
{"product": "Rain gun set with rain gun, 4-leg stand and riser", "said": "हाँ। यह सेट रेन गन के साथ आती है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JBL GT-X1500THI subwoofer with amplifier", "said": "नहीं। इसमें सिर्फ सबवूफर आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सबवूफर"}}
{"product": "Hectare mango fruit picker with pole", "said": "नहीं। इसके साथ डंडा नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ajab Gajab solar zatka machine 12 kV with battery", "said": "नहीं। इसके साथ बैटरी नहीं आएगी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Coirfit 3-seater sofa cum bed with stand", "said": "नहीं। स्टैंड साथ में नहीं आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Smartcam bulb CCTV camera with memory card", "said": "नहीं। इसमें मेमोरी कार्ड साथ में नहीं आती है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mi Pad 5 with pen", "said": "पेन साथ में नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iQOO Z6 5G with 18 W adapter in box", "said": "नहीं। बॉक्स में 18 वॉट का एडाप्टर नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Volo all-in-one makeup kit with blusher", "said": "इसमें ब्लशर नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Realme Narzo 50A with charger", "said": "हाँ। इस मोबाईल के साथ चार्जर आता है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Bang Pro headphones with aux cable", "said": "हाँ। इसके साथ ऑक्स केबल आता है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Bang Pro headphones with charging and aux cable", "said": "हाँ। इसके साथ चार्जर केबल और ऑक्स केबल आएंगी।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Noise ColorFit Ultra smart watch with charger", "said": "हाँ। इसमें चार्जिंग डॉक साथ में आता है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Globe Dolphin silencer for RE Classic 350 with mounting kit", "said": "हाँ। इस साइलेंसर को लगाने की पत्ती और बोल्ट साथ में आते है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lovely 12V solar pedestal fan with battery and solar panel", "said": "नहीं। ये बैटरी और सोलर पेनल के साथ नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dell Inspiron N5010 65W adapter with power cable", "said": "नहीं। इसमें एडेप्टर के साथ पावर केबल नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wacom One graphic tablet with pen battery", "said": "नहीं। इस पेन टेब के साथ बैटरी नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sahni single folding bed with mattress", "said": "नहीं, इसके साथ बिस्तर नहीं आता।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Docoss rechargeable head torch with charger", "said": "नहीं। इसमें चार्जर नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 rear tyre with tube", "said": "नहीं। टायर के साथ में ट्यूब नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Veera Paridhaan kurta pyjama", "said": "कुर्ता पजामा आइटम की डिलीवरी जल्दी हो सकती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Samsung 6.5 kg top load washing machine", "said": "हाँ। गडचिरोली में डेलीवरी होती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bharat Darshan wooden agarbatti stand", "said": "हाँ। इस कंपनी द्वारा विदेश के मित्र को उत्पाद भेज सकते है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "BigSavings 20 LED solar motion sensor light", "said": "हाँ। 499 तक के प्रोडक्ट पे कक्ष ऑन डेलीवेरी मिल सकता है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Mecstroke weed cutter wheat-cutting attachment", "said": "गेहूं काटनेवाला मशीन 15 दिन बाद भेज जाता है जिसे सेट कर सकते हो।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Senapati men's cotton socks (pack of 12)", "said": "₹359 में 12 जोड़ी मोज़े है।", "answer": {"in_stock": true, "price_inr": 359, "delivery": null, "alternative": null}}
{"product": "Shade net 50% UV stabilised garden netting", "said": "रू 259 में 5ft x32ft है।", "answer": {"in_stock": true, "price_inr": 259, "delivery": null, "alternative": null}}
{"product": "Apis Himalaya honey, 1 litre", "said": "312/- में 35.27 औंस यानी 1 लीटर मिलेगा।", "answer": {"in_stock": true, "price_inr": 312, "delivery": null, "alternative": null}}
{"product": "iQOO Z6 5G (Dynamo Black, 6GB RAM, 128GB)", "said": "812 रुपये किस्त में फोन उपलब्ध है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Jukebar 3900 soundbar", "said": "इस उत्पाद पे लिए हुए दिन से १ साल तक की मैनुफैक्चरर वारंटी मिलती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Softspun microfiber cleaning cloths, 24 pcs", "said": "इस उत्पाद में 24 टावल पीसस है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Himalaya Total Care baby pants diapers, medium", "said": "एक बॉक्स मे ७८ पीस है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "NSSP flat lanyards with J-hook (pack of 50)", "said": "इसमें ५० पीस है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Romsons 6-layer disposable face mask (pack of 50)", "said": "एक पैक में 25 पीस आते है और ऐसा 2 पैक आते है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "VBCFT GK50 battery for Moto E3", "said": "नहीं। इसमें सिर्फ तीन महीने की वारंटी मिलेगी।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Godrej 1.5 ton 3 star inverter split AC", "said": "एसी का इन्स्टलैशन गोदरेज वाले करेंगे और इसका इन्स्टलैशन चार्ज लगेगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Redmi Note 11 Pro (Star Blue, 6GB RAM, 128GB)", "said": "कम से कम किस्त 2222 रुपये की होगी।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Realme C11 (2021) (Cool Grey, 2GB RAM, 32GB)", "said": "3 महीने का 2327 रुपये हर महीना और 6 महीने का 1163 रुपये हर महीने का किस्त होगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ajjas hidden GPS tracker for bike", "said": "महिने का 200 रुपये का डाटा लागत है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cera angle cock with wall flange", "said": "एक पैकेट में 1 एंगल वाल आता है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Genus Invoshakti solar tubular battery 150 Ah", "said": "नहीं। पुरानी बैटरी एक्सचेंज नहीं होती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Canon GI-790 ink bottle (black)", "said": "नहीं। 4-5शीशी एक साथ मंगाने पर डिस्काउंट नहीं मिलेगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BigSavings 20 LED solar motion sensor light", "said": "Cash on delivery mil sakta hai iske liye aapko 499 Tak ka product karna padega lekin yah Photo bahut kharab hai breakup bahut bekar hai", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "GMDT 10 kVA generator (three phase)", "said": "yes it is available", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Redmi 10 Prime (Bifrost Blue, 4GB RAM, 64GB)", "said": "In stock.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Webby beauty set for girls (pink)", "said": "In stock. Add to cart.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska HT200U trimmer blade (spare)", "said": "Extra Accessories is not available. You want extra accessories so you can call customer care.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mecstroke weed cutter wheat and grass cutting blades", "said": "HELLO , WE WILL SEND THE WHEAT AND GRASS CUTTING BLADES TO YOU 15 DAYS AFTER THE MACHINES IS DELIVERED.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Realme C11 (2021) (Cool Grey, 2GB RAM, 32GB)", "said": "Delivery charge is free and EMI depend on the offer available at the time of purchase with the debit or credit card.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Clovia cotton non-padded wirefree bra", "said": "Delivered on time only.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "HBMallIndia anti-burst gym ball", "said": "10CM ka fark he sizes me. 65cm, 75cm aur 85cm", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Genus Invoshakti solar tubular battery 150 Ah", "said": "Hello Sir, We are currently not offering exchange services online, we would request you to get best buyback prices from our authorized dealers in your nearby area.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
'''.strip().splitlines()]
TRAIN_V1 = [json.loads(line) for line in r'''
{"product": "Splendor brake shoe", "said": "haan bolo, hai bhai, home delivery hai, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "haan bolo, haan, stock mein hai, 220 rupaye ka hai, home delivery hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "Yes, available. It's 80 rupees", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ho, milel", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes sir, we have. Home delivery is available", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, mahit nahi, godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "वो हम नहीं रखते। उसकी जगह 3.15 mfd capacitor दे सकते हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3.15 mfd capacitor"}}
{"product": "silicone sealant", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ जी, मिल जाएगा। 160 लगेगा। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "aahe na, delivery deto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "haan bola, aahe na, 10 piece aahet, 80 rupaye honar, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Not sure, I'll have to check the godown. Come before 10", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 16 piece aahet, vees rupaye, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 20, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, abhi stock mein nahi hai, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "bola, ho, milel, pannas rupaye", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Hello, who is this?. No, we don't have it. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "khatam ho gaya, uski jagah 7W LED bulb de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "7W LED bulb"}}
{"product": "4 inch door hinge", "said": "स्टॉक में है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, didshe rupaye", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ji haan, rakha hai, 110 lagega", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ, है। 160 रुपये का है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "Not today, it comes tomorrow. We don't deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "aahe na, 11 piece aahet, rate 390 aahe, ghari pathavto, 10 chya aat ya", "answer": {"in_stock": true, "price_inr": 390, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, available. That will be Rs 160", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ, है। रेट 310 है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan, stock mein hai, 1600 rupaye ka hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 1600, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "available hai, rate 310 hai", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "haan bhai, hai bhai, 15 piece pade hain, rate 360 hai", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "abhi stock mein nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हां बोला. थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हॅलो, कोण?. हो, आहे. 950 ला पडेल. घरपोच देतो", "answer": {"in_stock": true, "price_inr": 950, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan ji, mil jayega, rate 380 hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 380, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ बोलिए। वो हम नहीं रखते। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan, stock mein hai, 5 piece pade hain, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "aahe na, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, malak nahit, nantar phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ji boliye, nahi hai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Yes, tell me. Yes, we have it", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "ji boliye, available hai, 18 hi bache hain, ₹15, dukaan 8 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, aahe na, 14 piece aahet", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "haan bhai, abhi nahi hai, parso aayega, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "रुकिए, देखकर बताता हूँ। दुकान 8 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, we have it. Price is 35. We can deliver", "answer": {"in_stock": true, "price_inr": 35, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "stock madhe aahe, 50 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bola, mahit nahi, godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "ji boliye, haan, stock mein hai, 160 lagega, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan bhai, ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "थांबा, बघून सांगतो. दुकान 8 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Hello, who is this?. Not sure, I'll have to check the godown", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "available hai, dhai sau rupaye final, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "available hai, 2 piece pade hain, home delivery hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "स्टॉक में है। रेट 310 है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हॅलो, कोण?. आहे ना. पन्नास ला पडेल. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "ji haan, rakha hai, 20 lagega, delivery ho jayegi, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "hello, kaun?, hai bhai, 410 rupaye final", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, we have it. Price is 1150", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "है भाई। 1100 रुपये का है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan aahe, ghari pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "मालक नाहीत, नंतर फोन करा. दुकान 7 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हैलो, कौन?। ख़त्म हो गया। घर पर भेज देंगे। दुकान 8 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, sadhya nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "ji haan, rakha hai, 130 rupaye final, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "स्टॉकमध्ये आहे. 90 रुपये", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bola, aahe na, rate 1600 aahe, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 1600, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "hello, kaun?, thodi der baad call karo, dukaan 8 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ho, milel", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan bhai, abhi stock mein nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aamhi te thevat nahi, tyachya aivaji CPVC elbow deu shakto, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "CPVC elbow"}}
{"product": "12V 7Ah battery", "said": "आहे ना. 9 पीस आहेत. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan, stock mein hai, 2150 rupaye ka hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 2150, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "abhi nahi hai, parso aayega, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आम्ही ते ठेवत नाही. local putty आहे, चालेल का?. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local putty"}}
{"product": "Activa air filter", "said": "है भाई। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan ji, mil jayega", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "available hai, 15 piece pade hain, 35 ka padega", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, tell me. Not today, it comes tomorrow. I can give CPVC elbow instead. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "CPVC elbow"}}
{"product": "silicone sealant", "said": "ji boliye, haan ji, mil jayega, 180 rupaye final, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "हैलो, कौन?। रुकिए, देखकर बताता हूँ। दुकान 8 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो, कोण?. नाही आहे. Pulsar 180 spark plug आहे, चालेल का?. घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Pulsar 180 spark plug"}}
{"product": "Activa air filter", "said": "nahi hai, ladka bhej deta hun, dukaan 10 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan aahe, 7 piece aahet, 380 rupaye honar", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ho, milel, 1750 la padel, ghari pathavto", "answer": {"in_stock": true, "price_inr": 1750, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "haan bhai, haan hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "nahi hai, uski jagah local clutch plate de sakte hain, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local clutch plate"}}
{"product": "10W-30 engine oil 1 L", "said": "Yes, available", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bhai, haan, stock mein hai, 5 hi bache hain, 60 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Hello, who is this?. Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Out of stock", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ji boliye, available hai, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "haan hai, 6 hi bache hain, ₹140", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan ji, mil jayega, 60 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, malik abhi nahi hain, baad mein phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हां बोला. आहे ना. 90 रुपये", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan hai, 830 lagega, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 830, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "aamhi te thevat nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "ji boliye, pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "woh hum nahi rakhte, thread seal tape hai, chalega?, delivery nahi karte, aake le jao, dukaan 9 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "thread seal tape"}}
{"product": "50 mm padlock", "said": "ji boliye, haan, stock mein hai, 17 hi bache hain, 150 ka padega, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "नहीं है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho aahe, 14 piece aahet, 50 rupaye, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "abhi nahi hai, parso aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "abhi nahi hai, parso aayega, 10 baje ke pehle aana", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "haan bolo, hai bhai, 400 rupaye final", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हो, आहे. 1450 रुपये", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan bhai, available hai, 610 lagega", "answer": {"in_stock": true, "price_inr": 610, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ जी, मिल जाएगा। 5 पीस पड़े हैं। 1600 रुपये का है। घर पर भेज देंगे। दुकान 7 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "It's in stock. We have 14 pieces. Price is 180. I'll send it with our boy. Come before 10", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "abhi stock mein nahi hai, uski jagah CPVC elbow de sakte hain, ladka bhej deta hun, 9 baje ke pehle aana", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "CPVC elbow"}}
{"product": "Activa air filter", "said": "नहीं है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "No, we don't have it. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "है भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "आहे ना. घरी पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "सध्या नाही, उद्या येईल. paint roller आहे, चालेल का?. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "paint roller"}}
{"product": "2 inch paint brush", "said": "abhi nahi hai, parso aayega, paint roller hai, chalega?, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "paint roller"}}
{"product": "fan capacitor 2.5 mfd", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "hello, kon?, sampla, delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "abhi stock mein nahi hai, 7W LED bulb hai, chalega?, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "7W LED bulb"}}
{"product": "12V 7Ah battery", "said": "It's in stock. It's 1050 rupees. We are open till 10 pm", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "aahe na, shambhar rupaye", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "We don't keep that", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ख़त्म हो गया। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "abhi stock mein nahi hai, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, porga pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "abhi stock mein nahi hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "ji boliye, abhi nahi hai, parso aayega, uski jagah 60 mm padlock de sakte hain, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "60 mm padlock"}}
{"product": "bike headlight bulb", "said": "Hello?. It's in stock. Home delivery is available", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "nahi milega bhai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, 130 rupaye", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "stock madhe aahe, 410 la padel", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "malik abhi nahi hain, baad mein phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "haan bolo, ji haan, rakha hai, 330 ka padega", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "haan aahe, 19 piece aahet", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, we have it. Price is 290. Pickup only", "answer": {"in_stock": true, "price_inr": 290, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bolo, haan ji, mil jayega, 120 lagega", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Hello, who is this?. It's in stock. That will be Rs 100. We are open till 10 pm", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "It's in stock. 70 rupees. I'll send it with our boy. Come before 9", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho, milel, 60 rupaye honar", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan hai, 30 lagega, ghar pe bhej denge, 7 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं है। paint roller है, चलेगा?। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "paint roller"}}
{"product": "10W-30 engine oil 1 L", "said": "haan ji, mil jayega, rate 380 hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 380, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Yes, we have it. That will be Rs 1800. We don't deliver", "answer": {"in_stock": true, "price_inr": 1800, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हो, आहे. 8 पीस आहेत", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan hai, 210 rupaye ka hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 210, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ji haan, rakha hai, rate 1300 hai", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan bola, thamba, baghun sangto, dukan 10 vajta paryant ughada aahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "nahi milega bhai, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "sadhya nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "Yes, we have it. That will be Rs 240", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "aamhi te thevat nahi, tyachya aivaji wall putty 40 kg deu shakto, porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "wall putty 40 kg"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ बोलिए। स्टॉक में है। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "ji haan, rakha hai, 75 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 75, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan bhai, nahi milega bhai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, hai bhai, ₹1500", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हॅलो, कोण?. हो, आहे. 13 पीस आहेत. 120 रुपये. घरपोच देतो", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "haan, stock mein hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, tell me. Not today, it comes tomorrow. I can give 3/4 inch elbow instead. Pickup only", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3/4 inch elbow"}}
{"product": "9W LED bulb", "said": "आहे ना. 3 पीस आहेत. रेट 140 आहे", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 30 rupaye honar, delivery deto", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan bola, ho aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "रुकिए, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "मालक नाहीत, नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan ji, mil jayega, 80 rupaye final, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "ji boliye, hai bhai, rate pachaas hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "नहीं है। उसकी जगह 12V 9Ah battery दे सकते हैं। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12V 9Ah battery"}}
{"product": "teflon tape", "said": "ji boliye, haan ji, mil jayega, rate 30 hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, thamba, baghun sangto, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Not today, it comes tomorrow", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "woh hum nahi rakhte, LED headlight bulb hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "LED headlight bulb"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "अभी नहीं है, परसों आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan ji, mil jayega, 3 hi bache hain, ₹80, home delivery hai", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "abhi stock mein nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan bolo, haan ji, mil jayega, rate 50 hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "khatam ho gaya, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "hai bhai, 14 piece pade hain, delivery ho jayegi, 10 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "It's in stock. That will be Rs 1700", "answer": {"in_stock": true, "price_inr": 1700, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "नाही आहे. त्याच्याऐवजी 3.15 mfd capacitor देऊ शकतो. घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "3.15 mfd capacitor"}}
{"product": "Fevicol SH 1 kg", "said": "bola, ho aahe, 420 rupaye honar", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "स्टॉक में है। रेट 2200 है। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 2200, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "hai bhai, 7 hi bache hain, ₹120, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Not today, it comes tomorrow. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "ho, milel, 9 ch urle aahet, 70 rupaye honar", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हॅलो, कोण?. मालक नाहीत, नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ जी, मिल जाएगा। 1150 लगेगा", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ बोलिए। ख़त्म हो गया। दुकान 9 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "है भाई। 230 लगेगा। दुकान 10 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "ho, milel, 270 rupaye honar", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "stock madhe aahe, adichshe rupaye honar, ghari pathavto", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, rate 90 aahe, ghari pathavto, 10 chya aat ya", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "nahi milega bhai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "ji haan, rakha hai, 5 piece pade hain, 220 lagega", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ, है। 13 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "woh hum nahi rakhte, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bolo, hai bhai, 1450 rupaye ka hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes, we have it. Price is 45. Pickup only", "answer": {"in_stock": true, "price_inr": 45, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan bolo, nahi hai, uski jagah 60 mm padlock de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "60 mm padlock"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "khatam ho gaya, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "हैलो, कौन?। हाँ, है। 20 पीस पड़े हैं। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Hello, who is this?. Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Yes sir, we have. We have 4 pieces. That will be Rs 90. Pickup only", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हॅलो, कोण?. आहे ना. 80 रुपये. घरी पाठवतो. दुकान 8 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan ji, mil jayega, 730 ka padega", "answer": {"in_stock": true, "price_inr": 730, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "सध्या नाही, उद्या येईल. डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "ji haan, rakha hai, rate 240 hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ, है। 120 लगेगा। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "available hai, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "nahi milega bhai, halogen bulb hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "halogen bulb"}}
{"product": "1 inch PVC elbow", "said": "हां बोला. स्टॉकमध्ये आहे. तीस रुपये", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes, available. 65 rupees", "answer": {"in_stock": true, "price_inr": 65, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "It's in stock. No delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ji haan, rakha hai, 100 rupaye final, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. It's in stock. We have 9 pieces. It's 220 rupees. We can deliver", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "nahi milega bhai, local brake shoe hai, chalega?, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local brake shoe"}}
{"product": "4 socket extension board", "said": "hai bhai, 520 ka padega", "answer": {"in_stock": true, "price_inr": 520, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हॅलो, कोण?. थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "haan aahe, 10 rupaye, yeun nyava lagel, 9 chya aat ya", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "pata nahi, godown mein dekhna padega, dukaan 9 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Hello?. No, we don't have it. We have local seal kit, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "Activa clutch plate", "said": "malik abhi nahi hain, baad mein phone karo, 9 baje ke pehle aana", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "malak nahit, nantar phone kara, 9 chya aat ya", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ji haan, rakha hai, 1500 lagega, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Yes, tell me. We don't keep that. I can give local brake shoe instead. Home delivery is available. Come before 8", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local brake shoe"}}
{"product": "bike headlight bulb", "said": "hai bhai, 7 piece pade hain, home delivery hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "Yes, tell me. Yes sir, we have. Price is 350", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Sorry, finished. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "khatam ho gaya, 2.25 mfd capacitor rakha hai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "2.25 mfd capacitor"}}
{"product": "Pulsar 150 spark plug", "said": "आम्ही ते ठेवत नाही. local spark plug आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "9W LED bulb", "said": "Hello?. Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan, stock mein hai, ₹110, delivery nahi hai, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 110, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan bola, haan aahe, 170 rupaye, ghari pathavto, dukan 10 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "haan hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "No, we don't have it. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho aahe, 120 rupaye, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "available hai, 17 piece pade hain, rate 400 hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. Sorry, finished. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan bola, udya yeil, aaj nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "sampla, 8 chya aat ya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "hello, kaun?, abhi stock mein nahi hai, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bhai, haan, stock mein hai, 6 piece pade hain, 70 rupaye ka hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "hello, kaun?, thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Out of stock", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "abhi nahi hai, parso aayega, uski jagah 2.25 mfd capacitor de sakte hain, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "2.25 mfd capacitor"}}
{"product": "50 mm padlock", "said": "हाँ बोलिए। स्टॉक में है। 2 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "abhi stock mein nahi hai, local seal kit rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "Pulsar 150 spark plug", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हॅलो, कोण?. स्टॉकमध्ये आहे. 10 पीस आहेत. तीस रुपये. येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Hello?. Yes, we have it. Only 6 left. It's 1100 rupees", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ, है। 17 पीस पड़े हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "khatam ho gaya, local air filter hai, chalega?, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local air filter"}}
{"product": "Pulsar 150 spark plug", "said": "udya yeil, aaj nahi, Pulsar 180 spark plug aahe, chalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pulsar 180 spark plug"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Not today, it comes tomorrow. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ji haan, rakha hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "hello, kaun?, woh hum nahi rakhte, delivery nahi hai, dukaan 8 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "hello, kon?, sampla, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ji haan, rakha hai, 760 rupaye ka hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 760, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "haan bhai, khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हॅलो, कोण?. मालक नाहीत, नंतर फोन करा. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "है भाई। रेट 90 है। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "ख़त्म हो गया। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हां बोला. नाही आहे. त्याच्याऐवजी 2.5 sq mm wire देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.5 sq mm wire"}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, haan ji, mil jayega, 70 rupaye final, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "hello, kaun?, nahi hai, 12W LED bulb hai, chalega?, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12W LED bulb"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "woh hum nahi rakhte, 1 sq mm wire rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 sq mm wire"}}
{"product": "8 mm drill bit", "said": "haan bhai, available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "sampla, dukan 7 vajta paryant ughada aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan aahe, 75 rupaye, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 75, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes sir, we have. 930 rupees", "answer": {"in_stock": true, "price_inr": 930, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "woh hum nahi rakhte, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "Hello, who is this?. It's in stock. Price is 260", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "No, we don't have it. I can give Jupiter clutch plate instead. Home delivery is available", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Jupiter clutch plate"}}
{"product": "bike headlight bulb", "said": "aamhi te thevat nahi, delivery deto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "bola, sampla, tyachya aivaji 3 inch hinge deu shakto, delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3 inch hinge"}}
{"product": "wall putty 20 kg", "said": "haan hai, rate 710 hai", "answer": {"in_stock": true, "price_inr": 710, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, stock mein hai, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "nahi aahe, porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, stock mein hai, 4 hi bache hain, 110 rupaye ka hai, home delivery hai", "answer": {"in_stock": true, "price_inr": 110, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "bola, udya yeil, aaj nahi, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan, stock mein hai, 370 rupaye final, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 370, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "hello, kaun?, nahi hai, delivery ho jayegi", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan hai, 240 ka padega", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ बोलिए। हाँ जी, मिल जाएगा। 230 रुपये का है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "sampla", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, available hai, rate 1000 hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "hai bhai, 16 piece pade hain, ₹360", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, tell me. Not sure, I'll have to check the godown. We are open till 7 pm", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ जी, मिल जाएगा। 70 रुपये का है। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "aamhi te thevat nahi, tyachya aivaji 12V 5Ah battery deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12V 5Ah battery"}}
{"product": "4 socket extension board", "said": "haan bhai, hai bhai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हैलो, कौन?। हाँ जी, मिल जाएगा। रेट 140 है। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 140, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, available. Home delivery is available. Come before 7", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, udya yeil, aaj nahi, tyachya aivaji 12W LED bulb deu shakto, delivery deto, 9 chya aat ya", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12W LED bulb"}}
{"product": "9W LED bulb", "said": "ho aahe, 70 rupaye honar, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "stock madhe aahe, rate 2100 aahe, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 2100, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "available hai, 330 ka padega, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "abhi stock mein nahi hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "No, we don't have it. I can give 3 inch brush instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 inch brush"}}
{"product": "water pump seal kit", "said": "haan ji, mil jayega, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ बोलिए। नहीं है। घर पर भेज देंगे। दुकान 9 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "ji boliye, nahi milega bhai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Out of stock. I can give local brake shoe instead. We are open till 10 pm", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local brake shoe"}}
{"product": "50 mm padlock", "said": "woh hum nahi rakhte, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "It's in stock. Price is 400. We don't deliver", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नाही आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी, मिल जाएगा। 60 लगेगा", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "We don't keep that. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "है भाई। 4 पीस पड़े हैं। रेट सौ है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan bola, nahi aahe, tyachya aivaji LED headlight bulb deu shakto, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "LED headlight bulb"}}
{"product": "Activa air filter", "said": "malak nahit, nantar phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "संपलं. 40 mm padlock आहे, चालेल का?. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 mm padlock"}}
{"product": "bike headlight bulb", "said": "Yes, we have it. That will be Rs 220. Pickup only", "answer": {"in_stock": true, "price_inr": 220, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "ji boliye, hai bhai, ₹270, 10 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "ho aahe, 110 la padel", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ji boliye, haan ji, mil jayega, 14 piece pade hain, 80 rupaye final", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "nahi aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "आहे ना. 3 पीस आहेत. घरी पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "Not today, it comes tomorrow. I can give Jupiter clutch plate instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Jupiter clutch plate"}}
{"product": "4 inch door hinge", "said": "available hai, 80 lagega, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "ji haan, rakha hai, 3 hi bache hain, 320 lagega", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "stock madhe aahe, rate 330 aahe, porga pathavto", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Out of stock. Come before 7", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नाही आहे. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bola, haan aahe, 80 la padel, yeun nyava lagel, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes, available. We have 18 pieces. It's 780 rupees", "answer": {"in_stock": true, "price_inr": 780, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Yes sir, we have. 10 rupees", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "nahi hai, local spark plug hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "10W-30 engine oil 1 L", "said": "woh hum nahi rakhte, 10W-30 engine oil 900 ml rakha hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "10W-30 engine oil 900 ml"}}
{"product": "4 inch door hinge", "said": "ho, milel, 60 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ बोलिए। नहीं है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes sir, we have", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हॅलो, कोण?. संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, tell me. Yes, available. It's 1450 rupees. I'll send it with our boy", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "अभी नहीं है, परसों आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes sir, we have. Only 19 left. 170 rupees", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "hello, kon?, ho, milel, porga pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haan bola, ho, milel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ बोलिए। है भाई। 16 पीस पड़े हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "malik abhi nahi hain, baad mein phone karo, dukaan 9 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 35 rupaye honar", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Yes, available. Price is 100. Home delivery is available. We are open till 8 pm", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "abhi nahi hai, parso aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हां बोला. आम्ही ते ठेवत नाही. 12V 9Ah battery आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12V 9Ah battery"}}
{"product": "16A MCB", "said": "आहे ना. 180 रुपये. येऊन न्यावं लागेल. दुकान 7 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 180, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "woh hum nahi rakhte", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ho, milel, 600 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 600, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes, we have it. 280 rupees. We are open till 7 pm", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan ji, mil jayega, dhai sau ka padega, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Not sure, I'll have to check the godown", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हॅलो, कोण?. आहे ना. 5 पीस आहेत. रेट 200 आहे. येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 200, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "woh hum nahi rakhte", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "रुकिए, देखकर बताता हूँ। दुकान 10 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes, tell me. Yes, available. That will be Rs 170", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हैलो, कौन?। हाँ, है। 330 रुपये का है", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "hai bhai, 340 ka padega", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Not today, it comes tomorrow. I can give mechanical seal instead. We don't deliver. Come before 7", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "mechanical seal"}}
{"product": "1 inch PVC elbow", "said": "woh hum nahi rakhte, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan ji, mil jayega, 13 piece pade hain, ₹1100, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "Hello?. Yes, available. We can deliver", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "hello, kaun?, hai bhai, rate 340 hai", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "sampla, 60 mm padlock aahe, chalel ka?, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "60 mm padlock"}}
{"product": "9W LED bulb", "said": "नहीं है। उसकी जगह 7W LED bulb दे सकते हैं। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "7W LED bulb"}}
{"product": "2 inch paint brush", "said": "haan bolo, nahi milega bhai, home delivery hai, dukaan 7 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "haan, stock mein hai, 17 piece pade hain, ₹330", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ, है। चालीस लगेगा। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
'''.strip().splitlines()]
TRAIN_SET = globals().get("TRAIN_SET", "v2")
TRAIN_DATA = TRAIN_V2 + REAL_XPQA if TRAIN_SET == "v2" else TRAIN_V1
TEST = [json.loads(line) for line in r'''
{"product": "10 mm nut bolt (pack of 10)", "said": "haan bhai mil jayega, packet ka 60 lagega. aap aake le jao, bhejne wala koi nahi hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "nahi hai abhi, 500 gram wala hai chalega toh", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "500 gram"}}
{"product": "16A MCB", "said": "Yes we have, 220 rupees. Delivery also possible in the evening.", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "ho aahe, 850 la padel. ghari pathavto", "answer": {"in_stock": true, "price_inr": 850, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "संपली आहे, सोमवारी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी है, 90 रुपये का है", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "kaun? achha tape... ruko beta ko puchta hun, baad mein call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "original khatam, local spark plug hai chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "wall putty 20 kg", "said": "aahe na, 20 kilo cha bag 780. delivery nahi, tempo karun nyava lagel", "answer": {"in_stock": true, "price_inr": 780, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Sorry sir, that size is finished. 40 mm is there.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 mm"}}
{"product": "2 inch paint brush", "said": "है, 45 का एक। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 45, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan hai, 70. home delivery nahi karte", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Available hai sir, full roll 1850 ka. Ghar pe bhijwa denge", "answer": {"in_stock": true, "price_inr": 1850, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "थांबा हं, मालक जेवायला गेलेत. अर्ध्या तासाने फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "10W-30 nahi rakhte, 20W-40 milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "20W-40"}}
{"product": "bike headlight bulb", "said": "yes available, ₹150, but pickup only", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "हो आहे, ४५० रुपये", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "nahi hai bhai, kal shaam tak aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "hai, 3 piece bache hain, 60 ka ek. ladka chhod dega ghar pe", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Splendor ka hai na, 260. dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 260, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "We don't keep sealant. Try M-seal, we have that.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "M-seal"}}
{"product": "1 inch PVC elbow", "said": "हाँ है, बीस रुपये का एक", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "hello? haan bolo... seal kit? hai, 300. delivery hoti hai", "answer": {"in_stock": true, "price_inr": 300, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "aahe, 390. aamhi delivery deto pan 50 rupaye extra", "answer": {"in_stock": true, "price_inr": 390, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Let me check... yes one piece is left. 1350 rupees.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "MCB 16 amp? nahi hai, 20 amp wala hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "20 amp"}}
{"product": "9W LED bulb", "said": "पता नहीं, गोदाम में देखना पड़ेगा। शाम को फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नाही, आम्ही ॲक्टिव्हाचे पार्ट ठेवत नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan hai, 350 ka. 7 baje tak aa jao, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 350, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "hai, 15 rupaye. ek kaam karo, 10 roll le lo toh 12 ka laga dunga", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
'''.strip().splitlines()]
HARD = [json.loads(line) for line in r'''
{"product": "TVS Jupiter tubeless tyre 90/90-12", "said": "Jupiter ka tyre ek hi tha, woh subah bik gaya, ab dukaan mein ek bhi nahi bacha. Parso tak naya maal aayega. Activa wala tyre same size ka pada hai, MRF ka, woh chalega kya?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Activa wala tyre"}}
{"product": "Godrej mortise lock 6 lever", "said": "Pehle ka rate sawa gyarah sau tha, ab company ne badha ke barah sau kar diya hai. Ek hi piece pada hai, chahiye toh abhi rakhwa lo.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "Hero Splendor chain kit", "said": "Sir, I joined only last week, so the owner will tell you the price and all. He is at the bank now and will be back after 4. Please call then or tomorrow.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Taparia screwdriver set (6 pcs)", "said": "The set is there... sorry, that is the 4 piece set. 6 piece we don't have. 4 piece is there at 180, will you take that?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "4 piece set"}}
{"product": "Maruti Alto oil filter", "said": "Alto ka oil filter ek hi piece pada hai, 180 ka. Do chahiye the toh kal tak mangwa dunga. Ladka ghar tak chhod dega, uska 40 rupaye alag se lagega.", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "Stanley 300 g claw hammer", "said": "Claw hammer 300 gram is there. Rs 240 only, handle is wooden. Sir, you are breaking... yes 240. And you have to collect from the shop, we do not send anything home.", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "Bajaj mixer grinder carbon brush", "said": "ब्रश तो हैं, लेकिन बजाज मिक्सर वाला नहीं, फिलिप्स वाला पड़ा है, वो दे दूँ? बजाज वाला शनिवार तक आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स वाला"}}
{"product": "Havells Crabtree 16A 3-pin socket", "said": "Last month it was 85, now the new rate is 92 after GST change. Stock is there, five-six pieces.", "answer": {"in_stock": true, "price_inr": 92, "delivery": null, "alternative": null}}
{"product": "Anchor Penta 6A two-way switch", "said": "Yes sir, available only. One piece is Rs. 38/- only, but if you take the full box of 10 then 320, so 32 each.", "answer": {"in_stock": true, "price_inr": 38, "delivery": null, "alternative": null}}
{"product": "Bajaj Pulsar 150 chain kit", "said": "चेन किट है... अरे नहीं, वो ऍवेंजर वाला है, आपके काम का नहीं। पल्सर १५० का किट तो पिछले महीने से आया ही नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch cutting wheel (pack of 5)", "said": "Cutting wheel cha pack aahe, 5 cha pack 90 la. Pan mala sang, tumhi site var kiti vaparta? Roj vaparat asal tar 25 cha box ghya, to 380 la hoto, jasta parvadto.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Syska 20W LED batten", "said": "Syska ka batten hai, 260 mein. Wipro ka 310 mein milta hai, woh chahiye ho toh bata dena, par aapne Syska kaha tha na.", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Royal Enfield Bullet 350 clutch plate set", "said": "Malak aata jevayla gele aahet, mala rate pan mahit nahi ani stock pan baghayla lagel. Tumhi dupari don chya sumaras call kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Nylon cable tie 8 inch (pack of 100)", "said": "पाकीट मिळेल, १२० ला... नाही नाही, आता १४० झालं, काल रेट वाढला.", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Distilled water 5 L can", "said": "5 litre cha can nahiye ho, 1 litre chya bottle aahet, 25 rupaye chi ek. Chalel ka tumhala?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 litre chya bottle"}}
{"product": "Crompton 1200 mm ceiling fan", "said": "Hoy, 1200 mm cha pankha ready aahe, white colour madhe. Diwali offer chalu aahe mhanun pahile 2400 la hota, aata 2150 la deto, ani ekach piece urlay.", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "Fischer 8 mm plastic wall plug (pack of 100)", "said": "होय, १०० चं पाकीट आहे, ६० ला मिळतं. सुटे हवे असतील तर १ रुपयाला एक देतो.", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Crompton 9 inch exhaust fan", "said": "9 inch wala toh is baar aaya hi nahi, 12 inch wala pada hai, woh bhej dun? Delivery toh hum karte hi hain, kal shaam tak ho jayegi.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12 inch wala"}}
{"product": "Eveready 3 cell torch", "said": "टॉर्च आहे, पण आजच किंमत वाढली. आधी १८० ला होता, आता दोनशे दहा झाला. एक पीस पडलाय फक्त.", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "Luminous 150Ah inverter battery", "said": "Malik se baat karo bhai, rate wahi batate hain. Main toh bas counter pe baitha hun, mujhe stock bhi nahi pata. Kal subah 11 ke baad phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Amaron 35Ah car battery", "said": "Haan 35 wali hai... matlab thi, kal tak. Kal hi do piece nikal gaye, ab ek bhi nahi bacha.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SDS plus 12 mm drill bit", "said": "12 mm cha SDS bit nahiye ajun, order dila aahe. Aala ki aamhi tumchya site var gadi pathavu, delivery cha charge nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Honda Shine chain sprocket kit", "said": "Shine cha kit aala hota, pan kalach don piece gele. Aata ek pan urla nahi. Udya sakali gaadi yeil. Pathvaychi soy nahiye, tumhi yeun neta ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Cera 15 mm CP angle cock", "said": "Cera चा नाहीये, जॅग्वारचा आहे, चालेल ना? त्याची किंमत ३५० आहे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "जॅग्वारचा"}}
{"product": "Maruti Swift Dzire wiper blade (pair)", "said": "Dzire cha wiper aahe, pair madhe 380 la. Pan ekach pair urlay mhanun lavkar ghya, nahitar udya sampel.", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Anchor Roma 6A switch", "said": "Roma ka 6 amp wala switch? Woh toh paintalis ka aata hai, kal hi naya maal utra hai, abhi 2 dabbe pade hain counter ke neeche.", "answer": {"in_stock": true, "price_inr": 45, "delivery": null, "alternative": null}}
{"product": "Bajaj Pulsar 150 front brake pad", "said": "Pulsar cha pad aahe, pan to 220 cha. 150 cha nahiye amchyakade, ani 220 cha tumchya gadi la basnar nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ceiling fan regulator (step type)", "said": "Step wala regulator hai... ruko, arre woh toh electronic wala hai. Step wala toh pichhle mahine hi khatam hua, electronic wala 180 mein deta hun, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "electronic wala"}}
{"product": "UltraTech cement 50 kg bag", "said": "Cement aaj hi gaadi se utra hai, 50 kilo ka bag 395 ka. 20 bag se upar lo toh 385 laga dunga. Ghar tak nahi bhejte hum, khud tempo lekar aana padega.", "answer": {"in_stock": true, "price_inr": 395, "delivery": false, "alternative": null}}
{"product": "Bajaj Platina clutch cable (original)", "said": "Platina ka clutch cable rakha hai. Local wala nabbe ka milta hai, original Bajaj ka sawa do sau mein. Aapko original chahiye na, toh woh sawa do sau.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Wipro 20W LED tube light", "said": "Tube aahe, paune donshe la ek. Choke nako lagat, direct lavta yeto.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "आज रेट नहीं बता सकता, कल बोलना। सेठ जी बैंक गए हैं और स्टॉक की लिस्ट उन्हीं के पास है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch 10 mm hammer drill machine", "said": "Bosch ka toh stock mein hai hi nahi, Stanley ka 10 mm wala pada hai, woh 2200 mein de dunga, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Stanley ka 10 mm wala"}}
{"product": "Bosch GWS 600 angle grinder", "said": "It is not that we don't keep Bosch, grinders are always there. Just the 600 watt one got sold today, the 850 watt is there at 4200. Shall I keep that for you?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "850 watt"}}
{"product": "Cera half inch CP bib cock", "said": "Bib cock aahe, Cera cha ek doshe pannas la. Local cha pan aahe, to ekshe vees la.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Havells 4-way MCB box", "said": "४ वे चा बॉक्स संपलाय, पण आम्ही घरपोच करतो, फक्त माल आला की.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Orient 16 inch table fan blade", "said": "Blade aaj nahi milega, jo the woh sab Sunday ko nikal gaye. Naya maal Tuesday ko aayega, 3 baje tak phone kar lena.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Taparia 8-piece spanner set", "said": "No sir, the last Taparia set went yesterday evening. New stock will come on Thursday after 11. Whenever it comes you have to collect, we do not send things home. Other brand also we don't keep.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Havells musical door bell", "said": "Hoy aahe... ek minute, mi chukicha bolto, to musical nahi, buzzer aahe. Musical bell nahiye, buzzer deu ka 150 la?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "buzzer"}}
{"product": "Araldite 15 g tube", "said": "अरल्डाइट? आज सुबह आख़िरी ट्यूब चली गई। कल दोपहर २ बजे के बाद नया बॉक्स आएगा, घर तक भिजवा दूँगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Stanley 5 m measuring tape", "said": "Aahe... nahi thamba, to Freemans cha aahe, Stanley cha nahiye. Freemans cha 5 meter cha tape 120 la deu ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Freemans"}}
{"product": "Tata Ace coolant 1 L", "said": "Mi Nilesh bolto, malak kalach gavi gele. Rate te yeun sangtil. Stock baddal mala kahi mahit nahi, ya aathvadyat parat phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr. Fixit LW+ 1 litre", "said": "LW+ one litre bottle, yes. MRP is three twenty but I will give for two ninety, final. Hundred rupees extra if you want it sent to your site.", "answer": {"in_stock": true, "price_inr": 290, "delivery": true, "alternative": null}}
{"product": "Havells 25A DP MCB", "said": "DP 25 amp ka hai sir, 540 mein. Mera number likh lo 9822334455, kuch aur chahiye toh WhatsApp kar dena.", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "Hero Splendor Plus side mirror (left)", "said": "Left cha mirror aahe, right cha nahiye. Tumhala left cha pahije na? 130 la ek.", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "Asian Paints Tractor Emulsion 4 L (white)", "said": "White 4 litre abhi bacha nahi, cream 4 litre pada hai, usse bhi kaam chal jata hai. Uska 780 padta hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "cream 4 litre"}}
{"product": "Hero Passion Pro tail lamp", "said": "टेल लैंप पड़ा है, सवा दो सौ में दे दूँगा, बल्ब भी लगा हुआ आता है। घर तक भेजने का इंतज़ाम नहीं रखते, खुद उठाना पड़ेगा।", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "PVC conduit pipe 20 mm (3 m piece)", "said": "ISI conduit pipe, yes. Per piece 25 rupees, if you take full bundle of 10 then 230. Delivery I will check with the driver and tell you, not sure.", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Hacksaw blade 12 inch (pack of 10)", "said": "हाँ मिल जाएगा, १० वाले पैकेट के १४० रुपये ले लेता हूँ। पर अगर आप एक-दो ब्लेड लोगे तो १८ का एक पड़ेगा।", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Philips 18W LED panel light (round)", "said": "Round 18 watt cha panel aahe, 480 la. GST bill pahije asel tar 18 tax vegla lagto.", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "Activa 6G CVT belt", "said": "रेट आज नाही सांगता येणार, मालक बाहेर गेलेत. उद्या फोन करा, मग स्टॉक पण बघून सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 63A 2-pole ELCB", "said": "2 pole cha ELCB aahe ho, 1450 la dein. 4 pole cha pan aahe, to 3100 la, pan to tumhala nako asel na?", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "1500W water heater element", "said": "1500 watt is finished since morning, sir. 2000 watt is there, can you use that? Our boy can drop it at your place, no extra charge.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "2000 watt"}}
{"product": "Royal Enfield Classic 350 air filter (original)", "said": "Original or local? We keep both. Original RE one is 450, local is 160. Original is in stock, only 2 pieces, local is plenty.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Servo 80W-90 gear oil 1 L", "said": "Gear oil? Awaaz kat rahi hai... haan ab sunai diya. Servo ka 80W-90 hai, 3 litre ka can 900 ka aata hai aur 1 litre ka 330 ka.", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "ESAB 3.15 mm welding rod (1 kg packet)", "said": "3.15 cha rod aahe ho, 1 kg cha packet 190 la. Loose pan deto, 20 rod cha 70 la. Pathvaychi soy nahiye, tumchya manus la pathva.", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
'''.strip().splitlines()]
FRESH = [json.loads(line) for line in r'''
{"product": "Polycab 1 sq mm flexible wire", "said": "अरे भाई, अभी दुकान पर बहुत भीड़ है। Polycab का एक MM वाला तार है या नहीं, ये देखना पड़ेगा। आप आधे घंटे बाद दोबारा कॉल कर लो, तब तक मैं गोडाउन से पता कर लूँगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ashirvad 1/2 inch CPVC elbow", "said": "आधा इंच CPVC एल्बो Ashirvad का है, एक पीस १२ रुपये का, ५० वाला डिब्बा ५२० में मिलेगा। डिलीवरी नहीं है, आकर ले जाना पड़ेगा।", "answer": {"in_stock": true, "price_inr": 12, "delivery": false, "alternative": null}}
{"product": "25 mm PVC conduit pipe", "said": "Nahi bhaiya, 25 mm conduit pipe abhi khatam hai, agle hafte aayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Hitachi 4 inch cutting wheel", "said": "4 inch cha cutting wheel aahe. Ek wheel 25 rupaye, 25 wheel cha box 550 la deto. Pathvaycha asel tar courier ne pathavto, 40 rupaye lagtil. Kiti pahije tumhala?", "answer": {"in_stock": true, "price_inr": 25, "delivery": true, "alternative": null}}
{"product": "Maruti Swift clutch cable", "said": "स्विफ्टची क्लच केबल आत्ता नाहीये, पुढच्या आठवड्यात मागवून देतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Exide 35 Ah car battery", "said": "Seth ji abhi khana khane gaye hain, battery ka stock aur rate unko hi pata hai. Aap ek ghante baad call karo, 2 baje ke baad.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 16 amp socket", "said": "Yes, Havells 16 amp socket is available, 145 rupees each.", "answer": {"in_stock": true, "price_inr": 145, "delivery": null, "alternative": null}}
{"product": "Nerolac Excel exterior paint 20 litre", "said": "Nerolac Excel twenty litre, let me check with the godown sir. We had some, but painters took a lot this week. I will confirm and send you a WhatsApp in half an hour.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous 900 VA inverter", "said": "Luminous 900 VA inverter is there sir. MRP is 7200, but we are giving it for 6400 including GST. Delivery and installation we will do free of cost.", "answer": {"in_stock": true, "price_inr": 6400, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Fevicol SH ek kilo wala abhi nahi hai. Aadha kilo ke dabbe hain, dhai sau ka ek, wo chahiye to bata do.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Fevicol SH 500 g"}}
{"product": "Havells 16 amp single pole MCB", "said": "हाँ जी, Havells का १६ एम्पियर MCB है, १९० रुपये का एक।", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Wipro 18 watt round LED panel", "said": "18 watt cha Wipro panel ata nahiye, company kadun maal aala nahi don athavde zale.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MRF Nylogrip 2.75-18 tyre", "said": "Malak baher gelet, tyre cha stock tech baghtat. Sandhyakali saha nantar phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Pulsar 150 clutch plate", "said": "Nahi ho, Pulsar chi clutch plate sampli aahe, udya yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sintex 500 litre water tank", "said": "हो, सिंटेक्सची ५०० लिटरची टाकी आहे, साडेतीन हजार. घरपोच करतो.", "answer": {"in_stock": true, "price_inr": 3500, "delivery": true, "alternative": null}}
{"product": "Crompton 15 litre geyser", "said": "Crompton cha 15 litre geyser aahe, 6500... thamba thamba, chukla, to 25 litre cha rate hota. 15 litre cha 5800 aahe. Fitting sathi manus pathavto, delivery free aahe.", "answer": {"in_stock": true, "price_inr": 5800, "delivery": true, "alternative": null}}
{"product": "Finolex 1.5 sq mm wire coil", "said": "Finolex 1.5 ka coil abhi nahi hai ji.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient Apex 1200 mm ceiling fan", "said": "Orient Apex पंखा है। डिब्बे पर MRP तो दो हज़ार लिखा है, लेकिन आपको पौने सत्रह सौ में दे देंगे।", "answer": {"in_stock": true, "price_inr": 1675, "delivery": null, "alternative": null}}
{"product": "Bosch 13 mm impact drill", "said": "Bosch ka 13 mm drill abhi stock mein nahi hai, order karoge to 4 din lagega. Uski jagah Dewalt ka hai 3200 ka, wo le lo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Dewalt 13 mm drill"}}
{"product": "Syska 12 watt LED bulb", "said": "Syska cha 12 watt bulb aahe na, MRP 199 aahe pan aamhi 150 la deto. Don ghetle tar 280 la deto.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Polycab 4 sq mm wire coil", "said": "4 sq mm Polycab ka maal nahi hai bhaiya, itna mota wire hum rakhte hi nahi. Kasba Peth mein try karo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj 1200 mm ceiling fan", "said": "Bajaj ka 1200 mm wala fan hai na, MRP to 2450 likha hai dabbe pe, lekin aapko 2100 mein de dunga. Fitting ka mistri alag se lagega.", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Legrand 25 amp RCCB", "said": "लेग्रँडचा २५ अँपिअरचा RCCB संपलाय. हॅवेल्सचा आहे, तो अठराशेला पडेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 25 amp RCCB"}}
{"product": "Philips 20 watt LED batten", "said": "20 watt LED batten Philips ka hai. Single piece saade teen sau, 4 ka pack 1300 mein. Delivery chahiye to Kothrud tak 50 rupaye extra lagenge.", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Anchor 32 amp DP MCB", "said": "Anchor cha 32 amp DP nahiye. Legrand cha aahe tyach rating cha, to chalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Legrand 32 amp DP MCB"}}
{"product": "Amaron 5 Ah bike battery", "said": "Ho, Amaron chi 5 Ah battery aahe, solashe pannas la. Ghari yeun basvun deto, Hadapsar madhe asal tar tyache vegle 100 rupaye lagtil.", "answer": {"in_stock": true, "price_inr": 1650, "delivery": true, "alternative": null}}
{"product": "Honda Activa air filter", "said": "Activa cha air filter aahe, 180 rupaye. Delivery nahi karat aamhi, tumhala dukanat yava lagel.", "answer": {"in_stock": true, "price_inr": 180, "delivery": false, "alternative": null}}
{"product": "TVS Apache rear brake pad", "said": "अपाचे के पीछे वाले ब्रेक पैड ख़त्म हो गए हैं, नया माल सोमवार को आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Jaquar pillar cock", "said": "Jaquar ka pillar cock... ek minute, godown mein dekhna padega. Main check karke aapko wapas call karta hoon.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Asian Paints Tractor Emulsion white 10 litre", "said": "Tractor emulsion white 10 litre ka dabba hai, 2350 lagega. Delivery hum nahi karte bhai, aapko khud aake le jaana padega, dukaan 10 se 8 khuli hai.", "answer": {"in_stock": true, "price_inr": 2350, "delivery": false, "alternative": null}}
{"product": "Ambuja cement 50 kg bag", "said": "Ambuja ka bag hai. Pehle 380 tha, ab 395 ho gaya hai, GST mila ke. Tempo se bhej denge, 10 bag se upar delivery free hai.", "answer": {"in_stock": true, "price_inr": 395, "delivery": true, "alternative": null}}
{"product": "Usha 400 mm table fan", "said": "Usha ka table fan khatam ho gaya. Crompton ka 400 mm wala hai, wo chalega to bolo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 400 mm table fan"}}
{"product": "Kirloskar 0.5 HP water pump", "said": "किर्लोस्करचा अर्धा HP पंप आहे. आधी ४२०० होता, आता ४५०० झालाय. डिलिव्हरी नाही, दुकानातून घेऊन जावा लागेल.", "answer": {"in_stock": true, "price_inr": 4500, "delivery": false, "alternative": null}}
{"product": "Exide 150 Ah inverter battery", "said": "सेठ जी अभी बैंक गए हैं, बैटरी का रेट वही बताएँगे। आप शाम ५ बजे के बाद फ़ोन कीजिए।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Royal Enfield Classic 350 clutch lever", "said": "मालक आज गावाला गेलेत, बुलेटचे पार्ट्स तेच बघतात. उद्या सकाळी दहा नंतर फोन करा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Supreme 4 inch PVC pipe", "said": "४ इंच का Supreme पाइप अभी नहीं है, Finolex का है उसी साइज़ में, वो चलेगा तो बताइए।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Finolex 4 inch PVC pipe"}}
{"product": "Taparia 8 inch combination plier", "said": "टपारियाची ८ इंची पकड आहे. छापील किंमत २२० आहे, पण आम्ही तुम्हाला पावणे दोनशेला देतो.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Supreme 1 inch PVC elbow", "said": "Ho, 1 inch cha elbow aahe, 18 rupaye la ek.", "answer": {"in_stock": true, "price_inr": 18, "delivery": null, "alternative": null}}
{"product": "Jindal 1 inch GI pipe", "said": "एक इंची GI पाइप जिंदालचा... थांबा, गोदामात बघावं लागेल. मी तुम्हाला परत फोन करतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 9 watt LED bulb", "said": "9 watt Philips hai, 110 ka... nahi nahi sorry, rate badh gaya hai, ab 120 ka hai. Ghar pe bhijwa denge, 20 rupaye delivery lagega.", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "Bajaj 3 litre instant geyser", "said": "बजाज का ३ लीटर गीज़र नहीं है। Racold का ३ लीटर वाला है, वो साढ़े तीन हज़ार का पड़ेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Racold 3 litre instant geyser"}}
{"product": "Anchor Roma 6 amp switch", "said": "Haan ji, Anchor Roma 6 amp switch hai, 32 rupaye ka ek.", "answer": {"in_stock": true, "price_inr": 32, "delivery": null, "alternative": null}}
{"product": "Godrej 7 lever padlock", "said": "Main to yahan naya hoon bhaiya. Godrej ke taale wala counter malik dekhte hain, wo shaadi mein gaye hain, kal subah 11 baje ke baad phone karna, wo rate bhi bata denge.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Roff tile adhesive 20 kg bag", "said": "Roff chi bag... ek minute, godown madhe aahe ka baghun sangto. Tumhi daha minitani parat phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Myrius 6 amp 2 way switch", "said": "See, Legrand Myrius 2 way switch 6 amp we have. Single piece is 78, a box of 20 I can give you at 1400. No home delivery sir, you will have to collect it from the shop, we are near Swargate.", "answer": {"in_stock": true, "price_inr": 78, "delivery": false, "alternative": null}}
{"product": "15 mm brass bib cock", "said": "Pitalacha 15 mm bib cock sampla aahe. Stainless steel cha aahe tyach size cha, tyachi kimmat 350.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "stainless steel 15 mm bib cock"}}
{"product": "Dr. Fixit LW+ 1 litre", "said": "डॉ. फिक्सिट LW+ एक लीटर वाला है। पहले २१० का था, अब २३५ का हो गया है। घर पर भिजवा देंगे, कोई चार्ज नहीं।", "answer": {"in_stock": true, "price_inr": 235, "delivery": true, "alternative": null}}
{"product": "Stanley 5 m measuring tape", "said": "Stanley का ५ मीटर टेप है, ३२० का... नहीं, माफ़ कीजिए, ३५० का है, नया रेट आया है। भेजना हो तो ३० रुपये डिलीवरी लगेगी।", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "3M double sided foam tape 24 mm", "said": "The 24 mm 3M foam tape is finished sir. Only the 12 mm roll is there, if that works I can keep one aside for you.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3M foam tape 12 mm"}}
{"product": "Asian Paints Apcolite enamel 1 litre", "said": "Apcolite enamel ek litre aahe. Junya stock cha 420 hota, navin stock aala to 460 cha aahe, ata fakt navinch shillak aahe.", "answer": {"in_stock": true, "price_inr": 460, "delivery": null, "alternative": null}}
{"product": "Bosch spark plug for Maruti Alto", "said": "Sir, actually the person who handles spares is not here today, I am only sitting at the counter. Please call tomorrow after 11, he will tell you.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Jaquar 2 in 1 wall mixer", "said": "Jaquar wall mixer 2 in 1 is available. Price is 3450... sorry, 3650, I was looking at the old list. We can send it with our boy, 80 rupees delivery charge.", "answer": {"in_stock": true, "price_inr": 3650, "delivery": true, "alternative": null}}
{"product": "Motul 7100 10W-50 1 litre", "said": "No sir, Motul 7100 is out of stock since last week, the distributor has not supplied yet.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Castrol Activ 4T 20W-40 1 litre", "said": "Castrol Activ 20W-40 ek litre wala hai, chaar sau pachaas ka ek bottle. Teen bottle pade hain abhi, dukaan raat 9 baje tak khuli hai.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "M8 nut bolt", "said": "M8 ka nut bolt hai, ek piece 6 rupaye ka. Poora dabba 100 piece ka saade chaar sau mein de dunga agar zyada chahiye.", "answer": {"in_stock": true, "price_inr": 6, "delivery": null, "alternative": null}}
{"product": "Taparia 6 piece screwdriver set", "said": "Taparia screwdriver set, I am not sure if it is there, my brother keeps those in the cupboard and he has the key. He will be back by 6, please call then.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cera wash basin", "said": "सेराचं पांढरं वॉश बेसिन आहे, चौदाशेला एक. दोन घेतलेत तर सत्ताविसशेला देईन. घरपोच पाठवतो, कोथरूडला शंभर रुपये लागतील.", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "8 inch tower bolt", "said": "8 inch tower bolt nahiye, 6 inch cha aahe, to deu ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "6 inch tower bolt"}}
{"product": "Birla White wall putty 20 kg", "said": "Birla White putty chi 20 kilo chi bag aahe, saadesatshe la padel. Gharpoch karto aamhi, Warje paryant kahi charge nahi.", "answer": {"in_stock": true, "price_inr": 750, "delivery": true, "alternative": null}}
{"product": "Hero Splendor original brake shoe", "said": "Splendor ka brake shoe? Haan rakha hai. Local wala bhi hai 150 ka, par aapko original chahiye na, wo Hero ki genuine packing mein hai, 280 ka padega.", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
'''.strip().splitlines()]
SETS = {'test': TEST, 'hard': HARD, **({'fresh': FRESH} if FRESH else {})}
print(TRAIN_SET, len(TRAIN_DATA), "training rows;", {k: len(v) for k, v in SETS.items()}, "test calls")

In [ ]:
#@title 5. Load the base model (about 2–3 minutes)
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import importlib.metadata as md
import torch

print({p: md.version(p) for p in ("unsloth", "unsloth_zoo", "torch", "transformers", "trl", "peft")})

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=1024, load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")
# LoRA: train a small add-on (about 1% of the weights) instead of the whole model.
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

In [ ]:
#@title 6. Test the model BEFORE training
def ask(product, said):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": user_message(product, said)}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=96, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def run_tests(label):
    FastLanguageModel.for_inference(model)
    try:
        results = evaluate(ask, SETS)
    finally:
        FastLanguageModel.for_training(model)
    show(label, results)
    return results


before = run_tests("BEFORE training")

In [ ]:
#@title 7. Train (about 10–15 minutes)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer


def to_text(ex):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": user_message(ex["product"], ex["said"])},
                {"role": "assistant", "content": answer_text(ex["answer"])}]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


dataset = Dataset.from_list([to_text(ex) for ex in TRAIN_DATA])
print(dataset[0]["text"])

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text", max_length=1024,
        per_device_train_batch_size=4, gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=5, lr_scheduler_type="linear",
        optim="adamw_8bit", weight_decay=0.01, logging_steps=10, seed=3407,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        output_dir="outputs", report_to="none",
    ),
)
# Learn only the answers, not the questions.
trainer = train_on_responses_only(
    trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(f"Training took {stats.metrics['train_runtime'] / 60:.1f} min, loss {stats.metrics['train_loss']:.3f}")

In [ ]:
#@title 8. Test the model AFTER training
after = run_tests("AFTER training")
payload = results_payload(MODEL_NAME, EPOCHS, before, after)
payload.update(train_set=TRAIN_SET, train_rows=len(TRAIN_DATA))
print("\n== Before vs after (all 4 fields right)")
for name, res in payload["sets"].items():
    print(f"   {name:5s} {res['before']['all_correct']:4.0%} -> {res['after']['all_correct']:4.0%}  ({res['examples']} calls)")
print("\nRESULTS_JSON", json.dumps({k: {n: r[k] for n, r in payload["sets"].items()} for k in ("before", "after")}))
if ON_KAGGLE:  # run_on_kaggle.py downloads this
    with open("/kaggle/working/results.json", "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=1)

In [ ]:
#@title 9. Try your own sentence
product = "Activa clutch plate"  #@param {type:"string"}
said = "haan hai, 650 ka padega, ghar pe bhej denge"  #@param {type:"string"}
FastLanguageModel.for_inference(model)
print(ask(product, said))

In [ ]:
#@title 10. Save your model: Google Drive on Colab, the Output tab on Kaggle (the Ollama file takes 10–15 minutes)
import glob, re

lora_dir = f"{WORK}/{OUTPUT_NAME}-lora"
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
if SAVE_DIR:
    os.makedirs(SAVE_DIR, exist_ok=True)
    shutil.copytree(lora_dir, f"{SAVE_DIR}/lora", dirs_exist_ok=True)
    print("Saved the LoRA add-on to", f"{SAVE_DIR}/lora")

if MAKE_OLLAMA_FILE:
    gguf_dir = f"{WORK}/{OUTPUT_NAME}"
    result = model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="q4_k_m")
    files = (result or {}).get("gguf_files") if isinstance(result, dict) else None
    files = files or glob.glob(f"{gguf_dir}*/**/*.gguf", recursive=True)
    if not files:
        raise RuntimeError("No .gguf file was made. The LoRA add-on is still saved; send Claude the error above.")
    q4 = [f for f in files if "q4_k_m" in os.path.basename(f).lower()] or sorted(files, key=os.path.getsize)
    gguf = q4[0]
    name = f"{OUTPUT_NAME}.Q4_K_M.gguf"

    modelfile = make_modelfile(name)
    modelfile_path = os.path.join(os.path.dirname(gguf), "Modelfile.wimb")  # gguf_dir itself may be cleaned up
    with open(modelfile_path, "w", encoding="utf-8") as fh:
        fh.write(modelfile)
    print(modelfile)

    if SAVE_DIR:
        shutil.copy(gguf, f"{SAVE_DIR}/{name}")
        shutil.copy(modelfile_path, f"{SAVE_DIR}/Modelfile")

if SAVE_DIR:
    for path in sorted(glob.glob(f"{SAVE_DIR}/*")):
        size = os.path.getsize(path) / 1e6 if os.path.isfile(path) else 0
        print(f"   {path}  {size:,.0f} MB" if size else f"   {path}/")

In [ ]:
#@title 11. Test the exported GGUF file itself (CPU, up to 25 minutes; the answers Ollama would give)
import socket, subprocess, time, urllib.request

gguf_report = {"error": "not run"}
server = None
try:
    if not MAKE_OLLAMA_FILE:
        raise RuntimeError("MAKE_OLLAMA_FILE is off, so there is no GGUF file to test")
    binary = subprocess.run(
        ["bash", "-c", "find / -xdev -name llama-server -type f -perm -u+x 2>/dev/null | head -1"],
        capture_output=True, text=True).stdout.strip()
    if not binary:
        raise RuntimeError("llama-server was not found (Unsloth builds it while making the GGUF)")
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        port = probe.getsockname()[1]
    server = subprocess.Popen([binary, "-m", gguf, "--port", str(port), "--host", "127.0.0.1", "-c", "2048",
                               "-t", str(os.cpu_count() or 2), "--jinja"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    base = f"http://127.0.0.1:{port}"
    for _ in range(120):  # up to 6 minutes to load the model
        if server.poll() is not None:
            raise RuntimeError(f"llama-server stopped at start-up (exit code {server.returncode})")
        try:
            urllib.request.urlopen(base + "/health", timeout=5).read()
            break
        except Exception:
            time.sleep(3)
    else:
        raise RuntimeError("llama-server did not become ready")

    def ask_gguf(product, said):
        body = json.dumps({"messages": [{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": user_message(product, said)}],
                           "temperature": 0, "max_tokens": 96}).encode()
        req = urllib.request.Request(base + "/v1/chat/completions", body, {"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=300) as resp:
            return json.load(resp)["choices"][0]["message"]["content"].strip()

    started = time.time()
    gguf_results = evaluate(ask_gguf, SETS, deadline=started + 25 * 60)
    show("GGUF Q4_K_M", gguf_results)
    gguf_report = {"gguf": os.path.basename(gguf), "seconds": round(time.time() - started),
                   "sets": {n: {"examples": r["examples"], "of": r["of"],
                                **({k: round(r[k], 3) for k in KEYS} if r["examples"] else {}),
                                "mistakes": r["mistakes"], "replies": r["replies"]}
                            for n, r in gguf_results.items()}}
except Exception as e:  # the main results are already saved; this test must never fail the run
    print("GGUF test failed:", repr(e))
    gguf_report = {"error": repr(e)}
finally:
    if server is not None:
        server.terminate()
if ON_KAGGLE:
    with open("/kaggle/working/results_gguf.json", "w", encoding="utf-8") as fh:
        json.dump(gguf_report, fh, ensure_ascii=False, indent=1)

## Done. Next steps

**1. Save this notebook to GitHub** so Claude can read the results:
**File → Save a copy in GitHub** → `pratik9yadav/yok`, branch `main`,
path `train_colab.ipynb`.

**2. Run your model on your PC**
1. Install Ollama from ollama.com.
2. From Google Drive, download the `wimb-shop-model` folder (the `.gguf` file and `Modelfile`).
3. Open a terminal in that folder and run:
```
ollama create wimb-shop -f Modelfile
ollama run wimb-shop "Product: 16A MCB
Shopkeeper: haan hai, 220 ka, delivery nahi"
```